
# Track 2 V23 — High-Recall Missed-Event Recovery + Auto-Submission

This notebook is the submission-oriented follow-up to V22.

V22 established that the remaining V6 error is mostly **proposal recall**, not accepted-event quality:

- V6 final accepted-event recall on SELECT: ~82.7%
- 71 / 410 GT events missed
- 55 missed events have an exact source-disjoint BANK instance and are recovered by V13 when the event window is supplied
- 62 / 71 misses are `no_localized_proposal`

V23 therefore **never changes a V6-accepted event**.

It performs a second, high-recall scan only for additional events, confirms them with V13
noisy-only fingerprint retrieval, applies the same V6 single-event fitting safeguards, learns a
high-precision candidate gate on SELECT, audits natural controls and HOLD, and generates
hidden-test submission ZIPs **only if validation gates pass**.

To keep the second pass fast enough for the final day, the default recovery bank is the
small set of source-disjoint BANK fingerprints identified by V22 as exact recoveries of
missed SELECT events. This is validation-tuned fingerprint selection; HOLD is used as the
generalization check before any ZIP is built.

No GT window, clean reference, validation label or domain label is used in hidden-test inference.


In [ ]:

# 0. Configuration
from pathlib import Path
import os, sys

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    print('Outside Colab: edit paths below.')

WORK = Path('/content/drive/MyDrive/Datathon/work')
V5_DIR = WORK / 'v5_synthetic_residual_evidence'
VAL_CACHE = WORK / 'official_val_posteriors_epoch_2.pkl'

VAL_ROOT = None
VAL_ROOT_CANDIDATES = [
    WORK.parent / 'validation',
    Path('/content/drive/MyDrive/PHD/Datathon/validation'),
    Path('/content/drive/MyDrive/validation'),
]

V6_ROOT = WORK.parent / 'track2_v6_direct_reconstruction'
V6_RUN_KEY = '23523d3d7e5ed3ea'
V6_RUN = V6_ROOT / ('run_' + V6_RUN_KEY)

V13_ROOT = WORK.parent / 'track2_v13_high_recall_instance_retriever'
V22_ROOT = WORK.parent / 'track2_v22_missed_event_diagnostic'
V22_MISSED_CSV = V22_ROOT / 'missed_event_recurrence_retrieval.csv'

OUT_ROOT = WORK.parent / 'track2_v23_high_recall_missed_event_recovery'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SR = 16000
SEED = 20260920

# V13 frozen choices.
DEFAULT_CHEAP_RECIPE = 'balanced'
DEFAULT_SHORTLIST_K = 32
DEFAULT_FINE_RECIPE = 'fir_heavy'
TIME_BINS = 256
ENV_BINS = 64
SPEC_BINS = 96
FINE_SHIFT_GRID = [-64,-32,-16,-8,0,8,16,32,64]
FIR_TAPS = 5
FIR_RIDGE_REL = 2e-3
FIR_MAX_SAMPLES = 10000

# Recovery proposal generator. The bank is small, so this is much more aggressive
# than the original V6 scan while remaining practical.
RECOVERY_COARSE_FLOOR = 0.08
RECOVERY_PEAKS_PER_TEMPLATE = 12
RECOVERY_VERIFY_TOPK = 512
RECOVERY_RAW_RADIUS = 32

# Candidate pruning BEFORE V13 confirmation.
RECOVERY_MIN_VERIFY_QUALITY = 0.20
MAX_RECOVERY_CANDIDATES_PER_CLIP = 16
RECOVERY_NMS_IOU = 0.50
ACCEPTED_EXCLUSION_MARGIN_MS = 20.0

# Positive label for SELECT candidate diagnostics.
GT_POSITIVE_IOU = 0.50

# OOF candidate gate / end-to-end safety.
OUTER_FOLDS = 5
CANDIDATE_WIN_MARGIN_DB = 0.02
BAD_CLIP_MARGIN_DB = 1.0

BALANCED_MAX_SYN_HARM = 0.05
CONSERVATIVE_MAX_SYN_HARM = 0.02
BALANCED_MIN_CANDIDATE_PRECISION = 0.55
CONSERVATIVE_MIN_CANDIDATE_PRECISION = 0.70

BALANCED_NATURAL_ROUTE_CAP = 0.02
CONSERVATIVE_NATURAL_ROUTE_CAP = 0.01

# Require some real waveform upside before hidden-test build.
SELECT_MIN_GAIN_BALANCED_DB = 0.50
SELECT_MIN_GAIN_CONSERVATIVE_DB = 0.20

# HOLD is final generalization gate; thresholds are frozen before HOLD.
HOLD_MIN_GAIN_BALANCED_DB = 0.20
HOLD_MIN_GAIN_CONSERVATIVE_DB = 0.08
HOLD_MAX_HARM_BALANCED = 0.05
HOLD_MAX_HARM_CONSERVATIVE = 0.03
HOLD_MAX_WER_INCREASE = 0.002

EXPECTED_TEST_CLIPS = 5517
TEST_AUDIO_SOURCE = None
TEST_AUDIO_CANDIDATES = [
    Path('/content/data'),
    Path('/content/indoml_test_local'),
    Path('/content/test_wavs'),
    WORK.parent / 'input_data.zip',
    Path('/content/drive/MyDrive/PHD/Datathon'),
]

BUILD_BALANCED = True
BUILD_CONSERVATIVE = True
TRANSCRIPT_FIELD = 'transcript'

ASR_REPO = 'ARTPARK-IISc/SraVaani-1.0'
ASR_BATCH_SIZE = 8

print('V23 — high-recall missed-event recovery')
print('Output:', OUT_ROOT)


Mounted at /content/drive
V23 — high-recall missed-event recovery
Output: /content/drive/MyDrive/Datathon/track2_v23_high_recall_missed_event_recovery


In [ ]:

# 1. Imports / general helpers
import json, math, pickle, hashlib, copy
from collections import OrderedDict

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_file(root,name):
    p=Path(root)/name
    if p.is_file():return p
    hits=sorted(Path(root).glob('**/'+name))
    if not hits:raise FileNotFoundError(f'{name} not found under {root}')
    return hits[0]

def load_json(path):
    with open(path,encoding='utf-8') as f:return json.load(f)

def rms(x):
    x=np.asarray(x,np.float64)
    return float(np.sqrt(np.mean(x*x)+1e-20))

def si_sdr(est,ref,eps=1e-12):
    est=np.asarray(est,np.float64);ref=np.asarray(ref,np.float64)
    if est.shape!=ref.shape:raise ValueError('SI-SDR length mismatch')
    est=est-est.mean();ref=ref-ref.mean()
    ss=float(ref@ref)
    if ss<=1e-18:return np.nan
    proj=float(est@ref)*ref/(ss+eps)
    err=est-proj
    return float(min(120.,10*np.log10((float(proj@proj)+eps)/(float(err@err)+eps))))

def direct_fit_db(pred,target,eps=1e-12):
    pred=np.asarray(pred,np.float64);target=np.asarray(target,np.float64)
    return float(10*np.log10((float(target@target)+eps)/
                             (float((target-pred)@(target-pred))+eps)))

def pcm_roundtrip(x):
    x=np.clip(np.asarray(x,np.float32),-1,1-1/32768)
    return np.round(x*32768).astype(np.int16).astype(np.float32)/32768.0

def residual_change_db(a,b):
    a=np.asarray(a,np.float64);b=np.asarray(b,np.float64)
    return float(10*np.log10(max(float((a-b)@(a-b)),1e-24)/
                             max(float(b@b),1e-24)))

def event_list(v):
    if isinstance(v,dict):
        v=v.get('events',v.get('gt',[]))
    out=[]
    for e in v:
        if isinstance(e,dict):
            a=float(e['onset'] if 'onset' in e else e['start'])
            b=float(e['offset'] if 'offset' in e else e['end'])
        else:
            a,b=float(e[0]),float(e[1])
        if math.isfinite(a) and math.isfinite(b) and 0<=a<b:
            out.append((a,b))
    return out

def event_bounds_samples(ev,n):
    lo=max(0,min(n,int(round(float(ev[0])*SR))))
    hi=max(lo,min(n,int(round(float(ev[1])*SR))))
    return lo,hi

def read_mono_16k(path):
    y,sr=sf.read(path,dtype='float32',always_2d=True)
    if sr!=SR:raise ValueError(f'Expected {SR} Hz, got {sr}: {path}')
    y=y.mean(axis=1,dtype=np.float32)
    if not len(y) or not np.isfinite(y).all():raise ValueError(path)
    return y.astype(np.float32)

def index_wavs(root):
    out={p.stem:p for p in Path(root).rglob('*.wav')}
    if not out:raise FileNotFoundError(f'No WAVs under {root}')
    return out

def resample_1d(x,n):
    x=np.asarray(x,np.float32)
    if len(x)==n:return x.copy()
    if len(x)<2:return np.full(n,float(x[0]) if len(x) else 0,np.float32)
    return np.interp(np.linspace(0,1,n),np.linspace(0,1,len(x)),x).astype(np.float32)

def shift_zeropad(x,shift):
    x=np.asarray(x,np.float32);out=np.zeros_like(x)
    if shift==0:return x.copy()
    if shift>0:
        if shift<len(x):out[shift:]=x[:-shift]
    else:
        k=-shift
        if k<len(x):out[:-k]=x[k:]
    return out

def preemph(x,a=.95):
    x=np.asarray(x,np.float32)
    return np.r_[x[:1],x[1:]-float(a)*x[:-1]].astype(np.float32) if len(x) else x.copy()

def signed_corr(a,b,eps=1e-12):
    a=np.asarray(a,np.float64);b=np.asarray(b,np.float64)
    a=a-a.mean();b=b-b.mean()
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))

def cosine_abs(a,b):
    a=np.asarray(a,np.float64);b=np.asarray(b,np.float64)
    return float(abs(np.dot(a,b))/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))

def qstats(vals,prefix):
    a=np.asarray([float(v) for v in vals if v is not None and np.isfinite(v)],float)
    if not len(a):
        return {prefix+'_n':0,prefix+'_max':np.nan,prefix+'_min':np.nan,
                prefix+'_mean':np.nan,prefix+'_std':np.nan,prefix+'_gap12':np.nan}
    s=np.sort(a)[::-1]
    return {
        prefix+'_n':int(len(a)),
        prefix+'_max':float(s[0]),
        prefix+'_min':float(np.min(a)),
        prefix+'_mean':float(np.mean(a)),
        prefix+'_std':float(np.std(a)),
        prefix+'_gap12':float(s[0]-s[1]) if len(s)>1 else np.nan,
    }


In [ ]:
"""Full-clip residual-template detector. No query labels/clean audio in inference."""
from __future__ import annotations
import os, io, re, json, math, time, hashlib, pickle, zipfile, copy
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import OrderedDict, Counter
import numpy as np
import torch
import torch.nn.functional as F
import soundfile as sf

VERSION = 'residual_evidence_v5_20260917_r1'

@dataclass(frozen=True)
class MatcherConfig:
    sample_rate: int = 16000
    coarse_rate: int = 4000
    coarse_floor: float = 0.25
    template_batch: int = 32
    peaks_per_template: int = 3
    verify_topk: int = 128
    raw_radius: int = 12
    min_template_sec: float = 0.08
    max_template_sec: float = 6.0
    max_templates: int = 2000
    chunk_sec: float = 24.0
    fft_cache_mb: int = 384
    merge_iou: float = 0.65
    seed: int = 517


def digest(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(4 << 20), b''):
            h.update(b)
    return h.hexdigest()


def wave_sha(y):
    return hashlib.sha256(np.asarray(y, '<f4').tobytes()).hexdigest()


def atomic_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(tmp, p)


def atomic_pickle(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)


def event_list(v):
    if isinstance(v, dict):
        if 'events' not in v:
            raise ValueError('Prediction row has no events field.')
        v = v['events']
    out = []
    for e in v:
        if isinstance(e, dict):
            a = float(e['onset'] if 'onset' in e else e['start'])
            b = float(e['offset'] if 'offset' in e else e['end'])
        else:
            a, b = float(e[0]), float(e[1])
        if not (math.isfinite(a) and math.isfinite(b) and 0 <= a < b):
            raise ValueError(f'Invalid event: {e}')
        out.append((a, b))
    return out


def read_jsonl(path, predictions=True):
    out = {}
    with open(path, encoding='utf-8') as f:
        for n, line in enumerate(f, 1):
            if not line.strip(): continue
            r = json.loads(line)
            cid = str(r['clip_id'])
            if cid in out: raise ValueError(f'Duplicate clip_id on line {n}: {cid}')
            if predictions: event_list(r)
            out[cid] = r
    if not out: raise ValueError(f'Empty JSONL: {path}')
    return out


def source_group(cid):
    # Best-effort recording prefix, NOT a claim of verified speaker identity.
    # Example: ...844425030213767_010_LowerDibangvalley-SPECIFIC_...
    m = re.search(r'_\d+_(?:GENERIC|[^_]+-SPECIFIC)_', str(cid))
    return str(cid)[:m.start()] if m else str(cid).rsplit('_', 3)[0]


def split_bucket(group):
    return int(hashlib.sha256(('v5_bank_source_' + group).encode()).hexdigest()[:12], 16) % 10


def bandlimited_resample(y, src, dst):
    """Windowed-sinc resampling, no SciPy/librosa dependency; common grid for pairs."""
    y = np.asarray(y, np.float32)
    if src == dst: return y.copy()
    n = int(round(len(y) * dst / src))
    if n < 1: raise ValueError('Audio became empty during resampling.')
    cutoff = min(1.0, float(dst) / src) * 0.94
    radius = int(math.ceil(20 / cutoff))
    offsets = np.arange(-radius, radius + 1)
    out = np.empty(n, np.float32)
    for a in range(0, n, 2048):
        b = min(n, a + 2048)
        pos = np.arange(a, b, dtype=np.float64) * src / dst
        ix = np.floor(pos).astype(np.int64)[:, None] + offsets[None]
        distance = pos[:, None] - ix
        w = cutoff * np.sinc(cutoff * distance)
        w *= np.where(np.abs(distance) < radius, 0.5 + 0.5*np.cos(np.pi*distance/radius), 0)
        valid = (ix >= 0) & (ix < len(y))
        w *= valid
        w /= np.maximum(w.sum(1, keepdims=True), 1e-12)
        out[a:b] = np.sum(y[np.clip(ix, 0, len(y)-1)] * w, axis=1)
    return out


def load_audio(path_or_bytes, sr=16000):
    obj = io.BytesIO(path_or_bytes) if isinstance(path_or_bytes, (bytes, bytearray)) else path_or_bytes
    y, fs = sf.read(obj, dtype='float32', always_2d=True)
    y = np.mean(y, axis=1, dtype=np.float32)
    if not len(y) or not np.isfinite(y).all(): raise ValueError(f'Invalid audio: {str(path_or_bytes)[:100]}')
    return bandlimited_resample(y, int(fs), sr)


def condition_wave(y):
    """Mild pre-emphasis, identically applied to templates and query recordings."""
    y = np.asarray(y, np.float32)
    return np.concatenate([y[:1], y[1:] - 0.85*y[:-1]]).astype(np.float32)


def coarse_wave(y, cfg):
    ratio = cfg.sample_rate // cfg.coarse_rate
    if ratio * cfg.coarse_rate != cfg.sample_rate:
        raise ValueError('coarse_rate must divide sample_rate.')
    y = condition_wave(y)
    t = np.arange(-32, 33, dtype=np.float64)
    cutoff = 0.90 / ratio
    kernel = cutoff * np.sinc(cutoff*t) * np.hanning(len(t))
    kernel /= kernel.sum()
    # Odd, symmetric FIR; zero padding preserves the time origin.
    padded = np.pad(y, (32, 32))
    return np.convolve(padded, kernel, mode='valid')[::ratio].astype(np.float32)


def iou(a, b):
    inter = max(0., min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.


def closest_first_pairs(pred, ref):
    # Same explicitly named local scorer used in the user's notebooks.
    edges = []
    for i, (a,b) in enumerate(pred):
        for j, (s,e) in enumerate(ref):
            tol = max(0.20 * (e-s), 0.05)
            if abs(a-s) <= tol and abs(b-e) <= tol:
                edges.append((abs(a-s) + abs(b-e), i, j))
    used_p, used_r, pairs = set(), set(), []
    for _, i, j in sorted(edges):
        if i not in used_p and j not in used_r:
            used_p.add(i); used_r.add(j); pairs.append((i,j))
    return pairs


def score_events(pred, refs, durs, ids):
    tp = fp = fn = 0; dice = []; npred = ngt = 0
    for cid in ids:
        p, g = event_list(pred[cid]), event_list(refs[cid])
        nmatch = len(closest_first_pairs(p, g))
        tp += nmatch; fp += len(p)-nmatch; fn += len(g)-nmatch
        npred += len(p); ngt += len(g)
        n = max(1, int(round(durs[cid]/0.01)))
        pm = np.zeros(n, bool); gm = np.zeros(n, bool)
        for events, mask in ((p,pm),(g,gm)):
            for a,b in events:
                lo,hi = max(0,int(round(a/0.01))), min(n,int(round(b/0.01)))
                if hi > lo: mask[lo:hi] = True
        den = int(pm.sum()+gm.sum())
        dice.append(1. if den == 0 else 2.*np.logical_and(pm,gm).sum()/den)
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    f1 = 2.*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    d = float(np.mean(dice)) if dice else 0.
    return dict(f1=f1, precision=precision, recall=recall, dice=d, combined=f1+d,
                tp=tp, fp=fp, fn=fn, clips=len(ids), predicted_events=npred, gt_events=ngt)


def wilson_lower(success, total, z=1.96):
    if total == 0: return 0.
    p = success/total; den = 1.+z*z/total
    return (p+z*z/(2*total)-z*math.sqrt(p*(1-p)/total+z*z/(4*total*total)))/den


def build_bank(bank_ids, refs, audio_paths, clean_paths, groups, cfg, progress=lambda x, **k: x):
    records, audits, source_hashes = [], [], []
    sr = cfg.sample_rate
    for cid in progress(bank_ids, desc='BANK pairs only: noisy minus clean'):
        x = load_audio(audio_paths[cid], sr)
        s = load_audio(clean_paths[cid], sr)
        if len(x) != len(s):
            raise ValueError(f'Pair length mismatch: {cid}; no silent truncation allowed.')
        mask = np.zeros(len(x), bool)
        for a,b in refs[cid]:
            mask[max(0,int(round(a*sr))):min(len(x),int(round(b*sr)))] = True
        outside = ~mask
        # Gain alignment is estimated from BANK-only non-event samples.
        scale = float(np.dot(x[outside],s[outside])/(np.dot(s[outside],s[outside])+1e-12)) if outside.sum()>sr//10 else 1.
        scale = float(np.clip(scale,0.8,1.2))
        residual = x - scale*s
        inside_rms = float(np.sqrt(np.mean(residual[mask]**2)+1e-20)) if mask.any() else 0.
        outside_rms = float(np.sqrt(np.mean(residual[outside]**2)+1e-20)) if outside.any() else 0.
        ratio = outside_rms/max(inside_rms,1e-10)
        good = bool(inside_rms>1e-5 and ratio<=0.15)
        audits.append(dict(cid=cid, scale=scale, inside_rms=inside_rms, outside_rms=outside_rms,
                           outside_inside_ratio=ratio, used=good))
        source_hashes.append(wave_sha(x))
        if not good: continue
        for j,(a,b) in enumerate(refs[cid]):
            # Mixed overlapping events are not single-template examples.
            if any(k!=j and min(b,e)-max(a,s0)>0.005 for k,(s0,e) in enumerate(refs[cid])): continue
            if not cfg.min_template_sec <= b-a <= cfg.max_template_sec: continue
            lo,hi = int(round(a*sr)), int(round(b*sr))
            if lo < 0 or hi > len(x) + int(.02*sr): raise ValueError(f'BANK label out of audio range: {cid}')
            hi = min(hi, len(x))
            w = residual[lo:hi].astype(np.float32)
            if len(w)<16 or float(np.std(w))<1e-5: continue
            coarse = coarse_wave(w,cfg)
            coarse -= coarse.mean()
            norm = float(np.linalg.norm(coarse))
            if norm < 1e-7: continue
            raw = condition_wave(w)
            # Reject templates with effectively just one energetic quarter.
            chunks = np.array([np.sum(v*v) for v in np.array_split(raw,4)])
            if np.sum(chunks > .02*max(float(chunks.sum()),1e-10)) < 2: continue
            records.append(dict(tid=f'{cid}::{j}', source_cid=cid, group=groups[cid],
                                wave=w, raw=raw, coarse=(coarse/norm).astype(np.float32),
                                annotation_duration=b-a))
    # Exact waveform deduplication only; no fitting on SELECT/HOLD.
    unique = {}; dedup = 0
    for r in records:
        w = r['wave']/max(float(np.max(np.abs(r['wave']))),1e-9)
        key = hashlib.sha256(np.round(w*32700).astype('<i2').tobytes()).hexdigest()
        if key in unique: dedup += 1
        else: unique[key] = r
    records = sorted(unique.values(), key=lambda r: hashlib.sha256(r['tid'].encode()).hexdigest())
    capped = max(0,len(records)-cfg.max_templates)
    records = records[:cfg.max_templates]
    if not records: raise RuntimeError('No valid additive-noise templates. Stop: the waveform-template hypothesis is unsupported for these pairs.')
    return dict(version=VERSION, cfg=asdict(cfg), records=records, audits=audits,
                bank_ids=list(bank_ids), bank_groups=sorted({groups[c] for c in bank_ids}),
                source_hashes=source_hashes, deduplicated=dedup, capped=capped)


class ResidualMatcher:
    def __init__(self, bank, cfg, device=None):
        self.cfg, self.bank = cfg, bank
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.records = bank['records']
        self.source_hashes = set(bank.get('source_hashes',[]))
        self.cache = OrderedDict(); self.cache_bytes = 0
        self.signature = digest(dict(version=VERSION,cfg=asdict(cfg),
            templates=[(r['tid'], wave_sha(r['wave'])) for r in self.records]))

    def _spectrum(self, ix, nfft):
        key = (tuple(ix), nfft)
        if key in self.cache:
            z = self.cache.pop(key); self.cache[key] = z
            return z
        t = torch.zeros((len(ix),nfft),dtype=torch.float32,device=self.device)
        for j,i in enumerate(ix):
            x = self.records[i]['coarse']; t[j,:len(x)] = torch.as_tensor(x,device=self.device)
        z = torch.fft.rfft(t,n=nfft,dim=-1)
        cost = z.numel()*z.element_size(); max_bytes = self.cfg.fft_cache_mb*(1<<20)
        while self.cache and self.cache_bytes+cost > max_bytes:
            _,old = self.cache.popitem(last=False); self.cache_bytes-=old.numel()*old.element_size()
        if cost<=max_bytes: self.cache[key] = z; self.cache_bytes+=cost
        return z

    @torch.inference_mode()
    def _coarse_scan(self,y,exclude_group=None):
        x = coarse_wave(y,self.cfg); n = len(x)
        eligible = [i for i,r in enumerate(self.records)
                    if len(r['coarse'])<=n and (exclude_group is None or r['group']!=exclude_group)]
        if not eligible: return []
        mmax = max(len(self.records[i]['coarse']) for i in eligible)
        nfft = 1<<(n+mmax-2).bit_length()
        xt = torch.as_tensor(x,dtype=torch.float32,device=self.device)
        xf = torch.fft.rfft(xt,n=nfft)
        # Use float64 prefix sums to avoid variance cancellation near silence.
        xx = xt.double()
        pref = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),xx.cumsum(0)])
        pref2 = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),(xx*xx).cumsum(0)])
        starts = torch.arange(n,device=self.device)[None]
        rows = []
        for a in range(0,len(eligible),self.cfg.template_batch):
            ix = eligible[a:a+self.cfg.template_batch]
            lens = torch.tensor([len(self.records[i]['coarse']) for i in ix],device=self.device)[:,None]
            tf = self._spectrum(ix,nfft)
            dot = torch.fft.irfft(xf[None]*tf.conj(),n=nfft,dim=-1)[:,:n]
            ends = (starts+lens).clamp(max=n)
            total = pref[ends]-pref[starts]; total2 = pref2[ends]-pref2[starts]
            energy = (total2-total*total/lens).clamp(min=1e-16).float()
            q = (dot.abs()/torch.sqrt(energy)).clamp(max=1.)
            q = q.masked_fill((starts+lens>n)|(energy<1e-12),-1.)
            for _ in range(self.cfg.peaks_per_template):
                vals, pos = q.max(dim=1)
                for j,(v,p) in enumerate(zip(vals.cpu().tolist(),pos.cpu().tolist())):
                    if v>=self.cfg.coarse_floor:
                        rows.append(dict(template_index=ix[j], sample4k=int(p), coarse=float(v)))
                rad = torch.maximum(lens//2,torch.full_like(lens,int(.06*self.cfg.coarse_rate)))
                q = q.masked_fill((starts-pos[:,None]).abs()<=rad,-1.)
        return sorted(rows,key=lambda r:r['coarse'],reverse=True)[:self.cfg.verify_topk]

    def _verify(self,raw_query,proposal):
        r = self.records[proposal['template_index']]; t = r['raw']; m = len(t)
        center = proposal['sample4k']*(self.cfg.sample_rate//self.cfg.coarse_rate)
        lo,hi = max(0,center-self.cfg.raw_radius), min(len(raw_query)-m,center+self.cfg.raw_radius)
        if hi<lo: return None
        t0=t.astype(np.float64)-np.mean(t,dtype=np.float64); ten=float(t0@t0)
        if ten<1e-12: return None
        seg=raw_query[lo:hi+m].astype(np.float64)
        dots=np.correlate(seg,t0,mode='valid')
        cs=np.r_[0.,np.cumsum(seg)]; cs2=np.r_[0.,np.cumsum(seg*seg)]
        sums=cs[m:]-cs[:-m]; energies=np.maximum(cs2[m:]-cs2[:-m]-sums*sums/m,1e-15)
        corr=dots/np.sqrt(energies*ten)
        j=int(np.argmax(np.abs(corr))); start=lo+j; sign=1. if corr[j]>=0 else -1.
        qseg=raw_query[start:start+m]
        chunks=[]
        for inds in np.array_split(np.arange(m),4):
            tt=t[inds].astype(np.float64); yy=qseg[inds].astype(np.float64)
            tt-=tt.mean(); yy-=yy.mean(); et=float(tt@tt)
            if et < .015*ten: continue
            chunks.append(float(sign*np.dot(tt,yy)/math.sqrt(max(et*float(yy@yy),1e-20))))
        if len(chunks)<2: return None
        raw_corr=float(abs(corr[j])); chunk=float(np.median(chunks))
        quality=float(np.clip(min(raw_corr,chunk),0.,1.))
        return dict(onset=start/self.cfg.sample_rate,offset=(start+m)/self.cfg.sample_rate,
                    quality=quality,corr16=raw_corr,corr4k=proposal['coarse'],chunk_median=chunk,
                    template_id=r['tid'],template_source=r['source_cid'],polarity=int(sign))

    def scan(self,y,exclude_group=None):
        """Only noisy waveform and a training bank are inputs. No event/GT arguments."""
        y=np.asarray(y,np.float32)
        if not np.isfinite(y).all(): raise ValueError('Nonfinite query audio.')
        if wave_sha(y) in self.source_hashes:
            raise RuntimeError('Query waveform exactly duplicates a BANK source. Refusing self-retrieval.')
        chunk=int(self.cfg.chunk_sec*self.cfg.sample_rate)
        overlap=int((self.cfg.max_template_sec+.05)*self.cfg.sample_rate)
        if chunk<=overlap: raise ValueError('chunk_sec must exceed max_template_sec.')
        starts=list(range(0,max(1,len(y)-overlap),chunk-overlap))
        found=[]
        for a in starts:
            yy=y[a:min(len(y),a+chunk)]
            if len(yy)<int(self.cfg.min_template_sec*self.cfg.sample_rate): continue
            raw=condition_wave(yy)
            for p in self._coarse_scan(yy,exclude_group):
                v=self._verify(raw,p)
                if v is not None:
                    v['onset']+=a/self.cfg.sample_rate; v['offset']+=a/self.cfg.sample_rate
                    found.append(v)
        # Keep geometric alternatives; final threshold/NMS is applied later.
        best={}
        for r in found:
            k=(round(r['onset'],4),round(r['offset'],4),r['template_id'])
            if k not in best or r['quality']>best[k]['quality']: best[k]=r
        return sorted(best.values(),key=lambda r:r['quality'],reverse=True)


def select_matches(rows, threshold, duplicate_iou=.65):
    chosen=[]
    for r in sorted(rows,key=lambda z:z['quality'],reverse=True):
        if r['quality']<threshold: continue
        ev=(r['onset'],r['offset'])
        if any(iou(ev,(x['onset'],x['offset']))>=duplicate_iou or
               (abs(ev[0]-x['onset'])<=.02 and abs(ev[1]-x['offset'])<=.02) for x in chosen): continue
        chosen.append(r)
    return sorted(chosen,key=lambda z:(z['onset'],z['offset']))


def patch_events(base, matches, max_move=.25, minimum_iou=.50, allow_add=False, add_threshold=.85):
    """Conservative edit: only mutual one-to-one overlaps; unmatched base retained."""
    base=event_list(base); proposal=[(m['onset'],m['offset']) for m in matches]
    edges=[]
    p2b={j:[] for j in range(len(proposal))}; b2p={i:[] for i in range(len(base))}
    for j,p in enumerate(proposal):
        for i,b in enumerate(base):
            ov=iou(p,b)
            if ov>=minimum_iou and max(abs(p[0]-b[0]),abs(p[1]-b[1]))<=max_move:
                p2b[j].append(i); b2p[i].append(j); edges.append((i,j))
    out=list(base); used=set(); changed=0
    for i,j in edges:
        if len(p2b[j])==len(b2p[i])==1:
            out[i]=proposal[j]; used.add(j); changed+=int(out[i]!=base[i])
    added=0
    if allow_add:
        for j,p in enumerate(proposal):
            if j in used or matches[j]['quality']<add_threshold: continue
            # No addition overlapping or touching an existing interval.
            if any(min(p[1],b[1]) - max(p[0],b[0]) > -.04 for b in out): continue
            out.append(p); added+=1
    return sorted(out),dict(changed=changed,added=added)


class WaveSource:
    """Read WAVs from one explicit folder/ZIP without extracting or guessing IDs."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        if Path(path).is_file() and zipfile.is_zipfile(path):
            self.archive=zipfile.ZipFile(path)
            names=[i.filename for i in self.archive.infolist() if not i.is_dir() and i.filename.lower().endswith('.wav') and '__MACOSX' not in i.filename]
        elif Path(path).is_dir():
            names=[str(p) for p in Path(path).rglob('*.wav')]
        else: raise FileNotFoundError(path)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform stem in {path}: {cid}')
            self.index[cid]=name
        missing=set(required_ids)-set(self.index)
        if missing: raise ValueError(f'{path}: missing {len(missing)} required WAVs; examples {sorted(missing)[:3]}')
    def load(self,cid,sr=16000):
        loc=self.index[cid]
        return load_audio(self.archive.read(loc) if self.archive is not None else loc,sr)
    def duration(self,cid):
        loc=self.index[cid]
        x=io.BytesIO(self.archive.read(loc)) if self.archive is not None else loc
        info=sf.info(x)
        return info.frames/info.samplerate


def persist_query(matcher,y,path,exclude_group=None):
    wh=wave_sha(y); path=Path(path)
    signature=digest(dict(matcher=matcher.signature,wave=wh,exclude_group=exclude_group))
    if path.is_file():
        with open(path) as f: old=json.load(f)
        if old.get('signature')==signature: return old['proposals']
    rows=matcher.scan(y,exclude_group)
    atomic_json(path,dict(signature=signature,query_wave_sha256=wh,proposals=rows))
    return rows


def unit_tests():
    cfg=MatcherConfig(coarse_floor=.15,template_batch=2,peaks_per_template=2,verify_topk=20,fft_cache_mb=16)
    rng=np.random.default_rng(601)
    sr=cfg.sample_rate
    t=(rng.normal(0,.15,int(.36*sr))*np.hanning(int(.36*sr))).astype(np.float32)
    source=rng.normal(0,.02,int(1.8*sr)).astype(np.float32)
    a=1923; source[a:a+len(t)]+=t
    coarse=coarse_wave(t,cfg); coarse-=coarse.mean(); coarse/=np.linalg.norm(coarse)
    bank=dict(records=[dict(tid='test_template',source_cid='bank',group='g0',wave=t,raw=condition_wave(t),coarse=coarse)],source_hashes=[])
    m=ResidualMatcher(bank,cfg,device='cpu')
    got=select_matches(m.scan(source),.85)
    assert got and abs(got[0]['onset']-a/sr)<=1/sr, got
    assert abs(got[0]['offset']-(a+len(t))/sr)<=1/sr
    neg=m.scan(rng.normal(0,.02,len(source)).astype(np.float32))
    assert not select_matches(neg,.85)
    # Confirm no self-retrieval.
    m.source_hashes.add(wave_sha(source))
    try: m.scan(source)
    except RuntimeError: pass
    else: raise AssertionError('Self-retrieval guard failed.')
    refs={'x':[(1.,1.5)]}; durs={'x':2.}
    assert score_events(refs,refs,durs,['x'])['f1']==1.
    assert score_events({'x':[]},refs,durs,['x'])['fn']==1
    row=dict(onset=1.01,offset=1.51,quality=.99)
    patched,diag=patch_events([(1.,1.5)],[row])
    assert len(patched)==1 and diag['changed']==1
    patched,diag=patch_events([(1.,1.5)],[dict(onset=0.,offset=.3,quality=.99)])
    assert patched==[(1.,1.5)] and diag['added']==0
    print('PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.')
    return True

In [ ]:

# 2. Load split, validation audio and exact V6 frozen policy.
with open(V5_DIR/'frozen_split.json',encoding='utf-8') as f:
    SPLIT=json.load(f)

BANK_IDS=list(SPLIT['bank'])
SELECT_SYN=list(SPLIT['select_syn'])
SELECT_NAT=list(SPLIT.get('select_nat',[]))
HOLD_SYN=list(SPLIT.get('hold_syn',[]))
HOLD_NAT=list(SPLIT.get('hold_nat',[]))
GROUPS=dict(SPLIT['group_key'])

with open(VAL_CACHE,'rb') as f:
    raw=pickle.load(f)
records=raw if isinstance(raw,list) else [dict(v,clip_id=k) for k,v in raw.items()]
REFS={}
for r in records:
    REFS[str(r['clip_id'])]=event_list(r['gt'])
del raw,records

options=[Path(VAL_ROOT)] if VAL_ROOT is not None else VAL_ROOT_CANDIDATES
complete=[p for p in options if (p/'syntheticNoiseAudio').is_dir() and
          (p/'syntheticCleanRefAudio').is_dir()]
if not complete:raise FileNotFoundError('Set VAL_ROOT to released validation folder.')
VAL_ROOT=complete[0]

SYN_NOISY=index_wavs(VAL_ROOT/'syntheticNoiseAudio')
SYN_CLEAN=index_wavs(VAL_ROOT/'syntheticCleanRefAudio')
NAT_NOISY=index_wavs(VAL_ROOT/'naturalNoisyAudio') if (VAL_ROOT/'naturalNoisyAudio').is_dir() else {}

if not V6_RUN.is_dir():
    exact=[p for p in sorted(V6_ROOT.glob('run_*')) if V6_RUN_KEY in p.name]
    if not exact:raise FileNotFoundError('Exact V6 run not found.')
    V6_RUN=exact[0]

FROZEN=load_json(V6_RUN/'frozen_before_hold.json')
V6_METHOD=str(FROZEN['method'])
V6_THRESHOLD=float(FROZEN['threshold'])
V6_POLICY_ID=V6_METHOD+'__q'+format(V6_THRESHOLD,'.4f')
RCFG=dict(FROZEN.get('recon_config',{}))

V6_ALL=pd.read_csv(V6_RUN/'select_per_clip_waveform_metrics.csv')
V6_SYN=V6_ALL[(V6_ALL.policy_id==V6_POLICY_ID)&(V6_ALL.domain=='synthetic')].copy()
V6_NAT=V6_ALL[(V6_ALL.policy_id==V6_POLICY_ID)&(V6_ALL.domain=='natural')].copy()

if V6_SYN.empty:
    raise RuntimeError('Frozen V6 policy not found in per-clip SELECT metrics.')

V6_SYN=V6_SYN.rename(columns={
    'sisdr':'v6_sisdr','gain_db':'v6_gain_db','changed':'v6_changed',
    'accepted_templates':'v6_accepted_templates','change_db':'v6_change_db',
    'would_clip_rejections':'v6_clip_rejections'
})
if len(V6_NAT):
    V6_NAT=V6_NAT.rename(columns={
        'changed':'v6_changed','accepted_templates':'v6_accepted_templates',
        'change_db':'v6_change_db','would_clip_rejections':'v6_clip_rejections'
    })

print('Validation root:',VAL_ROOT)
print('V6 frozen policy:',V6_METHOD,V6_THRESHOLD)
print('SELECT synthetic:',len(V6_SYN),'natural:',len(V6_NAT))
print('HOLD synthetic:',len(HOLD_SYN),'natural:',len(HOLD_NAT))
print('V6 mean SI-SDR:',float(V6_SYN.v6_sisdr.mean()))


Validation root: /content/drive/MyDrive/PHD/Datathon/validation
V6 frozen policy: block_irls 0.55
SELECT synthetic: 128 natural: 338
HOLD synthetic: 141 natural: 408
V6 mean SI-SDR: 31.735548697034424


In [ ]:
"""Track-2 direct residual reconstruction. Inference never accepts a clean reference."""
from __future__ import annotations
import copy, hashlib, io, json, math, os, re, shutil, time, unicodedata, wave, zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import soundfile as sf

RECON_VERSION = 'track2_v6_direct_residual_20260918_r1'

@dataclass(frozen=True)
class ReconConfig:
    sample_rate: int = 16000
    max_components: int = 16
    duplicate_iou: float = 0.65
    alignment_radius: int = 4
    min_abs_gain: float = 0.01
    max_abs_gain: float = 8.0
    max_condition: float = 1e7
    max_gain_disagreement: float = 0.50
    max_removal_energy_ratio: float = 1.15
    max_output_energy_ratio: float = 1.05
    max_natural_modified_fraction: float = 0.05
    ridge_relative: float = 1e-9
    whitening_floor_fraction: float = 0.02
    gls_iterations: int = 3
    natural_damage_db: float = -24.0


def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x


def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)


def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()


def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')


def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0


def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()


def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()


def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a


def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()


class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()


def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))


def residual_change_db(a,b):
    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64)
    return float(10*np.log10(max(float((a-b)@(a-b)),1e-24)/max(float(b@b),1e-24)))


def fit_centered(A,y,weights=None,ridge=1e-9):
    A=np.asarray(A,np.float64); y=np.asarray(y,np.float64)
    if A.ndim==1: A=A[:,None]
    w=np.ones(len(y),np.float64) if weights is None else np.asarray(weights,np.float64)
    w=np.clip(w,0,None); w=w/max(float(w.mean()),1e-30)
    am=(A*w[:,None]).sum(0)/max(w.sum(),1e-30); ym=float((y*w).sum()/max(w.sum(),1e-30))
    B=A-am; v=y-ym
    gram=B.T@(w[:,None]*B); rhs=B.T@(w*v)
    diag=np.diag(gram); scale=np.sqrt(np.maximum(diag,1e-30))
    G=gram/scale[:,None]/scale[None,:]; r=rhs/scale
    cond=float(np.linalg.cond(G))
    g=np.linalg.solve(G+ridge*np.eye(G.shape[0]),r)/scale
    return g,cond


def emphasis(A,y):
    return A[1:]-.90*A[:-1],y[1:]-.90*y[:-1]


def residual_psd(residual,nfft,sr=16000):
    nwin=min(512,len(residual))
    if nwin<16: return np.ones(nfft//2+1)
    hop=max(1,nwin//2); win=np.hanning(nwin)
    starts=range(0,len(residual)-nwin+1,hop)
    # Median-Welch PSD limits unusually loud speech frames without target labels.
    frames=np.stack([residual[a:a+nwin]*win for a in starts])
    p=np.median(np.abs(np.fft.rfft(frames,n=512,axis=1))**2,axis=0)
    p=np.convolve(np.pad(p,(3,3),mode='edge'),np.ones(7)/7,mode='valid')
    return np.interp(np.linspace(0,sr/2,nfft//2+1),np.linspace(0,sr/2,len(p)),p)


def fit_spectral(A,y,cfg):
    g,_=fit_centered(*emphasis(A,y),ridge=cfg.ridge_relative)
    nfft=1<<(len(y)-1).bit_length()
    AF=np.fft.rfft(A,n=nfft,axis=0); YF=np.fft.rfft(y,n=nfft)
    freq=np.linspace(0,cfg.sample_rate/2,len(YF))
    cond=1.
    for _ in range(cfg.gls_iterations):
        psd=residual_psd(y-A@g,nfft,cfg.sample_rate)
        positive=psd[psd>0]
        floor=max(float(np.median(positive))*cfg.whitening_floor_fraction if len(positive) else 0.,1e-20)
        w=1./np.maximum(psd,floor); w/=max(float(np.median(w)),1e-30)
        w=np.clip(w,1e-3,1e3); w[freq<70.]=0.
        # One-sided real FFT Parseval weights.
        w[1:-1]*=2.
        gram=(AF.conj().T@(w[:,None]*AF)).real
        rhs=(AF.conj().T@(w*YF)).real
        scale=np.sqrt(np.maximum(np.diag(gram),1e-30))
        G=gram/scale[:,None]/scale[None,:]
        cond=float(np.linalg.cond(G))
        g=np.linalg.solve(G+cfg.ridge_relative*np.eye(len(g)),rhs/scale)/scale
    return g,cond


def estimate_gains(A,y,polarity,method,cfg):
    if method=='stored_gain': return np.asarray(polarity,np.float64),1.
    if method=='mixture_ols': return fit_centered(A,y,ridge=cfg.ridge_relative)
    P,v=emphasis(A,y)
    if method=='preemphasis_ols': return fit_centered(P,v,ridge=cfg.ridge_relative)
    if method=='spectral_gls': return fit_spectral(A,y,cfg)
    if method=='block_irls':
        g,cond=fit_centered(P,v,ridge=cfg.ridge_relative)
        block=max(64,int(.030*cfg.sample_rate))
        idx=np.arange(len(v))//block
        for _ in range(cfg.gls_iterations):
            residual=v-P@g
            counts=np.bincount(idx); power=np.bincount(idx,weights=residual**2)/counts
            floor=max(float(np.median(power))*.05,1e-15)
            bw=1./np.maximum(power,floor); bw/=max(float(np.median(bw)),1e-30)
            weights=np.clip(bw,0.02,50.)[idx]
            g,cond=fit_centered(P,v,weights,cfg.ridge_relative)
        return g,cond
    raise ValueError(f'Unknown deployable gain method: {method}')


def gain_disagreement(A,y,g,cfg):
    # Two interleaved block fits: agreement diagnostic, not a probability of correctness.
    P,v=emphasis(A,y); block=max(64,int(.020*cfg.sample_rate))
    parity=(np.arange(len(v))//block)%2
    folds=[]
    for f in (0,1):
        mask=parity==f
        if mask.sum()<4*A.shape[1] or np.any((P[mask]**2).sum(0)<1e-14): return math.inf
        try: gg,cc=fit_centered(P[mask],v[mask],ridge=cfg.ridge_relative)
        except np.linalg.LinAlgError: return math.inf
        if cc>cfg.max_condition: return math.inf
        folds.append(gg)
    return float(np.max(np.abs(folds[0]-folds[1])/np.maximum(np.abs(g),.025)))


def refine_placement(y,t,start,radius=4):
    """Integer-sample correlation refinement. The raw template is never faded/stretched."""
    t=np.asarray(t,np.float64); dt=t[1:]-.9*t[:-1]; dt=dt-dt.mean()
    ten=float(dt@dt)
    if ten<1e-18: return start
    best=(-math.inf,start)
    for a in range(max(0,start-radius),min(len(y)-len(t),start+radius)+1):
        seg=np.asarray(y[a:a+len(t)],np.float64); ds=seg[1:]-.9*seg[:-1]; ds-=ds.mean()
        q=abs(float(ds@dt))/math.sqrt(max(ten*float(ds@ds),1e-30))
        # Stable tie-breaking favours the verified V5 start.
        score=q-1e-12*abs(a-start)
        if score>best[0]: best=(score,a)
    return best[1]


def interval_iou(a,b):
    inter=max(0,min(a[1],b[1])-max(a[0],b[0])); union=max(a[1],b[1])-min(a[0],b[0])
    return inter/union if union>0 else 0.


def prepared_matches(y,proposals,bank_by_id,cfg):
    """Prepare ALL verified alternatives once; threshold/NMS are applied afterwards."""
    out=[]
    for p in proposals:
        tid=p['template_id']
        if tid not in bank_by_id: raise ValueError(f'Unknown cached template {tid}; wrong bank.')
        t=np.asarray(bank_by_id[tid]['wave'],np.float64)
        start=int(round(float(p['onset'])*cfg.sample_rate))
        if start<0 or start+len(t)>len(y): continue
        if abs((float(p['offset'])-float(p['onset']))*cfg.sample_rate-len(t))>2:
            raise ValueError('Match duration is incompatible with frozen template length.')
        a=refine_placement(y,t,start,cfg.alignment_radius)
        out.append(dict(start=int(a),end=int(a+len(t)),quality=float(p['quality']),
                        polarity=int(p['polarity']),template_id=tid,wave=t))
    return out


def select_evidence(prepared,threshold,cfg):
    keep=[]
    for r in sorted(prepared,key=lambda q:q['quality'],reverse=True):
        if r['quality']<threshold: continue
        if any(interval_iou((r['start'],r['end']),(v['start'],v['end']))>=cfg.duplicate_iou for v in keep): continue
        keep.append(r)
        if len(keep)>=cfg.max_components: break
    return sorted(keep,key=lambda q:q['start'])


def overlap_components(rows):
    groups=[]
    for r in sorted(rows,key=lambda q:q['start']):
        if groups and r['start']<max(x['end'] for x in groups[-1]): groups[-1].append(r)
        else: groups.append([r])
    return groups


def component_matrix(y,rows):
    lo=max(0,min(r['start'] for r in rows)-32)
    hi=min(len(y),max(r['end'] for r in rows)+32)
    A=np.zeros((hi-lo,len(rows)),np.float64)
    for j,r in enumerate(rows): A[r['start']-lo:r['end']-lo,j]=r['wave']
    return lo,hi,A,np.asarray(y[lo:hi],np.float64)


def reconstruct(y,prepared,policy,cfg):
    """Deployable path: input mixture, template matches, frozen policy. NO references."""
    y=np.asarray(y,np.float64)
    if y.ndim!=1 or not len(y) or not np.isfinite(y).all(): raise ValueError('Invalid noisy waveform.')
    selected=select_evidence(prepared,float(policy['threshold']),cfg)
    z=y.copy(); log=[]; accepted=[]
    for rows in overlap_components(selected):
        lo,hi,A,v=component_matrix(y,rows)
        why=None; cond=None; disagreement=None; gains=None
        try:
            gains,cond=estimate_gains(A,v,[r['polarity'] for r in rows],policy['method'],cfg)
            if not np.isfinite(gains).all() or cond>cfg.max_condition: why='ill_conditioned_fit'
            if why is None and (np.any(np.abs(gains)<cfg.min_abs_gain) or np.any(np.abs(gains)>cfg.max_abs_gain)):
                why='gain_out_of_bounds'
            if why is None and any(float(g)*r['polarity']<=0 for g,r in zip(gains,rows)): why='gain_polarity_mismatch'
            if why is None:
                disagreement=gain_disagreement(A,v,gains,cfg)
                if disagreement>cfg.max_gain_disagreement: why='inconsistent_gain'
            noise=A@gains if gains is not None else np.zeros(len(v))
            removed_ratio=float(noise@noise)/max(float(v@v),1e-20)
            candidate=v-noise
            if why is None and removed_ratio>cfg.max_removal_energy_ratio: why='excessive_removal_energy'
            if why is None and float(candidate@candidate)>cfg.max_output_energy_ratio*max(float(v@v),1e-20): why='output_energy_increase'
            if why is None and (candidate.min()<-1. or candidate.max()>32767./32768.+1e-8): why='would_clip_pcm16'
        except (np.linalg.LinAlgError,FloatingPointError,ValueError) as e:
            why=f'numerical_fit_rejected:{type(e).__name__}'
        if why is None:
            # Reconstruct from ORIGINAL mixture, once. No generic mask or second fade.
            z[lo:hi]-=noise
            accepted.extend(dict(template_id=r['template_id'],start=r['start'],end=r['end'],gain=float(g),quality=r['quality'])
                            for g,r in zip(gains,rows))
        log.append(dict(template_ids=[r['template_id'] for r in rows],accepted=why is None,reason=why,
                        condition=cond,gain_disagreement=disagreement,
                        gains=None if gains is None else gains.tolist()))
    q=pcm_roundtrip(z); base=pcm_roundtrip(y)
    return q,dict(method=policy['method'],threshold=float(policy['threshold']),
                  proposed=len(prepared),selected=len(selected),accepted_templates=len(accepted),
                  changed=bool(np.any(q!=base)),change_db=residual_change_db(q,base),components=log,matches=accepted)


def reference_assisted_diagnostic(y,clean,prepared,policy,cfg):
    """DIAGNOSTIC ONLY. Same noisy-only matches, but clean is a nuisance regressor for ideal gains.
    Never exported as audio. Does NOT search using clean/GT and is NOT a deployable score.
    """
    y=np.asarray(y,np.float64); clean=np.asarray(clean,np.float64)
    z=y.copy(); nfit=0
    for rows in overlap_components(select_evidence(prepared,policy['threshold'],cfg)):
        lo,hi,A,v=component_matrix(y,rows)
        B=np.column_stack((A,clean[lo:hi]))
        try:
            g,_=fit_centered(B,v,ridge=1e-12)
            if np.isfinite(g).all(): z[lo:hi]-=A@g[:-1]; nfit+=len(rows)
        except np.linalg.LinAlgError: pass
    return dict(reference_assisted_sisdr_float=si_sdr(z,clean),
                reference_assisted_sisdr_pcm16=si_sdr(pcm_roundtrip(z),clean),fitted_templates=nfit)


TAG_RE=re.compile(r'</?[^<>]*>|\[[^\[\]]*\]')
def normalize_text(text):
    s=TAG_RE.sub(' ',str(text))
    s=''.join(' ' if unicodedata.category(c).startswith('P') else c for c in s)
    return ' '.join(s.lower().split())


def word_errors(reference,hypothesis):
    a=normalize_text(reference).split(); b=normalize_text(hypothesis).split()
    row=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        nxt=[i]
        for j,y in enumerate(b,1): nxt.append(min(nxt[-1]+1,row[j]+1,row[j-1]+(x!=y)))
        row=nxt
    return row[-1],len(a)


def pooled_wer(ids,refs,transcripts):
    errors=words=0
    for cid in ids:
        e,n=word_errors(refs[cid],transcripts[cid]['text']); errors+=e; words+=n
    if words==0: raise ValueError('No reference words: real WER cannot be evaluated.')
    return dict(wer=errors/words,errors=errors,words=words,clips=len(ids))


def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')


class HashASR:
    """Model-revision + exact WAV byte hash cache. Never reuses an ID-only transcript."""
    def __init__(self,model,model_signature,cache_dir,batch_size=8):
        self.model=model; self.model_signature=model_signature
        self.cache_dir=Path(cache_dir); self.cache_dir.mkdir(parents=True,exist_ok=True)
        self.batch_size=batch_size
    def _call(self,paths):
        try:
            hs=self.model.transcribe([str(p) for p in paths],return_hypotheses=True)
            if isinstance(hs,(list,tuple)) and len(hs)==len(paths): return [hyp_text(h) for h in hs]
            if len(paths)==1: return [hyp_text(hs)]
        except Exception:
            if len(paths)==1: raise
        # A cardinality mismatch never silently truncates zip(...).
        return [self._call([p])[0] for p in paths]
    def transcribe(self,paths,progress=lambda x,**k:x):
        out={}; todo={}
        for cid,path in paths.items():
            sh=sha_file(path); key=sha_json(dict(model=self.model_signature,wav=sh))
            dest=self.cache_dir/(key+'.json')
            rec=None
            if dest.is_file():
                with open(dest,encoding='utf-8') as f: r=json.load(f)
                if r.get('model_signature')==self.model_signature and r.get('wav_sha256')==sh and isinstance(r.get('text'),str): rec=r
            if rec is not None: out[cid]=rec
            else:
                if key not in todo: todo[key]=dict(path=str(path),sha=sh,ids=[],dest=dest)
                todo[key]['ids'].append(cid)
        jobs=list(todo.values())
        print(f'ASR exact-hash hits: {len(out)} | unique WAVs to decode: {len(jobs)}')
        for a in progress(range(0,len(jobs),self.batch_size),desc='SraVaani final-waveform ASR'):
            ch=jobs[a:a+self.batch_size]; texts=self._call([j['path'] for j in ch])
            if len(texts)!=len(ch): raise RuntimeError('ASR cardinality mismatch.')
            for job,text in zip(ch,texts):
                # Empty output is allowed only when the real recognizer returned it.
                rec=dict(text=text,wav_sha256=job['sha'],model_signature=self.model_signature)
                put_json(job['dest'],rec)
                for cid in job['ids']: out[cid]=rec
        if set(out)!=set(paths): raise RuntimeError('Incomplete ASR coverage.')
        return out


def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


In [ ]:

# 3. Exact frozen V6 reconstruction setup.
with open(V5_DIR/'frozen_bank.pkl','rb') as f:
    V5_BANK=pickle.load(f)
V5_BANK_BY_ID={str(r['tid']):r for r in V5_BANK['records']}

RCFG_EXACT=ReconConfig(**FROZEN['recon_config'])
V6_POLICY=dict(method=V6_METHOD,threshold=V6_THRESHOLD)

if RCFG_EXACT.sample_rate!=SR:
    raise ValueError('V6 sample-rate mismatch.')

def load_v6_validation_proposals(cid):
    """Use only proposal caches produced/reused by the exact V6 run."""
    for p in [
        V5_DIR/'queries'/(str(cid)+'.json'),
        V6_RUN/'queries'/(str(cid)+'.json'),
    ]:
        if p.is_file():
            obj=load_json(p)
            rows=obj.get('proposals',obj if isinstance(obj,list) else None)
            if isinstance(rows,list):
                return rows
    raise FileNotFoundError(
        f'No cached V6 validation proposals for {cid}. '
        'The scored V6 run should already have generated these.'
    )

def exact_v6_validation(cid,y):
    props=load_v6_validation_proposals(cid)
    prep=prepared_matches(y,props,V5_BANK_BY_ID,RCFG_EXACT)
    z,info=reconstruct(y,prep,V6_POLICY,RCFG_EXACT)
    return z,info

def v6_residual_from_matches(n,matches):
    total=np.zeros(int(n),np.float32)
    pieces=[]
    for k,m in enumerate(matches):
        tid=str(m['template_id'])
        if tid not in V5_BANK_BY_ID:
            raise KeyError(f'Unknown V6 accepted template: {tid}')
        lo=int(m['start']);hi=int(m['end'])
        t=np.asarray(V5_BANK_BY_ID[tid]['wave'],np.float32)
        if hi-lo!=len(t):
            raise ValueError(f'Accepted-match/template duration mismatch: {tid}')
        pred=(float(m['gain'])*t).astype(np.float32)
        total[lo:hi]+=pred
        pieces.append(dict(
            accepted_index=k,template_id=tid,start=lo,end=hi,
            gain=float(m['gain']),quality=float(m['quality']),
            v6_pred=pred
        ))
    return total,pieces

def assert_v6_reconstruction(y,z_v6,matches,cid):
    total,_=v6_residual_from_matches(len(y),matches)
    check=pcm_roundtrip(np.asarray(y,np.float64)-total.astype(np.float64))

    # IMPORTANT: canonical_pcm() returns INT16 PCM COUNTS, not normalized
    # floating-point amplitudes. Therefore one PCM16 LSB is exactly 1 count
    # in this comparison domain (not 1/32768).
    #
    # Cast to int32 before subtraction so an INT16 difference cannot overflow.
    a=canonical_pcm(check).astype(np.int32)
    b=canonical_pcm(z_v6).astype(np.int32)

    max_lsb=int(np.max(np.abs(a-b))) if len(a) else 0

    # Rebuilding accepted residuals from saved float gains/templates can differ
    # by one final PCM16 rounding count. <=1 count is bit-depth-equivalent.
    if max_lsb>1:
        idx=int(np.argmax(np.abs(a-b)))
        raise RuntimeError(
            f'Accepted matches do not reproduce V6 within one PCM16 LSB for {cid}; '
            f'max diff={max_lsb} PCM count(s) at sample {idx}.'
        )
    return total

print('Exact ReconConfig:',RCFG_EXACT)
print('Exact V6 policy:',V6_POLICY)
print('Frozen templates:',len(V5_BANK_BY_ID))


Exact ReconConfig: ReconConfig(sample_rate=16000, max_components=16, duplicate_iou=0.65, alignment_radius=4, min_abs_gain=0.01, max_abs_gain=8.0, max_condition=10000000.0, max_gain_disagreement=0.5, max_removal_energy_ratio=1.15, max_output_energy_ratio=1.05, max_natural_modified_fraction=0.05, ridge_relative=1e-09, whitening_floor_fraction=0.02, gls_iterations=3, natural_damage_db=-24.0)
Exact V6 policy: {'method': 'block_irls', 'threshold': 0.55}
Frozen templates: 1349


In [ ]:

# 4. Build exact BANK residual-event library, as in V13.
class PairStore:
    def __init__(self,max_items=32):
        self.max_items=max_items;self.cache=OrderedDict()
    def get(self,cid):
        if cid in self.cache:
            v=self.cache.pop(cid);self.cache[cid]=v;return v
        y=read_mono_16k(SYN_NOISY[cid])
        s=read_mono_16k(SYN_CLEAN[cid])
        union=np.zeros(len(y),np.float32)
        for ev in REFS[cid]:
            lo,hi=event_bounds_samples(ev,len(y))
            union[lo:hi]=1
        outside=union<.5
        scale=float(np.dot(y[outside],s[outside])/
                    (np.dot(s[outside],s[outside])+1e-12)) if outside.sum()>SR//10 else 1.
        scale=float(np.clip(scale,.8,1.2))
        clean=(scale*s).astype(np.float32)
        res=(y-clean).astype(np.float32)
        v=(y.astype(np.float32),clean,res,union,scale)
        self.cache[cid]=v
        while len(self.cache)>self.max_items:self.cache.popitem(last=False)
        return v

PAIR_STORE=PairStore()

def residual_event_wave(cid,j):
    _,_,r,_,_=PAIR_STORE.get(cid)
    lo,hi=event_bounds_samples(REFS[cid][j],len(r))
    return r[lo:hi].copy()

rows=[]
for cid in tqdm(BANK_IDS,desc='BANK residual-event catalog'):
    _,_,res,_,_=PAIR_STORE.get(cid)
    for j,ev in enumerate(REFS[cid]):
        lo,hi=event_bounds_samples(ev,len(res))
        w=res[lo:hi]
        if hi-lo>=64 and rms(w)>=1e-6:
            rows.append(dict(
                event_id=f'{cid}__e{j}',cid=cid,event_index=j,
                group=GROUPS[cid],duration=(hi-lo)/SR
            ))

BANK_EVENTS=pd.DataFrame(rows)
BANK_EVENT_INDEX={str(r.event_id):i for i,r in enumerate(BANK_EVENTS.itertuples(index=False))}

def bank_event_wave(i):
    r=BANK_EVENTS.iloc[int(i)]
    return residual_event_wave(str(r.cid),int(r.event_index))

print('BANK residual events:',len(BANK_EVENTS))


BANK residual-event catalog:   0%|          | 0/426 [00:00<?, ?it/s]

BANK residual events: 1352


In [ ]:

# 5. V13 retriever: exact feature/ranking recipe, now applied to arbitrary V6 windows.
def unit_z(x):
    x=np.asarray(x,np.float64);x=x-x.mean()
    return (x/max(np.linalg.norm(x),1e-12)).astype(np.float32)

def time_feature(w,bins=TIME_BINS,a=0.):
    z=preemph(w,a) if a else np.asarray(w,np.float32)
    return unit_z(resample_1d(z,bins))

def env_feature(w,bins=ENV_BINS):
    return unit_z(np.abs(resample_1d(np.asarray(w,np.float32),bins)))

def spectral_feature(w,bins=SPEC_BINS):
    w=preemph(np.asarray(w,np.float32),.95)
    if len(w)<32:w=np.pad(w,(0,32-len(w)))
    nfft=1<<int(np.ceil(np.log2(min(max(len(w),256),16384))))
    z=np.log1p(np.abs(np.fft.rfft(w*np.hanning(len(w)),n=nfft)))
    e=np.linspace(0,len(z),bins+1).astype(int)
    p=np.array([z[e[i]:max(e[i]+1,e[i+1])].mean() for i in range(bins)])
    return unit_z(p)

BANK_TIME=np.stack([time_feature(bank_event_wave(i),TIME_BINS,0.) for i in range(len(BANK_EVENTS))])
BANK_PRE=np.stack([time_feature(bank_event_wave(i),TIME_BINS,.95) for i in range(len(BANK_EVENTS))])
BANK_SPEC=np.stack([spectral_feature(bank_event_wave(i)) for i in range(len(BANK_EVENTS))])
BANK_ENV=np.stack([env_feature(bank_event_wave(i)) for i in range(len(BANK_EVENTS))])
BANK_DUR=BANK_EVENTS.duration.to_numpy(float)
BANK_GROUPS=BANK_EVENTS.group.astype(str).to_numpy()

CHEAP_RECIPES={
    'balanced':dict(time=.22,pre=.36,spec=.27,env=.10,dur=.05),
    'preemph':dict(time=.10,pre=.53,spec=.24,env=.08,dur=.05),
    'spectral':dict(time=.10,pre=.28,spec=.47,env=.10,dur=.05),
    'temporal':dict(time=.32,pre=.43,spec=.12,env=.08,dur=.05),
}

FINE_RECIPES={
    'balanced':dict(cheap=.08,raw=.08,p85=.13,p95=.22,chunk=.20,stable=.06,
                    fir_corr=.15,spec=.05,fitfeat=.03),
    'pre_chunk':dict(cheap=.06,raw=.07,p85=.17,p95=.27,chunk=.27,stable=.07,
                     fir_corr=.05,spec=.03,fitfeat=.01),
    'fir_heavy':dict(cheap=.07,raw=.05,p85=.10,p95=.18,chunk=.14,stable=.06,
                     fir_corr=.29,spec=.05,fitfeat=.06),
    'v6_plus':dict(cheap=.05,raw=.15,p85=.24,p95=.17,chunk=.29,stable=.04,
                   fir_corr=.03,spec=.02,fitfeat=.01),
}

# Prefer the exact V13 frozen choices if its decision report exists.
CHEAP_RECIPE=DEFAULT_CHEAP_RECIPE
SHORTLIST_K=DEFAULT_SHORTLIST_K
FINE_RECIPE=DEFAULT_FINE_RECIPE
try:
    v13rep=load_json(find_file(V13_ROOT,'decision_report.json'))
    CHEAP_RECIPE=str(v13rep.get('chosen_cheap_recipe',CHEAP_RECIPE))
    SHORTLIST_K=int(v13rep.get('shortlist_k',SHORTLIST_K))
    FINE_RECIPE=str(v13rep.get('chosen_fine_recipe',FINE_RECIPE))
except Exception as e:
    print('Using V13 defaults:',e)

def cheap_components(q,dur):
    qt=time_feature(q,a=0.);qp=time_feature(q,a=.95)
    qs=spectral_feature(q);qe=env_feature(q)
    return dict(
        time=np.abs(BANK_TIME@qt),
        pre=np.abs(BANK_PRE@qp),
        spec=np.clip((BANK_SPEC@qs+1)/2,0,1),
        env=np.clip((BANK_ENV@qe+1)/2,0,1),
        dur=np.exp(-np.abs(np.log((BANK_DUR+1e-4)/(dur+1e-4)))/.35)
    )

def cheap_score(c,recipe=CHEAP_RECIPE):
    w=CHEAP_RECIPES[recipe]
    return sum(float(w[k])*c[k] for k in w)

def scalar_fit(t,q):
    t=np.asarray(t,np.float64);q=np.asarray(q,np.float64)
    g=float(np.dot(t,q)/(np.dot(t,t)+1e-12))
    return (g*t).astype(np.float32),g

def chunk_consistency_v13(t,q,sign,chunks=4):
    vals=[];g=[]
    for ind in np.array_split(np.arange(len(t)),chunks):
        if len(ind)<16:continue
        a=np.asarray(t[ind],float);b=np.asarray(q[ind],float)
        ac=a-a.mean();bc=b-b.mean()
        ea=ac@ac;eb=bc@bc
        if ea<1e-12 or eb<1e-12:continue
        vals.append(float(sign*(ac@bc)/math.sqrt(ea*eb)))
        g.append(float((a@b)/(a@a+1e-12)))
    if not vals:return 0.,0.
    ga=np.asarray(g);med=np.median(ga);mad=np.median(np.abs(ga-med))
    return float(np.clip(np.median(vals),0,1)),float(np.clip(np.exp(-mad/(abs(med)+1e-4)),0,1))

def fir_design(x,taps=FIR_TAPS):
    h=taps//2
    return np.stack([shift_zeropad(x,d) for d in range(-h,h+1)],1).astype(np.float64)

def fir_fit_noisy(t,q):
    X=fir_design(np.asarray(t,np.float32),FIR_TAPS)
    q=np.asarray(q,np.float32)
    if len(q)>FIR_MAX_SAMPLES:
        ix=np.linspace(0,len(q)-1,FIR_MAX_SAMPLES).astype(int)
        Xs=X[ix];qs=q[ix].astype(float)
    else:
        Xs=X;qs=q.astype(float)
    G=Xs.T@Xs
    ridge=FIR_RIDGE_REL*(np.trace(G)/max(1,G.shape[0])+1e-9)
    A=G+ridge*np.eye(G.shape[0]);b=Xs.T@qs
    try:h=np.linalg.solve(A,b)
    except np.linalg.LinAlgError:h=np.linalg.lstsq(A,b,rcond=None)[0]
    return (X@h).astype(np.float32),h.astype(np.float32)

def fine_noisy_metrics(q,c,cheap=0.):
    q=np.asarray(q,np.float32);c=np.asarray(c,np.float32)
    warped=resample_1d(c,len(q))
    q85=preemph(q,.85);q95=preemph(q,.95)
    best=None
    for sh in FINE_SHIFT_GRID:
        t=shift_zeropad(warped,sh)
        cr=signed_corr(t,q);sign=1 if cr>=0 else -1
        raw=abs(cr)
        p85=abs(signed_corr(preemph(t,.85),q85))
        p95=abs(signed_corr(preemph(t,.95),q95))
        chunk,stable=chunk_consistency_v13(t,q,sign)
        base=.18*raw+.34*p95+.20*p85+.23*chunk+.05*stable
        if best is None or base>best['base']:
            best=dict(base=base,shift=sh,t=t,raw=raw,p85=p85,p95=p95,
                      chunk=chunk,stable=stable)
    t=best.pop('t')
    sp,g=scalar_fit(t,q)
    fp,h=fir_fit_noisy(t,q)
    best.update(
        cheap=float(cheap),
        scalar_corr=cosine_abs(sp,q),
        fir_corr=cosine_abs(fp,q),
        spec=float(np.clip((spectral_feature(t)@spectral_feature(q)+1)/2,0,1)),
        fitfeat=float(np.tanh(max(0,direct_fit_db(fp,q))/12)),
        scalar_gain=float(g)
    )
    return best

def fine_score(m,recipe=FINE_RECIPE):
    return float(sum(float(w)*float(m[k]) for k,w in FINE_RECIPES[recipe].items()))

def retrieve_instance(q,exclude_group=None):
    q=np.asarray(q,np.float32)
    cs=cheap_score(cheap_components(q,len(q)/SR),CHEAP_RECIPE).astype(float)
    if exclude_group is not None:
        cs[BANK_GROUPS==str(exclude_group)]=-np.inf
    order=np.argsort(-cs)[:SHORTLIST_K]
    ranked=[]
    for bi in order:
        if not np.isfinite(cs[bi]):continue
        m=fine_noisy_metrics(q,bank_event_wave(int(bi)),float(cs[bi]))
        ranked.append((int(bi),m,fine_score(m,FINE_RECIPE)))
    ranked.sort(key=lambda z:z[2],reverse=True)
    if not ranked:return None
    top=ranked[0]
    second=ranked[1][2] if len(ranked)>1 else np.nan
    return dict(
        bank_index=top[0],
        candidate_event_id=str(BANK_EVENTS.iloc[top[0]].event_id),
        score=float(top[2]),
        score_gap=float(top[2]-second) if np.isfinite(second) else np.nan,
        metrics=top[1],
    )

print('V13 localized retrieval:',CHEAP_RECIPE,'K=',SHORTLIST_K,'fine=',FINE_RECIPE)


V13 localized retrieval: balanced K= 32 fine= fir_heavy


In [ ]:

# 6. Build the V22-targeted high-recall recovery fingerprint bank.
if not V22_MISSED_CSV.is_file():
    raise FileNotFoundError(
        f'{V22_MISSED_CSV}\nRun the V22 missed-event diagnostic first.'
    )

V22=pd.read_csv(V22_MISSED_CSV)

def bool_series(s):
    if s.dtype==bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true','1','yes','y'])

if 'noisy_retrieved_exact' not in V22.columns:
    raise ValueError('V22 CSV lacks noisy_retrieved_exact.')

V22['noisy_retrieved_exact']=bool_series(V22['noisy_retrieved_exact'])
target_ids=sorted(set(
    V22.loc[
        V22.noisy_retrieved_exact &
        V22.noisy_retrieved_event_id.notna(),
        'noisy_retrieved_event_id'
    ].astype(str)
))

if not target_ids:
    raise RuntimeError('V22 produced no exact noisy-retrieved missed fingerprints.')

# Build the aggressive matcher config.
REC_CFG=MatcherConfig(
    sample_rate=SR,
    coarse_rate=4000,
    coarse_floor=RECOVERY_COARSE_FLOOR,
    template_batch=32,
    peaks_per_template=RECOVERY_PEAKS_PER_TEMPLATE,
    verify_topk=RECOVERY_VERIFY_TOPK,
    raw_radius=RECOVERY_RAW_RADIUS,
    min_template_sec=.08,
    max_template_sec=6.0,
    max_templates=2000,
    chunk_sec=24.0,
    fft_cache_mb=384,
    merge_iou=.65,
    seed=517,
)

records=[]
seen_wave=set()

for eid in target_ids:
    if eid not in BANK_EVENT_INDEX:
        continue
    bi=int(BANK_EVENT_INDEX[eid])
    br=BANK_EVENTS.iloc[bi]
    w=np.asarray(bank_event_wave(bi),np.float32)

    wh=wave_sha(w)
    if wh in seen_wave:
        continue
    seen_wave.add(wh)

    cw=coarse_wave(w,REC_CFG).astype(np.float32)
    cw=cw-float(cw.mean())
    n=float(np.linalg.norm(cw))
    if n<1e-8:
        continue
    cw/=n

    records.append(dict(
        tid='REC::'+eid,
        source_cid=str(br.cid),
        group=str(br.group),
        wave=w,
        raw=condition_wave(w),
        coarse=cw,
        source_event_id=eid,
    ))

if not records:
    raise RuntimeError('No usable targeted recovery fingerprints.')

REC_BANK=dict(
    version='v23_targeted_v22_recovery_bank',
    cfg=asdict(REC_CFG),
    records=records,
    source_hashes=[],
)
REC_BANK_BY_ID={str(r['tid']):r for r in records}
REC_MATCHER=ResidualMatcher(REC_BANK,REC_CFG,device=DEVICE)

TARGET_EVENT_IDS=set(r['source_event_id'] for r in records)

print('V22 exact noisy-retrieved event IDs:',len(target_ids))
print('Unique recovery waveforms:',len(records))
print('High-recall matcher config:',asdict(REC_CFG))


V22 exact noisy-retrieved event IDs: 48
Unique recovery waveforms: 48
High-recall matcher config: {'sample_rate': 16000, 'coarse_rate': 4000, 'coarse_floor': 0.08, 'template_batch': 32, 'peaks_per_template': 12, 'verify_topk': 512, 'raw_radius': 32, 'min_template_sec': 0.08, 'max_template_sec': 6.0, 'max_templates': 2000, 'chunk_sec': 24.0, 'fft_cache_mb': 384, 'merge_iou': 0.65, 'seed': 517}


In [ ]:

# 7. Recovery scanning, V13 confirmation and exact V6 single-event fit.

CANDIDATE_FEATURES=[
    'hr_quality','hr_corr16','hr_corr4k','hr_chunk_median',
    'duration_sec',
    'v13_score','v13_gap',
    'v13_raw','v13_p85','v13_p95','v13_chunk','v13_stable',
    'v13_fir_corr','v13_spec','v13_fitfeat',
    'abs_gain','removed_energy_ratio','mix_corr','mix_fit_db',
    'proposal_retrieved_wave_corr'
]

def interval_gap_samples(a,b):
    # 0 for overlap, positive for separation.
    if a[1]<b[0]:
        return b[0]-a[1]
    if b[1]<a[0]:
        return a[0]-b[1]
    return 0

def overlaps_v6_accepted(start,end,accepted):
    margin=int(round(ACCEPTED_EXCLUSION_MARGIN_MS*SR/1000.0))
    for m in accepted:
        a=(int(m['start']),int(m['end']))
        b=(int(start),int(end))
        if interval_iou(a,b)>0 or interval_gap_samples(a,b)<=margin:
            return True
    return False

def recovery_scan_cache_path(cid,phase,exclude_group):
    key=sha_json(dict(
        matcher=REC_MATCHER.signature,
        cid=str(cid),
        phase=str(phase),
        exclude_group=None if exclude_group is None else str(exclude_group)
    ))[:16]
    folder=OUT_ROOT/'recovery_queries'/phase
    folder.mkdir(parents=True,exist_ok=True)
    return folder/(str(cid)+'_'+key+'.json')

def recovery_proposals(cid,y,phase,accepted,exclude_group=None):
    path=recovery_scan_cache_path(cid,phase,exclude_group)
    rows=persist_query(
        REC_MATCHER,y,path,
        exclude_group=None if exclude_group is None else str(exclude_group)
    )

    keep=[]
    for p in sorted(rows,key=lambda z:z['quality'],reverse=True):
        if float(p['quality'])<RECOVERY_MIN_VERIFY_QUALITY:
            continue
        lo=int(round(float(p['onset'])*SR))
        hi=int(round(float(p['offset'])*SR))
        if lo<0 or hi>len(y) or hi<=lo:
            continue
        if overlaps_v6_accepted(lo,hi,accepted):
            continue

        if any(
            interval_iou((lo,hi),(q['lo'],q['hi']))>=RECOVERY_NMS_IOU
            for q in keep
        ):
            continue

        q=dict(p)
        q['lo']=lo;q['hi']=hi
        keep.append(q)
        if len(keep)>=MAX_RECOVERY_CANDIDATES_PER_CLIP:
            break
    return keep

def single_candidate_fit(y,p):
    # Apply the exact V6 fit/acceptance safeguards to this candidate only.
    prep=prepared_matches(y,[p],REC_BANK_BY_ID,RCFG_EXACT)
    if not prep:
        return None
    _,info=reconstruct(
        y,prep,
        dict(method=V6_METHOD,threshold=0.0),
        RCFG_EXACT
    )
    if int(info['accepted_templates'])!=1 or len(info['matches'])!=1:
        return None

    m=info['matches'][0]
    tid=str(m['template_id'])
    w=np.asarray(REC_BANK_BY_ID[tid]['wave'],np.float32)
    lo=int(m['start']);hi=int(m['end'])
    if hi-lo!=len(w):
        return None

    pred=(float(m['gain'])*w).astype(np.float32)
    q=np.asarray(y[lo:hi],np.float32)
    eng=max(float(np.dot(q,q)),1e-20)

    return dict(
        start=lo,end=hi,
        template_id=tid,
        gain=float(m['gain']),
        pred=pred,
        abs_gain=float(abs(m['gain'])),
        removed_energy_ratio=float(np.dot(pred,pred)/eng),
        mix_corr=float(cosine_abs(pred,q)),
        mix_fit_db=float(direct_fit_db(pred,q)),
    )

def candidate_record(cid,y,p,exclude_group=None):
    fit=single_candidate_fit(y,p)
    if fit is None:
        return None

    lo,hi=fit['start'],fit['end']
    q=np.asarray(y[lo:hi],np.float32)
    ret=retrieve_instance(q,exclude_group=exclude_group)
    if ret is None:
        return None

    rm=ret['metrics']
    retrieved=bank_event_wave(int(ret['bank_index']))
    prop_wave=np.asarray(REC_BANK_BY_ID[str(fit['template_id'])]['wave'],np.float32)
    retrieved_warp=resample_1d(retrieved,len(prop_wave))

    feats=dict(
        hr_quality=float(p['quality']),
        hr_corr16=float(p.get('corr16',np.nan)),
        hr_corr4k=float(p.get('corr4k',np.nan)),
        hr_chunk_median=float(p.get('chunk_median',np.nan)),
        duration_sec=float((hi-lo)/SR),
        v13_score=float(ret['score']),
        v13_gap=float(ret['score_gap']) if np.isfinite(ret['score_gap']) else np.nan,
        v13_raw=float(rm['raw']),
        v13_p85=float(rm['p85']),
        v13_p95=float(rm['p95']),
        v13_chunk=float(rm['chunk']),
        v13_stable=float(rm['stable']),
        v13_fir_corr=float(rm['fir_corr']),
        v13_spec=float(rm['spec']),
        v13_fitfeat=float(rm['fitfeat']),
        abs_gain=float(fit['abs_gain']),
        removed_energy_ratio=float(fit['removed_energy_ratio']),
        mix_corr=float(fit['mix_corr']),
        mix_fit_db=float(fit['mix_fit_db']),
        proposal_retrieved_wave_corr=float(
            cosine_abs(prop_wave,retrieved_warp)
        ),
    )

    return dict(
        candidate_key=None,
        cid=str(cid),
        start=lo,end=hi,
        template_id=str(fit['template_id']),
        retrieved_event_id=str(ret['candidate_event_id']),
        pred=fit['pred'],
        features=feats,
    )

def build_recovery_candidates(cid,y,phase,accepted,exclude_group=None):
    props=recovery_proposals(
        cid,y,phase,accepted,
        exclude_group=exclude_group
    )
    out=[]
    for k,p in enumerate(props):
        c=candidate_record(cid,y,p,exclude_group=exclude_group)
        if c is None:
            continue
        c['candidate_key']=f'{cid}__rec{k}_{c["start"]}_{c["end"]}'
        out.append(c)
    return out

print('Candidate features:',len(CANDIDATE_FEATURES))


Candidate features: 20


In [ ]:

# 8. SELECT synthetic: candidate recall, labels, and true end-to-end contribution.
SELECT_CACHE={}
candidate_rows=[]
clip_rows=[]

def missed_gt_intervals(cid,n,accepted):
    out=[]
    for j,ev in enumerate(REFS[cid]):
        lo,hi=event_bounds_samples(ev,n)
        if hi-lo<64:
            continue
        best=max(
            [interval_iou((lo,hi),(int(m['start']),int(m['end'])))
             for m in accepted] or [0.0]
        )
        if best<GT_POSITIVE_IOU:
            out.append(dict(event_id=f'{cid}__e{j}',start=lo,end=hi))
    return out

for cid in tqdm(SELECT_SYN,desc='V23 SELECT high-recall candidates'):
    cid=str(cid)
    y=read_mono_16k(SYN_NOISY[cid])
    clean=read_mono_16k(SYN_CLEAN[cid])

    z_v6,info=exact_v6_validation(cid,y)
    accepted=info['matches']
    assert_v6_reconstruction(y,z_v6,accepted,cid)

    missed=missed_gt_intervals(cid,len(y),accepted)
    candidates=build_recovery_candidates(
        cid,y,'select_syn',accepted,
        exclude_group=str(GROUPS[cid])
    )

    # Label each candidate by overlap with a GT event missed by final V6 acceptance.
    for c in candidates:
        best_iou=0.0
        best_event=''
        for g in missed:
            ov=interval_iou(
                (c['start'],c['end']),
                (g['start'],g['end'])
            )
            if ov>best_iou:
                best_iou=ov;best_event=g['event_id']

        positive=bool(best_iou>=GT_POSITIVE_IOU)

        # True isolated clip impact is diagnostic only and never a model feature.
        z=np.asarray(z_v6,np.float64).copy()
        z[c['start']:c['end']]-=np.asarray(c['pred'],np.float64)
        z=pcm_roundtrip(z)
        gain=float(si_sdr(z,clean)-si_sdr(z_v6,clean))

        candidate_rows.append(dict(
            candidate_key=c['candidate_key'],
            cid=cid,
            group=str(GROUPS[cid]),
            start=c['start'],end=c['end'],
            positive_missed_event=positive,
            matched_missed_event=best_event,
            gt_iou=float(best_iou),
            isolated_clip_gain_db=gain,
            isolated_harm=bool(gain<-BAD_CLIP_MARGIN_DB),
            **c['features']
        ))

    # Proposal/candidate recall over missed GT events.
    recovered_events=set()
    for c in candidates:
        for g in missed:
            if interval_iou(
                (c['start'],c['end']),
                (g['start'],g['end'])
            )>=GT_POSITIVE_IOU:
                recovered_events.add(g['event_id'])

    SELECT_CACHE[cid]=dict(
        y=y,clean=clean,
        v6_pcm=np.asarray(z_v6,np.float32),
        accepted=accepted,
        candidates=candidates,
        missed=missed,
    )

    clip_rows.append(dict(
        cid=cid,
        missed_gt_events=len(missed),
        surfaced_missed_events=len(recovered_events),
        recovery_candidates=len(candidates),
        v6_sisdr=float(si_sdr(z_v6,clean)),
    ))

SELECT_CAND=pd.DataFrame(candidate_rows)
SELECT_CLIPS=pd.DataFrame(clip_rows)

if SELECT_CAND.empty:
    raise RuntimeError('High-recall recovery matcher produced no fitted SELECT candidates.')

print('SELECT clips:',len(SELECT_CLIPS))
print('V6-missed GT events:',int(SELECT_CLIPS.missed_gt_events.sum()))
print('Missed events surfaced after high-recall + V6 fit:',
      int(SELECT_CLIPS.surfaced_missed_events.sum()))
print('Candidate-generator missed-event recall:',
      float(SELECT_CLIPS.surfaced_missed_events.sum()/
            max(SELECT_CLIPS.missed_gt_events.sum(),1)))
print('Fitted candidates:',len(SELECT_CAND))
print('Positive candidate fraction:',
      float(SELECT_CAND.positive_missed_event.mean()))
print('Positive candidate isolated mean gain:',
      float(SELECT_CAND.loc[
          SELECT_CAND.positive_missed_event,'isolated_clip_gain_db'
      ].mean()) if SELECT_CAND.positive_missed_event.any() else np.nan)

SELECT_CAND.to_csv(OUT_ROOT/'select_candidate_table.csv',index=False)
SELECT_CLIPS.to_csv(OUT_ROOT/'select_clip_table.csv',index=False)

display(SELECT_CAND[
    ['positive_missed_event','isolated_clip_gain_db']+CANDIDATE_FEATURES
].groupby('positive_missed_event').median(numeric_only=True).T)


V23 SELECT high-recall candidates:   0%|          | 0/128 [00:00<?, ?it/s]

SELECT clips: 128
V6-missed GT events: 71
Missed events surfaced after high-recall + V6 fit: 55
Candidate-generator missed-event recall: 0.7746478873239436
Fitted candidates: 63
Positive candidate fraction: 0.873015873015873
Positive candidate isolated mean gain: 31.771431827126385


positive_missed_event,False,True
isolated_clip_gain_db,-10.880117,41.787738
hr_quality,0.215000,0.990199
hr_corr16,0.217280,0.992220
hr_corr4k,0.232024,0.682111
hr_chunk_median,0.286207,0.990199
duration_sec,0.242000,0.525000
v13_score,0.273638,0.928334
v13_gap,0.003880,0.000115
v13_raw,0.229939,0.919023
v13_p85,0.214895,0.992201


In [ ]:

# 9. OOF candidate gate + true clip-level threshold sweep.
def make_gate(name):
    if name=='logistic':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler(),
            LogisticRegression(
                C=.5,class_weight='balanced',
                max_iter=4000,random_state=SEED
            )
        )
    if name=='rf':
        return make_pipeline(
            SimpleImputer(strategy='median'),
            RandomForestClassifier(
                n_estimators=700,max_depth=5,min_samples_leaf=3,
                max_features=.75,class_weight='balanced_subsample',
                random_state=SEED,n_jobs=-1
            )
        )
    raise KeyError(name)

def predict_fold(model,Xtr,ytr,Xte):
    if len(np.unique(ytr))<2:
        return np.full(len(Xte),float(np.mean(ytr)),float)
    model.fit(Xtr,ytr)
    return model.predict_proba(Xte)[:,1]

X=SELECT_CAND[CANDIDATE_FEATURES]
y=SELECT_CAND.positive_missed_event.astype(int).to_numpy()
groups=SELECT_CAND.group.astype(str).to_numpy()

if len(np.unique(y))<2:
    raise RuntimeError('Candidate labels contain only one class.')

nfold=min(OUTER_FOLDS,len(np.unique(groups)))
cv=GroupKFold(n_splits=nfold)

for name in ['logistic','rf']:
    p=np.full(len(SELECT_CAND),np.nan)
    for tr,te in cv.split(X,y,groups):
        p[te]=predict_fold(
            make_gate(name),X.iloc[tr],y[tr],X.iloc[te]
        )
    if not np.isfinite(p).all():
        raise RuntimeError(f'Incomplete OOF probabilities: {name}')
    SELECT_CAND[name+'_oof_prob']=p

CAND_BY_KEY={
    c['candidate_key']:c
    for cache in SELECT_CACHE.values()
    for c in cache['candidates']
}

def route_candidates(cache,prob_map,threshold):
    ranked=[]
    for c in cache['candidates']:
        p=float(prob_map.get(c['candidate_key'],-np.inf))
        if p>=float(threshold):
            ranked.append((p,c))
    ranked.sort(key=lambda z:z[0],reverse=True)

    chosen=[]
    for p,c in ranked:
        iv=(c['start'],c['end'])
        if any(interval_iou(iv,(q['start'],q['end']))>0.05 for q in chosen):
            continue
        chosen.append(c)
    return chosen

def apply_candidates(cache,prob_map,threshold):
    chosen=route_candidates(cache,prob_map,threshold)
    z=np.asarray(cache['v6_pcm'],np.float64).copy()
    for c in chosen:
        z[c['start']:c['end']]-=np.asarray(c['pred'],np.float64)
    return pcm_roundtrip(z),chosen

def sweep_for_probs(probs):
    prob_map=dict(zip(
        SELECT_CAND.candidate_key.astype(str),
        np.asarray(probs,float)
    ))

    ths=np.unique(np.clip(np.r_[
        np.linspace(.10,.99,46),
        np.quantile(np.asarray(probs,float),np.linspace(.05,.98,24))
    ],.001,.9999))

    rows=[]
    positives=set(
        SELECT_CAND.loc[
            SELECT_CAND.positive_missed_event,
            'candidate_key'
        ].astype(str)
    )

    for th in sorted(ths):
        sis=[];v6=[];bad=0;sw=0;good=0;swclips=0
        recovered_event_ids=set()

        for cid,cache in SELECT_CACHE.items():
            z,chosen=apply_candidates(cache,prob_map,th)
            s=si_sdr(z,cache['clean'])
            b=si_sdr(cache['v6_pcm'],cache['clean'])
            sis.append(s);v6.append(b)
            bad+=int(s<b-BAD_CLIP_MARGIN_DB)
            swclips+=int(bool(chosen))

            for c in chosen:
                sw+=1
                if c['candidate_key'] in positives:
                    good+=1
                    rr=SELECT_CAND.loc[
                        SELECT_CAND.candidate_key==c['candidate_key']
                    ].iloc[0]
                    if rr.matched_missed_event:
                        recovered_event_ids.add(str(rr.matched_missed_event))

        sis=np.asarray(sis,float);v6=np.asarray(v6,float)
        rows.append(dict(
            threshold=float(th),
            mean_sisdr=float(sis.mean()),
            gain_over_v6_db=float((sis-v6).mean()),
            median_gain_db=float(np.median(sis-v6)),
            p10_gain_db=float(np.quantile(sis-v6,.10)),
            harm_fraction=float(bad/max(len(sis),1)),
            switched_clip_fraction=float(swclips/max(len(sis),1)),
            switched_candidates=int(sw),
            candidate_precision=float(good/max(sw,1)),
            recovered_missed_events=int(len(recovered_event_ids)),
        ))
    return pd.DataFrame(rows)

GATE_TABLES={}
summ=[]

for name in ['logistic','rf']:
    tab=sweep_for_probs(SELECT_CAND[name+'_oof_prob'].to_numpy(float))
    GATE_TABLES[name]=tab

    elig=tab[
        (tab.harm_fraction<=BALANCED_MAX_SYN_HARM) &
        (tab.candidate_precision>=BALANCED_MIN_CANDIDATE_PRECISION)
    ]
    best=(elig if len(elig) else tab).sort_values(
        ['mean_sisdr','candidate_precision'],
        ascending=[False,False]
    ).iloc[0]
    summ.append(dict(model=name,**best.to_dict()))

GATE_RESULTS=pd.DataFrame(summ).sort_values(
    ['mean_sisdr','candidate_precision'],
    ascending=[False,False]
).reset_index(drop=True)
display(GATE_RESULTS)

BEST_GATE=str(GATE_RESULTS.iloc[0].model)
BEST_TAB=GATE_TABLES[BEST_GATE]
OOF_PROBS=SELECT_CAND[BEST_GATE+'_oof_prob'].to_numpy(float)

bal=BEST_TAB[
    (BEST_TAB.harm_fraction<=BALANCED_MAX_SYN_HARM) &
    (BEST_TAB.candidate_precision>=BALANCED_MIN_CANDIDATE_PRECISION) &
    (BEST_TAB.gain_over_v6_db>=SELECT_MIN_GAIN_BALANCED_DB)
]
con=BEST_TAB[
    (BEST_TAB.harm_fraction<=CONSERVATIVE_MAX_SYN_HARM) &
    (BEST_TAB.candidate_precision>=CONSERVATIVE_MIN_CANDIDATE_PRECISION) &
    (BEST_TAB.gain_over_v6_db>=SELECT_MIN_GAIN_CONSERVATIVE_DB)
]

if len(bal):
    BALANCED_THRESHOLD=float(
        bal.sort_values(
            ['mean_sisdr','candidate_precision'],
            ascending=[False,False]
        ).iloc[0].threshold
    )
else:
    BALANCED_THRESHOLD=np.nan

if len(con):
    CONSERVATIVE_THRESHOLD=float(
        con.sort_values(
            ['mean_sisdr','candidate_precision'],
            ascending=[False,False]
        ).iloc[0].threshold
    )
else:
    CONSERVATIVE_THRESHOLD=np.nan

print('Best gate:',BEST_GATE)
print('Initial thresholds:',
      BALANCED_THRESHOLD,CONSERVATIVE_THRESHOLD)

if not np.isfinite(BALANCED_THRESHOLD) and not np.isfinite(CONSERVATIVE_THRESHOLD):
    raise RuntimeError(
        'High-recall candidates do not deliver enough OOF SELECT gain safely. Keep V6.'
    )


,model,threshold,mean_sisdr,gain_over_v6_db,median_gain_db,p10_gain_db,harm_fraction,switched_clip_fraction,switched_candidates,candidate_precision,recovered_missed_events
0,logistic,0.119778,48.584323,16.848774,0.0,0.0,0.0,0.351562,55.0,1.000000,55.0
1,rf,0.218667,48.311278,16.575729,0.0,0.0,0.0,0.351562,56.0,0.982143,55.0


Best gate: logistic
Initial thresholds: 0.11977777777777779 0.11977777777777779


In [ ]:

# 10. Fit final candidate gate and tighten thresholds using SELECT natural controls.
FINAL_GATE=make_gate(BEST_GATE)
FINAL_GATE.fit(
    SELECT_CAND[CANDIDATE_FEATURES],
    y
)

NAT_CACHE={}
nat_rows=[]

for cid in tqdm(SELECT_NAT,desc='V23 SELECT natural audit'):
    cid=str(cid)
    if cid not in NAT_NOISY:
        continue
    y0=read_mono_16k(NAT_NOISY[cid])
    z_v6,info=exact_v6_validation(cid,y0)
    accepted=info['matches']

    cands=build_recovery_candidates(
        cid,y0,'select_nat',accepted,
        exclude_group=str(GROUPS[cid])
    )
    NAT_CACHE[cid]=dict(
        y=y0,v6_pcm=np.asarray(z_v6,np.float32),
        accepted=accepted,candidates=cands
    )

    for c in cands:
        nat_rows.append(dict(
            candidate_key=c['candidate_key'],
            cid=cid,
            **c['features']
        ))

NAT_CAND=pd.DataFrame(nat_rows)
if len(NAT_CAND):
    NAT_CAND['gate_prob']=FINAL_GATE.predict_proba(
        NAT_CAND[CANDIDATE_FEATURES]
    )[:,1]
else:
    NAT_CAND=pd.DataFrame(
        columns=['candidate_key','cid','gate_prob']+CANDIDATE_FEATURES
    )

NAT_PROBS=dict(zip(
    NAT_CAND.candidate_key.astype(str),
    NAT_CAND.gate_prob.astype(float)
))

def natural_route_fraction(th):
    if not np.isfinite(th) or not len(NAT_CACHE):
        return 0.0
    routed=0
    for cache in NAT_CACHE.values():
        _,chosen=apply_candidates(cache,NAT_PROBS,th)
        routed+=int(bool(chosen))
    return float(routed/max(len(NAT_CACHE),1))

def tighten_threshold(th,cap):
    if not np.isfinite(th) or NAT_CAND.empty:
        return th
    vals=np.unique(np.r_[
        float(th),
        NAT_CAND.gate_prob.to_numpy(float)+1e-8,
        .999999
    ])
    for x in np.sort(vals[vals>=float(th)]):
        if natural_route_fraction(float(x))<=cap:
            return float(x)
    return .999999

if np.isfinite(BALANCED_THRESHOLD):
    BALANCED_THRESHOLD=tighten_threshold(
        BALANCED_THRESHOLD,
        BALANCED_NATURAL_ROUTE_CAP
    )

if np.isfinite(CONSERVATIVE_THRESHOLD):
    CONSERVATIVE_THRESHOLD=tighten_threshold(
        CONSERVATIVE_THRESHOLD,
        CONSERVATIVE_NATURAL_ROUTE_CAP
    )

# Re-evaluate OOF SELECT at tightened thresholds exactly.
OOF_MAP=dict(zip(
    SELECT_CAND.candidate_key.astype(str),
    OOF_PROBS
))

def eval_select_threshold(th):
    sis=[];v6=[];bad=0;sw=0;good=0
    positives=set(
        SELECT_CAND.loc[
            SELECT_CAND.positive_missed_event,
            'candidate_key'
        ].astype(str)
    )
    for cache in SELECT_CACHE.values():
        z,chosen=apply_candidates(cache,OOF_MAP,th)
        s=si_sdr(z,cache['clean'])
        b=si_sdr(cache['v6_pcm'],cache['clean'])
        sis.append(s);v6.append(b)
        bad+=int(s<b-BAD_CLIP_MARGIN_DB)
        for c in chosen:
            sw+=1
            good+=int(c['candidate_key'] in positives)
    sis=np.asarray(sis,float);v6=np.asarray(v6,float)
    return dict(
        threshold=float(th),
        mean_sisdr=float(sis.mean()),
        gain_over_v6_db=float((sis-v6).mean()),
        harm_fraction=float(bad/max(len(sis),1)),
        switched_candidates=int(sw),
        candidate_precision=float(good/max(sw,1))
    )

SELECT_VARIANTS=[]
if np.isfinite(BALANCED_THRESHOLD):
    r=eval_select_threshold(BALANCED_THRESHOLD)
    r.update(
        variant='BALANCED',
        natural_route_fraction=natural_route_fraction(BALANCED_THRESHOLD)
    )
    SELECT_VARIANTS.append(r)

if np.isfinite(CONSERVATIVE_THRESHOLD):
    r=eval_select_threshold(CONSERVATIVE_THRESHOLD)
    r.update(
        variant='CONSERVATIVE',
        natural_route_fraction=natural_route_fraction(CONSERVATIVE_THRESHOLD)
    )
    SELECT_VARIANTS.append(r)

SELECT_VARIANTS=pd.DataFrame(SELECT_VARIANTS)
display(SELECT_VARIANTS)
SELECT_VARIANTS.to_csv(
    OUT_ROOT/'select_frozen_variants.csv',
    index=False
)

# Re-apply minimum gain after natural tightening.
qualified_names=[]
for r in SELECT_VARIANTS.itertuples(index=False):
    if r.variant=='BALANCED':
        ok=(
            r.gain_over_v6_db>=SELECT_MIN_GAIN_BALANCED_DB and
            r.harm_fraction<=BALANCED_MAX_SYN_HARM and
            r.natural_route_fraction<=BALANCED_NATURAL_ROUTE_CAP
        )
    else:
        ok=(
            r.gain_over_v6_db>=SELECT_MIN_GAIN_CONSERVATIVE_DB and
            r.harm_fraction<=CONSERVATIVE_MAX_SYN_HARM and
            r.natural_route_fraction<=CONSERVATIVE_NATURAL_ROUTE_CAP
        )
    if ok:
        qualified_names.append(r.variant)

if not qualified_names:
    raise RuntimeError(
        'Natural-control tightening removes the SELECT advantage. Keep V6.'
    )

print('SELECT-qualified variants:',qualified_names)


V23 SELECT natural audit:   0%|          | 0/338 [00:00<?, ?it/s]

,threshold,mean_sisdr,gain_over_v6_db,harm_fraction,switched_candidates,candidate_precision,variant,natural_route_fraction
0,0.119778,48.584323,16.848774,0.0,55,1.0,BALANCED,0.002959
1,0.119778,48.584323,16.848774,0.0,55,1.0,CONSERVATIVE,0.002959


SELECT-qualified variants: ['BALANCED', 'CONSERVATIVE']


In [ ]:

# 7. Released validation reference transcripts for the final HOLD WER audit.
METADATA_JSON=VAL_ROOT/'validationMetadata.json'
if not METADATA_JSON.is_file():
    raise FileNotFoundError(METADATA_JSON)

with open(METADATA_JSON,encoding='utf-8') as f:
    meta=json.load(f)
if not isinstance(meta,list):
    raise TypeError('Expected validationMetadata.json to be a list.')

TEXT_REFS={}
for r in meta:
    cid=Path(r['segmentFileName']).stem
    if cid in TEXT_REFS:
        raise ValueError(f'Duplicate transcript ID: {cid}')
    if TRANSCRIPT_FIELD in r and isinstance(r[TRANSCRIPT_FIELD],str):
        TEXT_REFS[cid]=r[TRANSCRIPT_FIELD]

missing=[
    c for c in SELECT_SYN+SELECT_NAT+HOLD_SYN+HOLD_NAT
    if c not in TEXT_REFS or not normalize_text(TEXT_REFS[c])
]
if missing:
    raise ValueError(f'Missing reference transcripts; examples: {missing[:5]}')

print('Reference transcripts:',len(TEXT_REFS))


Reference transcripts: 2501


In [ ]:

# 12. Frozen HOLD waveform evaluation. Thresholds/model are not retuned here.
HOLD_CACHE={}
hold_rows=[]

for cid in tqdm(HOLD_SYN+HOLD_NAT,desc='V23 HOLD recovery inference'):
    cid=str(cid)
    domain='synthetic' if cid in set(HOLD_SYN) else 'natural'
    source=SYN_NOISY if domain=='synthetic' else NAT_NOISY
    y0=read_mono_16k(source[cid])

    z_v6,info=exact_v6_validation(cid,y0)
    accepted=info['matches']

    cands=build_recovery_candidates(
        cid,y0,'hold',accepted,
        exclude_group=str(GROUPS[cid])
    )
    probs=(
        FINAL_GATE.predict_proba(
            pd.DataFrame([c['features'] for c in cands])[CANDIDATE_FEATURES]
        )[:,1]
        if cands else np.array([],float)
    )
    pmap=dict(zip([c['candidate_key'] for c in cands],probs))

    cache=dict(
        y=y0,v6_pcm=np.asarray(z_v6,np.float32),
        accepted=accepted,candidates=cands
    )
    HOLD_CACHE[cid]=dict(cache=cache,pmap=pmap,domain=domain)

HOLD_VARIANTS=[]

for variant,th in [
    ('BALANCED',BALANCED_THRESHOLD),
    ('CONSERVATIVE',CONSERVATIVE_THRESHOLD),
]:
    if variant not in qualified_names or not np.isfinite(th):
        continue

    folder=OUT_ROOT/'hold_audio'/variant.lower()
    folder.mkdir(parents=True,exist_ok=True)

    paths={}
    gains=[]
    harms=0
    nat_routed=0

    for cid,obj in HOLD_CACHE.items():
        cache=obj['cache']
        z,chosen=apply_candidates(cache,obj['pmap'],th)

        p=folder/(cid+'.wav')
        write_pcm(p,z,SR)
        wav_audit(p,len(cache['y']),SR)
        paths[cid]=str(p)

        if obj['domain']=='synthetic':
            clean=read_mono_16k(SYN_CLEAN[cid])
            b=si_sdr(cache['v6_pcm'],clean)
            n=si_sdr(z,clean)
            gains.append(n-b)
            harms+=int(n<b-BAD_CLIP_MARGIN_DB)
        else:
            nat_routed+=int(bool(chosen))

    gains=np.asarray(gains,float)

    if variant=='BALANCED':
        waveform_safe=bool(
            gains.mean()>=HOLD_MIN_GAIN_BALANCED_DB and
            harms/max(len(gains),1)<=HOLD_MAX_HARM_BALANCED
        )
    else:
        waveform_safe=bool(
            gains.mean()>=HOLD_MIN_GAIN_CONSERVATIVE_DB and
            harms/max(len(gains),1)<=HOLD_MAX_HARM_CONSERVATIVE
        )

    HOLD_VARIANTS.append(dict(
        variant=variant,
        threshold=float(th),
        gain_over_v6_db=float(gains.mean()),
        median_gain_db=float(np.median(gains)),
        p10_gain_db=float(np.quantile(gains,.10)),
        harm_fraction=float(harms/max(len(gains),1)),
        natural_routed_fraction=float(nat_routed/max(len(HOLD_NAT),1)),
        waveform_safe=waveform_safe,
        paths=paths,
    ))

HOLD_WAVE_TABLE=pd.DataFrame([
    {k:v for k,v in r.items() if k!='paths'}
    for r in HOLD_VARIANTS
])
display(HOLD_WAVE_TABLE)
HOLD_WAVE_TABLE.to_csv(
    OUT_ROOT/'hold_waveform_evaluation.csv',
    index=False
)

if not any(r['waveform_safe'] for r in HOLD_VARIANTS):
    raise RuntimeError(
        'V23 does not generalize safely to HOLD waveform metrics. Keep V6.'
    )


V23 HOLD recovery inference:   0%|          | 0/549 [00:00<?, ?it/s]

,variant,threshold,gain_over_v6_db,median_gain_db,p10_gain_db,harm_fraction,natural_routed_fraction,waveform_safe
0,BALANCED,0.119778,9.887157,0.0,0.0,0.0,0.004902,True
1,CONSERVATIVE,0.119778,9.887157,0.0,0.0,0.0,0.004902,True


In [ ]:

# 13. Pin exact scored V6 hidden-test baseline and original hidden-test source.
V6_TEST_WAVS=V6_RUN/'final_wavs'
V6_TEST_RECORDS=V6_RUN/'test_records'
V6_TEST_AUDIT_PATH=V6_RUN/'test_reconstruction_audit.csv'

if not V6_TEST_WAVS.is_dir() or not V6_TEST_RECORDS.is_dir():
    raise FileNotFoundError('Exact V6 test WAVs/records are required.')

BASELINE_PATHS={p.stem:str(p) for p in V6_TEST_WAVS.glob('*.wav')}
TEST_IDS=sorted(BASELINE_PATHS)

if len(TEST_IDS)!=EXPECTED_TEST_CLIPS:
    raise ValueError(f'Expected {EXPECTED_TEST_CLIPS} V6 WAVs, found {len(TEST_IDS)}.')

EXPECTED_FRAMES={}
for cid,p in tqdm(BASELINE_PATHS.items(),desc='Audit exact V6 baseline'):
    info=sf.info(p)
    if info.samplerate!=SR or info.channels!=1 or info.subtype!='PCM_16':
        raise ValueError(f'Invalid V6 baseline WAV: {cid}')
    EXPECTED_FRAMES[cid]=int(info.frames)

options=[Path(TEST_AUDIO_SOURCE)] if TEST_AUDIO_SOURCE is not None else TEST_AUDIO_CANDIDATES
TEST_SOURCE=None;errors=[]
for p in options:
    if not p.exists():
        continue
    try:
        TEST_SOURCE=StrictWaveSource(p,TEST_IDS)
        break
    except Exception as e:
        errors.append(f'{p}: {e}')

if TEST_SOURCE is None:
    raise FileNotFoundError(
        'Set TEST_AUDIO_SOURCE to the ORIGINAL input WAV folder/ZIP.\n' +
        '\n'.join(errors)
    )

def load_v6_test_record(cid):
    p=V6_TEST_RECORDS/(str(cid)+'.json')
    if not p.is_file():
        raise FileNotFoundError(p)
    r=load_json(p)
    if 'details' not in r or 'matches' not in r['details']:
        raise ValueError(f'V6 record lacks accepted matches: {cid}')
    return r

# Verify that accepted match logs are sufficient to recreate the scored V6 output.
checked=0
for cid in TEST_IDS:
    r=load_v6_test_record(cid)
    if not r['details']['matches']:
        continue
    y0=TEST_SOURCE.load(cid)
    baseline=read_wave_strict(BASELINE_PATHS[cid])
    assert_v6_reconstruction(y0,baseline,r['details']['matches'],cid)
    checked+=1
    if checked>=25:
        break
print('Verified accepted-match reconstruction on',checked,'matched test clips.')


Audit exact V6 baseline:   0%|          | 0/5517 [00:00<?, ?it/s]

Verified accepted-match reconstruction on 25 matched test clips.


In [ ]:

# 14. Hidden-test V23 recovery scan. V6 accepted matches are NEVER changed.
RUN_KEY=sha_json(dict(
    version='v23_high_recall_missed_event_20260920',
    parent_v6=V6_RUN_KEY,
    recovery_matcher=REC_MATCHER.signature,
    recovery_bank_events=sorted(TARGET_EVENT_IDS),
    gate=BEST_GATE,
    balanced_threshold=(
        None if not np.isfinite(BALANCED_THRESHOLD)
        else float(BALANCED_THRESHOLD)
    ),
    conservative_threshold=(
        None if not np.isfinite(CONSERVATIVE_THRESHOLD)
        else float(CONSERVATIVE_THRESHOLD)
    ),
    v13=dict(
        cheap=CHEAP_RECIPE,
        shortlist=SHORTLIST_K,
        fine=FINE_RECIPE
    )
))[:16]

RUN_DIR=OUT_ROOT/('run_'+RUN_KEY)
RUN_DIR.mkdir(parents=True,exist_ok=True)
BAL_DIR=RUN_DIR/'balanced_changed_wavs'
CON_DIR=RUN_DIR/'conservative_changed_wavs'
BAL_DIR.mkdir(exist_ok=True)
CON_DIR.mkdir(exist_ok=True)

BAL_PATHS={}
CON_PATHS={}
route_rows=[]

for cid in tqdm(TEST_IDS,desc='V23 hidden-test recovery'):
    rec=load_v6_test_record(cid)
    accepted=rec['details']['matches']

    y0=TEST_SOURCE.load(cid)
    baseline=read_wave_strict(BASELINE_PATHS[cid])
    if len(y0)!=len(baseline):
        raise ValueError(cid)

    cands=build_recovery_candidates(
        cid,y0,'test',accepted,
        exclude_group=None
    )
    probs=(
        FINAL_GATE.predict_proba(
            pd.DataFrame([c['features'] for c in cands])[CANDIDATE_FEATURES]
        )[:,1]
        if cands else np.array([],float)
    )
    pmap=dict(zip([c['candidate_key'] for c in cands],probs))

    cache=dict(
        y=y0,
        v6_pcm=np.asarray(baseline,np.float32),
        accepted=accepted,
        candidates=cands
    )

    nbal=ncon=0

    if 'BALANCED' in qualified_names and np.isfinite(BALANCED_THRESHOLD):
        z,chosen=apply_candidates(cache,pmap,BALANCED_THRESHOLD)
        nbal=len(chosen)
        if nbal:
            p=BAL_DIR/(cid+'.wav')
            write_pcm(p,z,SR)
            wav_audit(p,len(y0),SR)
            BAL_PATHS[cid]=str(p)
        else:
            BAL_PATHS[cid]=BASELINE_PATHS[cid]
    else:
        BAL_PATHS[cid]=BASELINE_PATHS[cid]

    if 'CONSERVATIVE' in qualified_names and np.isfinite(CONSERVATIVE_THRESHOLD):
        z,chosen=apply_candidates(cache,pmap,CONSERVATIVE_THRESHOLD)
        ncon=len(chosen)
        if ncon:
            p=CON_DIR/(cid+'.wav')
            write_pcm(p,z,SR)
            wav_audit(p,len(y0),SR)
            CON_PATHS[cid]=str(p)
        else:
            CON_PATHS[cid]=BASELINE_PATHS[cid]
    else:
        CON_PATHS[cid]=BASELINE_PATHS[cid]

    route_rows.append(dict(
        clip_id=cid,
        v6_accepted_events=len(accepted),
        recovery_candidates=len(cands),
        balanced_recovered_events=int(nbal),
        conservative_recovered_events=int(ncon),
        max_gate_prob=float(np.max(probs)) if len(probs) else np.nan,
    ))

TEST_ROUTE=pd.DataFrame(route_rows)
TEST_ROUTE.to_csv(RUN_DIR/'test_route_audit.csv',index=False)

print('Balanced changed clips:',
      int((TEST_ROUTE.balanced_recovered_events>0).sum()))
print('Conservative changed clips:',
      int((TEST_ROUTE.conservative_recovered_events>0).sum()))
print('Total recovery candidates:',
      int(TEST_ROUTE.recovery_candidates.sum()))


V23 hidden-test recovery:   0%|          | 0/5517 [00:00<?, ?it/s]

Balanced changed clips: 625
Conservative changed clips: 625
Total recovery candidates: 1486


In [ ]:

# 15. Load the exact pinned SraVaani revision used by V6.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModel

token=os.environ.get('HF_TOKEN')
if token is None:
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:
        token=None

pin_path=V6_ROOT/'sravaani_revision_pin.json'
if not pin_path.is_file():
    raise FileNotFoundError(pin_path)
pin=load_json(pin_path)

if pin.get('repo')!=ASR_REPO:
    raise ValueError('SraVaani repo pin mismatch.')
resolved_revision=pin['revision']
if not isinstance(resolved_revision,str) or len(resolved_revision)!=40:
    raise ValueError('Expected immutable SraVaani commit hash.')

ASR_MODEL_SIGNATURE=sha_json(dict(
    repo=ASR_REPO,
    revision=resolved_revision,
    task='transcribe_return_hypotheses'
))
if FROZEN.get('asr_signature')!=ASR_MODEL_SIGNATURE:
    raise ValueError('SraVaani signature differs from exact V6.')

print('Pinned SraVaani:',resolved_revision)

asr_model=AutoModel.from_pretrained(
    ASR_REPO,
    revision=resolved_revision,
    code_revision=resolved_revision,
    trust_remote_code=True,
    token=token
).to(DEVICE).eval()

ASR_ENGINE=HashASR(
    asr_model,
    ASR_MODEL_SIGNATURE,
    V6_ROOT/'asr_cache_by_content',
    ASR_BATCH_SIZE
)


In [ ]:

# 16. HOLD WER audit versus exact V6 HOLD output.
HOLD_V6_DIR=OUT_ROOT/'hold_audio'/'v6_exact'
HOLD_V6_DIR.mkdir(parents=True,exist_ok=True)
HOLD_V6_PATHS={}

for cid,obj in tqdm(HOLD_CACHE.items(),desc='Write exact V6 HOLD'):
    cache=obj['cache']
    p=HOLD_V6_DIR/(cid+'.wav')
    write_pcm(p,cache['v6_pcm'],SR)
    wav_audit(p,len(cache['y']),SR)
    HOLD_V6_PATHS[cid]=str(p)

V6_HOLD_ASR=ASR_ENGINE.transcribe(
    HOLD_V6_PATHS,
    progress=tqdm
)
V6_WER={
    'syn':pooled_wer(HOLD_SYN,TEXT_REFS,V6_HOLD_ASR),
    'nat':pooled_wer(HOLD_NAT,TEXT_REFS,V6_HOLD_ASR),
    'all':pooled_wer(HOLD_SYN+HOLD_NAT,TEXT_REFS,V6_HOLD_ASR),
}

final_rows=[]
FINAL_QUALIFIED={}

for r in HOLD_VARIANTS:
    name=r['variant']
    if not r['waveform_safe']:
        FINAL_QUALIFIED[name]=False
        continue

    hyp=ASR_ENGINE.transcribe(
        r['paths'],
        progress=tqdm
    )
    ws=pooled_wer(HOLD_SYN,TEXT_REFS,hyp)
    wn=pooled_wer(HOLD_NAT,TEXT_REFS,hyp)
    wa=pooled_wer(HOLD_SYN+HOLD_NAT,TEXT_REFS,hyp)

    ds=float(ws['wer']-V6_WER['syn']['wer'])
    dn=float(wn['wer']-V6_WER['nat']['wer'])
    da=float(wa['wer']-V6_WER['all']['wer'])

    wer_safe=bool(
        ds<=HOLD_MAX_WER_INCREASE and
        dn<=HOLD_MAX_WER_INCREASE and
        da<=HOLD_MAX_WER_INCREASE
    )
    FINAL_QUALIFIED[name]=bool(r['waveform_safe'] and wer_safe)

    final_rows.append(dict(
        variant=name,
        waveform_gain_over_v6_db=float(r['gain_over_v6_db']),
        waveform_harm_fraction=float(r['harm_fraction']),
        natural_routed_fraction=float(r['natural_routed_fraction']),
        synthetic_wer_change_vs_v6=ds,
        natural_wer_change_vs_v6=dn,
        pooled_wer_change_vs_v6=da,
        wer_safe=wer_safe,
        qualified_for_submission=FINAL_QUALIFIED[name],
    ))

HOLD_FINAL_TABLE=pd.DataFrame(final_rows)
display(HOLD_FINAL_TABLE)
HOLD_FINAL_TABLE.to_csv(
    RUN_DIR/'hold_final_gate.csv',
    index=False
)

print('FINAL QUALIFICATION:',FINAL_QUALIFIED)

if not any(FINAL_QUALIFIED.values()):
    raise RuntimeError(
        'V23 fails final HOLD SI-SDR/WER gate. Keep V6.'
    )


In [ ]:

# 17. Final hidden-test transcripts and competition ZIP(s).
OUTPUTS={}

def package_variant(name,paths,threshold):
    final_asr=ASR_ENGINE.transcribe(
        paths,
        progress=tqdm
    )

    stem=f'submission_Track2_V23_RECOVER_{name}_{RUN_KEY[:8]}'
    zip_path=RUN_DIR/(stem+'.zip')

    package=pack_submission(
        paths,final_asr,
        EXPECTED_FRAMES,
        zip_path,SR
    )

    manifest=dict(
        version='v23_high_recall_missed_event_20260920',
        run_key=RUN_KEY,
        variant=name,
        threshold=float(threshold),
        parent_v6_run=V6_RUN_KEY,
        recovery_bank_events=len(TARGET_EVENT_IDS),
        recovery_unique_waveforms=len(REC_BANK['records']),
        recovery_matcher_cfg=asdict(REC_CFG),
        gate_model=BEST_GATE,
        v13=dict(
            cheap_recipe=CHEAP_RECIPE,
            shortlist_k=SHORTLIST_K,
            fine_recipe=FINE_RECIPE
        ),
        hold_final=(
            HOLD_FINAL_TABLE[
                HOLD_FINAL_TABLE.variant==name
            ].iloc[0].to_dict()
        ),
        clips=len(TEST_IDS),
        changed_clips=int(
            (TEST_ROUTE[
                name.lower()+'_recovered_events'
            ]>0).sum()
        ),
        recovered_events=int(
            TEST_ROUTE[
                name.lower()+'_recovered_events'
            ].sum()
        ),
        no_test_clean_or_gt=True,
        base_for_unrecovered_audio='exact scored V6 WAV',
        package=package,
    )
    put_json(
        RUN_DIR/(stem+'_manifest.json'),
        manifest
    )

    print(name,'READY:',zip_path)
    print('SHA256:',package['zip_sha256'])
    return manifest

if (
    BUILD_BALANCED and
    FINAL_QUALIFIED.get('BALANCED',False)
):
    OUTPUTS['BALANCED']=package_variant(
        'BALANCED',
        BAL_PATHS,
        BALANCED_THRESHOLD
    )

if (
    BUILD_CONSERVATIVE and
    FINAL_QUALIFIED.get('CONSERVATIVE',False)
):
    OUTPUTS['CONSERVATIVE']=package_variant(
        'CONSERVATIVE',
        CON_PATHS,
        CONSERVATIVE_THRESHOLD
    )

if not OUTPUTS:
    raise RuntimeError(
        'No V23 ZIP qualified. Keep V6.'
    )

print('\n'+'='*88)
print('V23 SUBMISSION ZIP(S) READY')
print('='*88)
for k,v in OUTPUTS.items():
    print(k,'->',v['package']['zip_path'])
print('Existing V6 remains the leaderboard fallback.')


In [ ]:

# 18. Final decision report.
report=dict(
    run_key=RUN_KEY,
    recovery_bank_events=int(len(TARGET_EVENT_IDS)),
    recovery_unique_waveforms=int(len(REC_BANK['records'])),
    select_missed_gt_events=int(SELECT_CLIPS.missed_gt_events.sum()),
    select_surfaced_missed_events=int(SELECT_CLIPS.surfaced_missed_events.sum()),
    select_candidate_generator_recall=float(
        SELECT_CLIPS.surfaced_missed_events.sum()/
        max(SELECT_CLIPS.missed_gt_events.sum(),1)
    ),
    select_candidates=int(len(SELECT_CAND)),
    select_positive_candidate_fraction=float(
        SELECT_CAND.positive_missed_event.mean()
    ),
    best_gate_model=BEST_GATE,
    select_variants=SELECT_VARIANTS.to_dict('records'),
    hold_final=HOLD_FINAL_TABLE.to_dict('records'),
    test_balanced_changed_clips=int(
        (TEST_ROUTE.balanced_recovered_events>0).sum()
    ),
    test_conservative_changed_clips=int(
        (TEST_ROUTE.conservative_recovered_events>0).sum()
    ),
    outputs={
        k:v['package']['zip_path']
        for k,v in OUTPUTS.items()
    },
)

put_json(
    RUN_DIR/'decision_report.json',
    report
)

print(json.dumps(
    report,
    indent=2,
    allow_nan=False
))

print('\nSEND BACK:')
print('  candidate-generator recall')
print('  GATE_RESULTS')
print('  SELECT_VARIANTS')
print('  HOLD_WAVE_TABLE')
print('  HOLD_FINAL_TABLE')
print('  decision_report')
print('  ZIP path(s), if created')



## Decision rule

This notebook is intentionally allowed to stop.

A stop before the hidden-test ZIP means the high-recall trigger does not generalize safely
enough and the scored V6 submission should remain the Phase-1 result.

If ZIPs are produced, submit **BALANCED first**, followed by **CONSERVATIVE** if a second
attempt is available. The competition's retained best score keeps V6 as the fallback.



## Final organizer code-submission ZIP

Run this cell only after the V23 run has completed successfully. It serializes the exact frozen V23 inference assets and creates `speechie.zip` with the required top-level `speechie/` folder, `main.py`, requirements, README/contact, model assets, and pinned SraVaani call.


In [24]:
# Run this INSIDE the already-executed V23 Colab notebook namespace.
# Recommended: upload this file to /content and run: %run -i /content/speechie_V23_Final_Code_Submission_Packager.py
from pathlib import Path
import os, json, pickle, hashlib, shutil, zipfile, subprocess, sys
import numpy as np

EXPECTED_RUN_KEY = "a8dfcafd81bd5717"
TEAM_NAME = "speechie"

required_names = [
    "RUN_KEY","REC_BANK","FINAL_GATE","BEST_GATE","BANK_EVENTS","BANK_TIME","BANK_PRE","BANK_SPEC","BANK_ENV","BANK_DUR",
    "bank_event_wave","CHEAP_RECIPE","SHORTLIST_K","FINE_RECIPE","V5_DIR","V6_RUN","V6_ROOT","FROZEN","V6_METHOD","V6_THRESHOLD",
    "BALANCED_THRESHOLD","SELECT_VARIANTS","HOLD_FINAL_TABLE","DECISION_REPORT"
]
missing=[n for n in required_names if n not in globals()]
# DECISION_REPORT is not a live variable in the original notebook; load it from disk if needed.
if "DECISION_REPORT" in missing:
    missing.remove("DECISION_REPORT")
if missing:
    raise RuntimeError("This packager must be run with %run -i at the END of the executed V23 notebook. Missing: "+", ".join(missing))
if str(RUN_KEY) != EXPECTED_RUN_KEY:
    raise RuntimeError(f"Refusing to package unexpected V23 run key: {RUN_KEY} != {EXPECTED_RUN_KEY}")
if str(BEST_GATE) != "logistic":
    raise RuntimeError(f"This frozen packager expects the scored run's logistic gate, got {BEST_GATE}")

PRIMARY_MOBILE = input("Primary contact mobile (include country code): ").strip()
PRIMARY_EMAIL = input("Primary contact email: ").strip()
if not PRIMARY_MOBILE or not PRIMARY_EMAIL or "@" not in PRIMARY_EMAIL:
    raise ValueError("Valid primary mobile and email are required by the organizer.")

BUILD_ROOT=Path('/content/speechie_v23_code_submission_build')
TEAM_DIR=BUILD_ROOT/TEAM_NAME
ASSET_DIR=TEAM_DIR/'assets'
if BUILD_ROOT.exists(): shutil.rmtree(BUILD_ROOT)
ASSET_DIR.mkdir(parents=True,exist_ok=True)

ENGINE_SOURCE='from __future__ import annotations\n\n"""Full-clip residual-template detector. No query labels/clean audio in inference."""\nimport os, io, re, json, math, time, hashlib, pickle, zipfile, copy\nfrom pathlib import Path\nfrom dataclasses import dataclass, asdict\nfrom collections import OrderedDict, Counter\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nimport soundfile as sf\n\nVERSION = \'residual_evidence_v5_20260917_r1\'\n\n@dataclass(frozen=True)\nclass MatcherConfig:\n    sample_rate: int = 16000\n    coarse_rate: int = 4000\n    coarse_floor: float = 0.25\n    template_batch: int = 32\n    peaks_per_template: int = 3\n    verify_topk: int = 128\n    raw_radius: int = 12\n    min_template_sec: float = 0.08\n    max_template_sec: float = 6.0\n    max_templates: int = 2000\n    chunk_sec: float = 24.0\n    fft_cache_mb: int = 384\n    merge_iou: float = 0.65\n    seed: int = 517\n\n\ndef digest(x):\n    return hashlib.sha256(json.dumps(x, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\n\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(4 << 20), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef wave_sha(y):\n    return hashlib.sha256(np.asarray(y, \'<f4\').tobytes()).hexdigest()\n\n\ndef atomic_json(path, obj):\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_name(p.name + \'.tmp\')\n    with open(tmp, \'w\', encoding=\'utf-8\') as f:\n        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)\n    os.replace(tmp, p)\n\n\ndef atomic_pickle(path, obj):\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    tmp = p.with_name(p.name + \'.tmp\')\n    with open(tmp, \'wb\') as f:\n        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp, p)\n\n\ndef event_list(v):\n    if isinstance(v, dict):\n        if \'events\' not in v:\n            raise ValueError(\'Prediction row has no events field.\')\n        v = v[\'events\']\n    out = []\n    for e in v:\n        if isinstance(e, dict):\n            a = float(e[\'onset\'] if \'onset\' in e else e[\'start\'])\n            b = float(e[\'offset\'] if \'offset\' in e else e[\'end\'])\n        else:\n            a, b = float(e[0]), float(e[1])\n        if not (math.isfinite(a) and math.isfinite(b) and 0 <= a < b):\n            raise ValueError(f\'Invalid event: {e}\')\n        out.append((a, b))\n    return out\n\n\ndef read_jsonl(path, predictions=True):\n    out = {}\n    with open(path, encoding=\'utf-8\') as f:\n        for n, line in enumerate(f, 1):\n            if not line.strip(): continue\n            r = json.loads(line)\n            cid = str(r[\'clip_id\'])\n            if cid in out: raise ValueError(f\'Duplicate clip_id on line {n}: {cid}\')\n            if predictions: event_list(r)\n            out[cid] = r\n    if not out: raise ValueError(f\'Empty JSONL: {path}\')\n    return out\n\n\ndef source_group(cid):\n    # Best-effort recording prefix, NOT a claim of verified speaker identity.\n    # Example: ...844425030213767_010_LowerDibangvalley-SPECIFIC_...\n    m = re.search(r\'_\\d+_(?:GENERIC|[^_]+-SPECIFIC)_\', str(cid))\n    return str(cid)[:m.start()] if m else str(cid).rsplit(\'_\', 3)[0]\n\n\ndef split_bucket(group):\n    return int(hashlib.sha256((\'v5_bank_source_\' + group).encode()).hexdigest()[:12], 16) % 10\n\n\ndef bandlimited_resample(y, src, dst):\n    """Windowed-sinc resampling, no SciPy/librosa dependency; common grid for pairs."""\n    y = np.asarray(y, np.float32)\n    if src == dst: return y.copy()\n    n = int(round(len(y) * dst / src))\n    if n < 1: raise ValueError(\'Audio became empty during resampling.\')\n    cutoff = min(1.0, float(dst) / src) * 0.94\n    radius = int(math.ceil(20 / cutoff))\n    offsets = np.arange(-radius, radius + 1)\n    out = np.empty(n, np.float32)\n    for a in range(0, n, 2048):\n        b = min(n, a + 2048)\n        pos = np.arange(a, b, dtype=np.float64) * src / dst\n        ix = np.floor(pos).astype(np.int64)[:, None] + offsets[None]\n        distance = pos[:, None] - ix\n        w = cutoff * np.sinc(cutoff * distance)\n        w *= np.where(np.abs(distance) < radius, 0.5 + 0.5*np.cos(np.pi*distance/radius), 0)\n        valid = (ix >= 0) & (ix < len(y))\n        w *= valid\n        w /= np.maximum(w.sum(1, keepdims=True), 1e-12)\n        out[a:b] = np.sum(y[np.clip(ix, 0, len(y)-1)] * w, axis=1)\n    return out\n\n\ndef load_audio(path_or_bytes, sr=16000):\n    obj = io.BytesIO(path_or_bytes) if isinstance(path_or_bytes, (bytes, bytearray)) else path_or_bytes\n    y, fs = sf.read(obj, dtype=\'float32\', always_2d=True)\n    y = np.mean(y, axis=1, dtype=np.float32)\n    if not len(y) or not np.isfinite(y).all(): raise ValueError(f\'Invalid audio: {str(path_or_bytes)[:100]}\')\n    return bandlimited_resample(y, int(fs), sr)\n\n\ndef condition_wave(y):\n    """Mild pre-emphasis, identically applied to templates and query recordings."""\n    y = np.asarray(y, np.float32)\n    return np.concatenate([y[:1], y[1:] - 0.85*y[:-1]]).astype(np.float32)\n\n\ndef coarse_wave(y, cfg):\n    ratio = cfg.sample_rate // cfg.coarse_rate\n    if ratio * cfg.coarse_rate != cfg.sample_rate:\n        raise ValueError(\'coarse_rate must divide sample_rate.\')\n    y = condition_wave(y)\n    t = np.arange(-32, 33, dtype=np.float64)\n    cutoff = 0.90 / ratio\n    kernel = cutoff * np.sinc(cutoff*t) * np.hanning(len(t))\n    kernel /= kernel.sum()\n    # Odd, symmetric FIR; zero padding preserves the time origin.\n    padded = np.pad(y, (32, 32))\n    return np.convolve(padded, kernel, mode=\'valid\')[::ratio].astype(np.float32)\n\n\ndef iou(a, b):\n    inter = max(0., min(a[1], b[1]) - max(a[0], b[0]))\n    union = max(a[1], b[1]) - min(a[0], b[0])\n    return inter / union if union > 0 else 0.\n\n\ndef closest_first_pairs(pred, ref):\n    # Same explicitly named local scorer used in the user\'s notebooks.\n    edges = []\n    for i, (a,b) in enumerate(pred):\n        for j, (s,e) in enumerate(ref):\n            tol = max(0.20 * (e-s), 0.05)\n            if abs(a-s) <= tol and abs(b-e) <= tol:\n                edges.append((abs(a-s) + abs(b-e), i, j))\n    used_p, used_r, pairs = set(), set(), []\n    for _, i, j in sorted(edges):\n        if i not in used_p and j not in used_r:\n            used_p.add(i); used_r.add(j); pairs.append((i,j))\n    return pairs\n\n\ndef score_events(pred, refs, durs, ids):\n    tp = fp = fn = 0; dice = []; npred = ngt = 0\n    for cid in ids:\n        p, g = event_list(pred[cid]), event_list(refs[cid])\n        nmatch = len(closest_first_pairs(p, g))\n        tp += nmatch; fp += len(p)-nmatch; fn += len(g)-nmatch\n        npred += len(p); ngt += len(g)\n        n = max(1, int(round(durs[cid]/0.01)))\n        pm = np.zeros(n, bool); gm = np.zeros(n, bool)\n        for events, mask in ((p,pm),(g,gm)):\n            for a,b in events:\n                lo,hi = max(0,int(round(a/0.01))), min(n,int(round(b/0.01)))\n                if hi > lo: mask[lo:hi] = True\n        den = int(pm.sum()+gm.sum())\n        dice.append(1. if den == 0 else 2.*np.logical_and(pm,gm).sum()/den)\n    precision = tp/(tp+fp) if tp+fp else 0.\n    recall = tp/(tp+fn) if tp+fn else 0.\n    f1 = 2.*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.\n    d = float(np.mean(dice)) if dice else 0.\n    return dict(f1=f1, precision=precision, recall=recall, dice=d, combined=f1+d,\n                tp=tp, fp=fp, fn=fn, clips=len(ids), predicted_events=npred, gt_events=ngt)\n\n\ndef wilson_lower(success, total, z=1.96):\n    if total == 0: return 0.\n    p = success/total; den = 1.+z*z/total\n    return (p+z*z/(2*total)-z*math.sqrt(p*(1-p)/total+z*z/(4*total*total)))/den\n\n\ndef build_bank(bank_ids, refs, audio_paths, clean_paths, groups, cfg, progress=lambda x, **k: x):\n    records, audits, source_hashes = [], [], []\n    sr = cfg.sample_rate\n    for cid in progress(bank_ids, desc=\'BANK pairs only: noisy minus clean\'):\n        x = load_audio(audio_paths[cid], sr)\n        s = load_audio(clean_paths[cid], sr)\n        if len(x) != len(s):\n            raise ValueError(f\'Pair length mismatch: {cid}; no silent truncation allowed.\')\n        mask = np.zeros(len(x), bool)\n        for a,b in refs[cid]:\n            mask[max(0,int(round(a*sr))):min(len(x),int(round(b*sr)))] = True\n        outside = ~mask\n        # Gain alignment is estimated from BANK-only non-event samples.\n        scale = float(np.dot(x[outside],s[outside])/(np.dot(s[outside],s[outside])+1e-12)) if outside.sum()>sr//10 else 1.\n        scale = float(np.clip(scale,0.8,1.2))\n        residual = x - scale*s\n        inside_rms = float(np.sqrt(np.mean(residual[mask]**2)+1e-20)) if mask.any() else 0.\n        outside_rms = float(np.sqrt(np.mean(residual[outside]**2)+1e-20)) if outside.any() else 0.\n        ratio = outside_rms/max(inside_rms,1e-10)\n        good = bool(inside_rms>1e-5 and ratio<=0.15)\n        audits.append(dict(cid=cid, scale=scale, inside_rms=inside_rms, outside_rms=outside_rms,\n                           outside_inside_ratio=ratio, used=good))\n        source_hashes.append(wave_sha(x))\n        if not good: continue\n        for j,(a,b) in enumerate(refs[cid]):\n            # Mixed overlapping events are not single-template examples.\n            if any(k!=j and min(b,e)-max(a,s0)>0.005 for k,(s0,e) in enumerate(refs[cid])): continue\n            if not cfg.min_template_sec <= b-a <= cfg.max_template_sec: continue\n            lo,hi = int(round(a*sr)), int(round(b*sr))\n            if lo < 0 or hi > len(x) + int(.02*sr): raise ValueError(f\'BANK label out of audio range: {cid}\')\n            hi = min(hi, len(x))\n            w = residual[lo:hi].astype(np.float32)\n            if len(w)<16 or float(np.std(w))<1e-5: continue\n            coarse = coarse_wave(w,cfg)\n            coarse -= coarse.mean()\n            norm = float(np.linalg.norm(coarse))\n            if norm < 1e-7: continue\n            raw = condition_wave(w)\n            # Reject templates with effectively just one energetic quarter.\n            chunks = np.array([np.sum(v*v) for v in np.array_split(raw,4)])\n            if np.sum(chunks > .02*max(float(chunks.sum()),1e-10)) < 2: continue\n            records.append(dict(tid=f\'{cid}::{j}\', source_cid=cid, group=groups[cid],\n                                wave=w, raw=raw, coarse=(coarse/norm).astype(np.float32),\n                                annotation_duration=b-a))\n    # Exact waveform deduplication only; no fitting on SELECT/HOLD.\n    unique = {}; dedup = 0\n    for r in records:\n        w = r[\'wave\']/max(float(np.max(np.abs(r[\'wave\']))),1e-9)\n        key = hashlib.sha256(np.round(w*32700).astype(\'<i2\').tobytes()).hexdigest()\n        if key in unique: dedup += 1\n        else: unique[key] = r\n    records = sorted(unique.values(), key=lambda r: hashlib.sha256(r[\'tid\'].encode()).hexdigest())\n    capped = max(0,len(records)-cfg.max_templates)\n    records = records[:cfg.max_templates]\n    if not records: raise RuntimeError(\'No valid additive-noise templates. Stop: the waveform-template hypothesis is unsupported for these pairs.\')\n    return dict(version=VERSION, cfg=asdict(cfg), records=records, audits=audits,\n                bank_ids=list(bank_ids), bank_groups=sorted({groups[c] for c in bank_ids}),\n                source_hashes=source_hashes, deduplicated=dedup, capped=capped)\n\n\nclass ResidualMatcher:\n    def __init__(self, bank, cfg, device=None):\n        self.cfg, self.bank = cfg, bank\n        self.device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n        self.records = bank[\'records\']\n        self.source_hashes = set(bank.get(\'source_hashes\',[]))\n        self.cache = OrderedDict(); self.cache_bytes = 0\n        self.signature = digest(dict(version=VERSION,cfg=asdict(cfg),\n            templates=[(r[\'tid\'], wave_sha(r[\'wave\'])) for r in self.records]))\n\n    def _spectrum(self, ix, nfft):\n        key = (tuple(ix), nfft)\n        if key in self.cache:\n            z = self.cache.pop(key); self.cache[key] = z\n            return z\n        t = torch.zeros((len(ix),nfft),dtype=torch.float32,device=self.device)\n        for j,i in enumerate(ix):\n            x = self.records[i][\'coarse\']; t[j,:len(x)] = torch.as_tensor(x,device=self.device)\n        z = torch.fft.rfft(t,n=nfft,dim=-1)\n        cost = z.numel()*z.element_size(); max_bytes = self.cfg.fft_cache_mb*(1<<20)\n        while self.cache and self.cache_bytes+cost > max_bytes:\n            _,old = self.cache.popitem(last=False); self.cache_bytes-=old.numel()*old.element_size()\n        if cost<=max_bytes: self.cache[key] = z; self.cache_bytes+=cost\n        return z\n\n    @torch.inference_mode()\n    def _coarse_scan(self,y,exclude_group=None):\n        x = coarse_wave(y,self.cfg); n = len(x)\n        eligible = [i for i,r in enumerate(self.records)\n                    if len(r[\'coarse\'])<=n and (exclude_group is None or r[\'group\']!=exclude_group)]\n        if not eligible: return []\n        mmax = max(len(self.records[i][\'coarse\']) for i in eligible)\n        nfft = 1<<(n+mmax-2).bit_length()\n        xt = torch.as_tensor(x,dtype=torch.float32,device=self.device)\n        xf = torch.fft.rfft(xt,n=nfft)\n        # Use float64 prefix sums to avoid variance cancellation near silence.\n        xx = xt.double()\n        pref = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),xx.cumsum(0)])\n        pref2 = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),(xx*xx).cumsum(0)])\n        starts = torch.arange(n,device=self.device)[None]\n        rows = []\n        for a in range(0,len(eligible),self.cfg.template_batch):\n            ix = eligible[a:a+self.cfg.template_batch]\n            lens = torch.tensor([len(self.records[i][\'coarse\']) for i in ix],device=self.device)[:,None]\n            tf = self._spectrum(ix,nfft)\n            dot = torch.fft.irfft(xf[None]*tf.conj(),n=nfft,dim=-1)[:,:n]\n            ends = (starts+lens).clamp(max=n)\n            total = pref[ends]-pref[starts]; total2 = pref2[ends]-pref2[starts]\n            energy = (total2-total*total/lens).clamp(min=1e-16).float()\n            q = (dot.abs()/torch.sqrt(energy)).clamp(max=1.)\n            q = q.masked_fill((starts+lens>n)|(energy<1e-12),-1.)\n            for _ in range(self.cfg.peaks_per_template):\n                vals, pos = q.max(dim=1)\n                for j,(v,p) in enumerate(zip(vals.cpu().tolist(),pos.cpu().tolist())):\n                    if v>=self.cfg.coarse_floor:\n                        rows.append(dict(template_index=ix[j], sample4k=int(p), coarse=float(v)))\n                rad = torch.maximum(lens//2,torch.full_like(lens,int(.06*self.cfg.coarse_rate)))\n                q = q.masked_fill((starts-pos[:,None]).abs()<=rad,-1.)\n        return sorted(rows,key=lambda r:r[\'coarse\'],reverse=True)[:self.cfg.verify_topk]\n\n    def _verify(self,raw_query,proposal):\n        r = self.records[proposal[\'template_index\']]; t = r[\'raw\']; m = len(t)\n        center = proposal[\'sample4k\']*(self.cfg.sample_rate//self.cfg.coarse_rate)\n        lo,hi = max(0,center-self.cfg.raw_radius), min(len(raw_query)-m,center+self.cfg.raw_radius)\n        if hi<lo: return None\n        t0=t.astype(np.float64)-np.mean(t,dtype=np.float64); ten=float(t0@t0)\n        if ten<1e-12: return None\n        seg=raw_query[lo:hi+m].astype(np.float64)\n        dots=np.correlate(seg,t0,mode=\'valid\')\n        cs=np.r_[0.,np.cumsum(seg)]; cs2=np.r_[0.,np.cumsum(seg*seg)]\n        sums=cs[m:]-cs[:-m]; energies=np.maximum(cs2[m:]-cs2[:-m]-sums*sums/m,1e-15)\n        corr=dots/np.sqrt(energies*ten)\n        j=int(np.argmax(np.abs(corr))); start=lo+j; sign=1. if corr[j]>=0 else -1.\n        qseg=raw_query[start:start+m]\n        chunks=[]\n        for inds in np.array_split(np.arange(m),4):\n            tt=t[inds].astype(np.float64); yy=qseg[inds].astype(np.float64)\n            tt-=tt.mean(); yy-=yy.mean(); et=float(tt@tt)\n            if et < .015*ten: continue\n            chunks.append(float(sign*np.dot(tt,yy)/math.sqrt(max(et*float(yy@yy),1e-20))))\n        if len(chunks)<2: return None\n        raw_corr=float(abs(corr[j])); chunk=float(np.median(chunks))\n        quality=float(np.clip(min(raw_corr,chunk),0.,1.))\n        return dict(onset=start/self.cfg.sample_rate,offset=(start+m)/self.cfg.sample_rate,\n                    quality=quality,corr16=raw_corr,corr4k=proposal[\'coarse\'],chunk_median=chunk,\n                    template_id=r[\'tid\'],template_source=r[\'source_cid\'],polarity=int(sign))\n\n    def scan(self,y,exclude_group=None):\n        """Only noisy waveform and a training bank are inputs. No event/GT arguments."""\n        y=np.asarray(y,np.float32)\n        if not np.isfinite(y).all(): raise ValueError(\'Nonfinite query audio.\')\n        if wave_sha(y) in self.source_hashes:\n            raise RuntimeError(\'Query waveform exactly duplicates a BANK source. Refusing self-retrieval.\')\n        chunk=int(self.cfg.chunk_sec*self.cfg.sample_rate)\n        overlap=int((self.cfg.max_template_sec+.05)*self.cfg.sample_rate)\n        if chunk<=overlap: raise ValueError(\'chunk_sec must exceed max_template_sec.\')\n        starts=list(range(0,max(1,len(y)-overlap),chunk-overlap))\n        found=[]\n        for a in starts:\n            yy=y[a:min(len(y),a+chunk)]\n            if len(yy)<int(self.cfg.min_template_sec*self.cfg.sample_rate): continue\n            raw=condition_wave(yy)\n            for p in self._coarse_scan(yy,exclude_group):\n                v=self._verify(raw,p)\n                if v is not None:\n                    v[\'onset\']+=a/self.cfg.sample_rate; v[\'offset\']+=a/self.cfg.sample_rate\n                    found.append(v)\n        # Keep geometric alternatives; final threshold/NMS is applied later.\n        best={}\n        for r in found:\n            k=(round(r[\'onset\'],4),round(r[\'offset\'],4),r[\'template_id\'])\n            if k not in best or r[\'quality\']>best[k][\'quality\']: best[k]=r\n        return sorted(best.values(),key=lambda r:r[\'quality\'],reverse=True)\n\n\ndef select_matches(rows, threshold, duplicate_iou=.65):\n    chosen=[]\n    for r in sorted(rows,key=lambda z:z[\'quality\'],reverse=True):\n        if r[\'quality\']<threshold: continue\n        ev=(r[\'onset\'],r[\'offset\'])\n        if any(iou(ev,(x[\'onset\'],x[\'offset\']))>=duplicate_iou or\n               (abs(ev[0]-x[\'onset\'])<=.02 and abs(ev[1]-x[\'offset\'])<=.02) for x in chosen): continue\n        chosen.append(r)\n    return sorted(chosen,key=lambda z:(z[\'onset\'],z[\'offset\']))\n\n\ndef patch_events(base, matches, max_move=.25, minimum_iou=.50, allow_add=False, add_threshold=.85):\n    """Conservative edit: only mutual one-to-one overlaps; unmatched base retained."""\n    base=event_list(base); proposal=[(m[\'onset\'],m[\'offset\']) for m in matches]\n    edges=[]\n    p2b={j:[] for j in range(len(proposal))}; b2p={i:[] for i in range(len(base))}\n    for j,p in enumerate(proposal):\n        for i,b in enumerate(base):\n            ov=iou(p,b)\n            if ov>=minimum_iou and max(abs(p[0]-b[0]),abs(p[1]-b[1]))<=max_move:\n                p2b[j].append(i); b2p[i].append(j); edges.append((i,j))\n    out=list(base); used=set(); changed=0\n    for i,j in edges:\n        if len(p2b[j])==len(b2p[i])==1:\n            out[i]=proposal[j]; used.add(j); changed+=int(out[i]!=base[i])\n    added=0\n    if allow_add:\n        for j,p in enumerate(proposal):\n            if j in used or matches[j][\'quality\']<add_threshold: continue\n            # No addition overlapping or touching an existing interval.\n            if any(min(p[1],b[1]) - max(p[0],b[0]) > -.04 for b in out): continue\n            out.append(p); added+=1\n    return sorted(out),dict(changed=changed,added=added)\n\n\nclass WaveSource:\n    """Read WAVs from one explicit folder/ZIP without extracting or guessing IDs."""\n    def __init__(self,path,required_ids):\n        self.path=str(path); self.archive=None; self.index={}\n        if Path(path).is_file() and zipfile.is_zipfile(path):\n            self.archive=zipfile.ZipFile(path)\n            names=[i.filename for i in self.archive.infolist() if not i.is_dir() and i.filename.lower().endswith(\'.wav\') and \'__MACOSX\' not in i.filename]\n        elif Path(path).is_dir():\n            names=[str(p) for p in Path(path).rglob(\'*.wav\')]\n        else: raise FileNotFoundError(path)\n        for name in names:\n            cid=Path(name).stem\n            if cid not in required_ids: continue\n            if cid in self.index: raise ValueError(f\'Duplicate waveform stem in {path}: {cid}\')\n            self.index[cid]=name\n        missing=set(required_ids)-set(self.index)\n        if missing: raise ValueError(f\'{path}: missing {len(missing)} required WAVs; examples {sorted(missing)[:3]}\')\n    def load(self,cid,sr=16000):\n        loc=self.index[cid]\n        return load_audio(self.archive.read(loc) if self.archive is not None else loc,sr)\n    def duration(self,cid):\n        loc=self.index[cid]\n        x=io.BytesIO(self.archive.read(loc)) if self.archive is not None else loc\n        info=sf.info(x)\n        return info.frames/info.samplerate\n\n\ndef persist_query(matcher,y,path,exclude_group=None):\n    wh=wave_sha(y); path=Path(path)\n    signature=digest(dict(matcher=matcher.signature,wave=wh,exclude_group=exclude_group))\n    if path.is_file():\n        with open(path) as f: old=json.load(f)\n        if old.get(\'signature\')==signature: return old[\'proposals\']\n    rows=matcher.scan(y,exclude_group)\n    atomic_json(path,dict(signature=signature,query_wave_sha256=wh,proposals=rows))\n    return rows\n\n\ndef unit_tests():\n    cfg=MatcherConfig(coarse_floor=.15,template_batch=2,peaks_per_template=2,verify_topk=20,fft_cache_mb=16)\n    rng=np.random.default_rng(601)\n    sr=cfg.sample_rate\n    t=(rng.normal(0,.15,int(.36*sr))*np.hanning(int(.36*sr))).astype(np.float32)\n    source=rng.normal(0,.02,int(1.8*sr)).astype(np.float32)\n    a=1923; source[a:a+len(t)]+=t\n    coarse=coarse_wave(t,cfg); coarse-=coarse.mean(); coarse/=np.linalg.norm(coarse)\n    bank=dict(records=[dict(tid=\'test_template\',source_cid=\'bank\',group=\'g0\',wave=t,raw=condition_wave(t),coarse=coarse)],source_hashes=[])\n    m=ResidualMatcher(bank,cfg,device=\'cpu\')\n    got=select_matches(m.scan(source),.85)\n    assert got and abs(got[0][\'onset\']-a/sr)<=1/sr, got\n    assert abs(got[0][\'offset\']-(a+len(t))/sr)<=1/sr\n    neg=m.scan(rng.normal(0,.02,len(source)).astype(np.float32))\n    assert not select_matches(neg,.85)\n    # Confirm no self-retrieval.\n    m.source_hashes.add(wave_sha(source))\n    try: m.scan(source)\n    except RuntimeError: pass\n    else: raise AssertionError(\'Self-retrieval guard failed.\')\n    refs={\'x\':[(1.,1.5)]}; durs={\'x\':2.}\n    assert score_events(refs,refs,durs,[\'x\'])[\'f1\']==1.\n    assert score_events({\'x\':[]},refs,durs,[\'x\'])[\'fn\']==1\n    row=dict(onset=1.01,offset=1.51,quality=.99)\n    patched,diag=patch_events([(1.,1.5)],[row])\n    assert len(patched)==1 and diag[\'changed\']==1\n    patched,diag=patch_events([(1.,1.5)],[dict(onset=0.,offset=.3,quality=.99)])\n    assert patched==[(1.,1.5)] and diag[\'added\']==0\n    print(\'PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.\')\n    return True\n\n"""Track-2 direct residual reconstruction. Inference never accepts a clean reference."""\nimport copy, hashlib, io, json, math, os, re, shutil, time, unicodedata, wave, zipfile\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\nimport numpy as np\nimport soundfile as sf\n\nRECON_VERSION = \'track2_v6_direct_residual_20260918_r1\'\n\n@dataclass(frozen=True)\nclass ReconConfig:\n    sample_rate: int = 16000\n    max_components: int = 16\n    duplicate_iou: float = 0.65\n    alignment_radius: int = 4\n    min_abs_gain: float = 0.01\n    max_abs_gain: float = 8.0\n    max_condition: float = 1e7\n    max_gain_disagreement: float = 0.50\n    max_removal_energy_ratio: float = 1.15\n    max_output_energy_ratio: float = 1.05\n    max_natural_modified_fraction: float = 0.05\n    ridge_relative: float = 1e-9\n    whitening_floor_fraction: float = 0.02\n    gls_iterations: int = 3\n    natural_damage_db: float = -24.0\n\n\ndef jsonable(x):\n    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}\n    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]\n    if isinstance(x, Path): return str(x)\n    if isinstance(x, np.ndarray): return jsonable(x.tolist())\n    if isinstance(x, np.generic): return jsonable(x.item())\n    if isinstance(x, float) and not math.isfinite(x): return None\n    return x\n\n\ndef put_json(path, obj):\n    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_name(p.name+\'.tmp\')\n    with open(t,\'w\',encoding=\'utf-8\') as f:\n        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)\n    os.replace(t,p)\n\n\ndef sha_file(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(4<<20),b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef sha_json(obj):\n    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\n\n\ndef canonical_pcm(y):\n    a=np.asarray(y,np.float64)\n    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():\n        raise ValueError(\'Wave must be finite, nonempty, and mono.\')\n    # Deterministic rounding, not peak normalization. Used for metrics AND export.\n    return np.clip(np.rint(a*32768.0),-32768,32767).astype(\'<i2\')\n\n\ndef pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0\n\n\ndef wav_bytes(y, sr=16000):\n    buf=io.BytesIO()\n    with wave.open(buf,\'wb\') as w:\n        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)\n        w.writeframes(canonical_pcm(y).tobytes())\n    return buf.getvalue()\n\n\ndef write_pcm(path,y,sr=16000):\n    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)\n    data=wav_bytes(y,sr); tmp=p.with_name(p.name+\'.tmp\')\n    with open(tmp,\'wb\') as f: f.write(data)\n    os.replace(tmp,p)\n    return hashlib.sha256(data).hexdigest()\n\n\ndef read_wave_strict(path_or_bytes,sr=16000):\n    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes\n    y,fs=sf.read(source,dtype=\'float32\',always_2d=True)\n    if fs!=sr or y.shape[1]!=1:\n        raise ValueError(f\'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.\')\n    a=y[:,0]\n    if not len(a) or not np.isfinite(a).all(): raise ValueError(\'Empty or invalid audio.\')\n    return a\n\n\ndef wav_audit(path_or_bytes,expected_frames,sr=16000):\n    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()\n    with wave.open(io.BytesIO(data),\'rb\') as w:\n        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())\n        wanted=(sr,1,2,int(expected_frames),\'NONE\')\n        if got!=wanted: raise ValueError(f\'WAV compliance error: {got} != {wanted}\')\n        pcm=w.readframes(expected_frames)\n        if len(pcm)!=expected_frames*2: raise ValueError(\'Truncated PCM payload.\')\n    return hashlib.sha256(data).hexdigest()\n\n\nclass StrictWaveSource:\n    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""\n    def __init__(self,path,required_ids):\n        self.path=str(path); self.archive=None; self.index={}\n        required_ids=set(map(str,required_ids))\n        p=Path(path)\n        if p.is_file() and zipfile.is_zipfile(p):\n            self.archive=zipfile.ZipFile(p)\n            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith(\'.wav\') and \'__MACOSX\' not in z.filename]\n        elif p.is_dir(): names=[str(q) for q in p.rglob(\'*\') if q.is_file() and q.suffix.lower()==\'.wav\']\n        else: raise FileNotFoundError(p)\n        for name in names:\n            cid=Path(name).stem\n            if cid not in required_ids: continue\n            if cid in self.index: raise ValueError(f\'Duplicate waveform ID: {cid} in {p}\')\n            self.index[cid]=name\n        missing=required_ids-set(self.index)\n        if missing: raise ValueError(f\'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}\')\n    def bytes(self,cid):\n        loc=self.index[cid]\n        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()\n    def load(self,cid): return read_wave_strict(self.bytes(cid))\n    def close(self):\n        if self.archive is not None: self.archive.close()\n\n\ndef si_sdr(estimate,reference):\n    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""\n    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)\n    if x.shape!=s.shape or x.ndim!=1: raise ValueError(\'SI-SDR length/shape mismatch.\')\n    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError(\'Nonfinite SI-SDR input.\')\n    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)\n    if ss<=1e-18: raise ValueError(\'Silent reference: SI-SDR undefined, not silently excluded.\')\n    target=(float(x@s)/ss)*s; e=x-target\n    num=float(target@target); den=float(e@e)\n    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))\n    return float(min(120.,value))\n\n\ndef residual_change_db(a,b):\n    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64)\n    return float(10*np.log10(max(float((a-b)@(a-b)),1e-24)/max(float(b@b),1e-24)))\n\n\ndef fit_centered(A,y,weights=None,ridge=1e-9):\n    A=np.asarray(A,np.float64); y=np.asarray(y,np.float64)\n    if A.ndim==1: A=A[:,None]\n    w=np.ones(len(y),np.float64) if weights is None else np.asarray(weights,np.float64)\n    w=np.clip(w,0,None); w=w/max(float(w.mean()),1e-30)\n    am=(A*w[:,None]).sum(0)/max(w.sum(),1e-30); ym=float((y*w).sum()/max(w.sum(),1e-30))\n    B=A-am; v=y-ym\n    gram=B.T@(w[:,None]*B); rhs=B.T@(w*v)\n    diag=np.diag(gram); scale=np.sqrt(np.maximum(diag,1e-30))\n    G=gram/scale[:,None]/scale[None,:]; r=rhs/scale\n    cond=float(np.linalg.cond(G))\n    g=np.linalg.solve(G+ridge*np.eye(G.shape[0]),r)/scale\n    return g,cond\n\n\ndef emphasis(A,y):\n    return A[1:]-.90*A[:-1],y[1:]-.90*y[:-1]\n\n\ndef residual_psd(residual,nfft,sr=16000):\n    nwin=min(512,len(residual))\n    if nwin<16: return np.ones(nfft//2+1)\n    hop=max(1,nwin//2); win=np.hanning(nwin)\n    starts=range(0,len(residual)-nwin+1,hop)\n    # Median-Welch PSD limits unusually loud speech frames without target labels.\n    frames=np.stack([residual[a:a+nwin]*win for a in starts])\n    p=np.median(np.abs(np.fft.rfft(frames,n=512,axis=1))**2,axis=0)\n    p=np.convolve(np.pad(p,(3,3),mode=\'edge\'),np.ones(7)/7,mode=\'valid\')\n    return np.interp(np.linspace(0,sr/2,nfft//2+1),np.linspace(0,sr/2,len(p)),p)\n\n\ndef fit_spectral(A,y,cfg):\n    g,_=fit_centered(*emphasis(A,y),ridge=cfg.ridge_relative)\n    nfft=1<<(len(y)-1).bit_length()\n    AF=np.fft.rfft(A,n=nfft,axis=0); YF=np.fft.rfft(y,n=nfft)\n    freq=np.linspace(0,cfg.sample_rate/2,len(YF))\n    cond=1.\n    for _ in range(cfg.gls_iterations):\n        psd=residual_psd(y-A@g,nfft,cfg.sample_rate)\n        positive=psd[psd>0]\n        floor=max(float(np.median(positive))*cfg.whitening_floor_fraction if len(positive) else 0.,1e-20)\n        w=1./np.maximum(psd,floor); w/=max(float(np.median(w)),1e-30)\n        w=np.clip(w,1e-3,1e3); w[freq<70.]=0.\n        # One-sided real FFT Parseval weights.\n        w[1:-1]*=2.\n        gram=(AF.conj().T@(w[:,None]*AF)).real\n        rhs=(AF.conj().T@(w*YF)).real\n        scale=np.sqrt(np.maximum(np.diag(gram),1e-30))\n        G=gram/scale[:,None]/scale[None,:]\n        cond=float(np.linalg.cond(G))\n        g=np.linalg.solve(G+cfg.ridge_relative*np.eye(len(g)),rhs/scale)/scale\n    return g,cond\n\n\ndef estimate_gains(A,y,polarity,method,cfg):\n    if method==\'stored_gain\': return np.asarray(polarity,np.float64),1.\n    if method==\'mixture_ols\': return fit_centered(A,y,ridge=cfg.ridge_relative)\n    P,v=emphasis(A,y)\n    if method==\'preemphasis_ols\': return fit_centered(P,v,ridge=cfg.ridge_relative)\n    if method==\'spectral_gls\': return fit_spectral(A,y,cfg)\n    if method==\'block_irls\':\n        g,cond=fit_centered(P,v,ridge=cfg.ridge_relative)\n        block=max(64,int(.030*cfg.sample_rate))\n        idx=np.arange(len(v))//block\n        for _ in range(cfg.gls_iterations):\n            residual=v-P@g\n            counts=np.bincount(idx); power=np.bincount(idx,weights=residual**2)/counts\n            floor=max(float(np.median(power))*.05,1e-15)\n            bw=1./np.maximum(power,floor); bw/=max(float(np.median(bw)),1e-30)\n            weights=np.clip(bw,0.02,50.)[idx]\n            g,cond=fit_centered(P,v,weights,cfg.ridge_relative)\n        return g,cond\n    raise ValueError(f\'Unknown deployable gain method: {method}\')\n\n\ndef gain_disagreement(A,y,g,cfg):\n    # Two interleaved block fits: agreement diagnostic, not a probability of correctness.\n    P,v=emphasis(A,y); block=max(64,int(.020*cfg.sample_rate))\n    parity=(np.arange(len(v))//block)%2\n    folds=[]\n    for f in (0,1):\n        mask=parity==f\n        if mask.sum()<4*A.shape[1] or np.any((P[mask]**2).sum(0)<1e-14): return math.inf\n        try: gg,cc=fit_centered(P[mask],v[mask],ridge=cfg.ridge_relative)\n        except np.linalg.LinAlgError: return math.inf\n        if cc>cfg.max_condition: return math.inf\n        folds.append(gg)\n    return float(np.max(np.abs(folds[0]-folds[1])/np.maximum(np.abs(g),.025)))\n\n\ndef refine_placement(y,t,start,radius=4):\n    """Integer-sample correlation refinement. The raw template is never faded/stretched."""\n    t=np.asarray(t,np.float64); dt=t[1:]-.9*t[:-1]; dt=dt-dt.mean()\n    ten=float(dt@dt)\n    if ten<1e-18: return start\n    best=(-math.inf,start)\n    for a in range(max(0,start-radius),min(len(y)-len(t),start+radius)+1):\n        seg=np.asarray(y[a:a+len(t)],np.float64); ds=seg[1:]-.9*seg[:-1]; ds-=ds.mean()\n        q=abs(float(ds@dt))/math.sqrt(max(ten*float(ds@ds),1e-30))\n        # Stable tie-breaking favours the verified V5 start.\n        score=q-1e-12*abs(a-start)\n        if score>best[0]: best=(score,a)\n    return best[1]\n\n\ndef interval_iou(a,b):\n    inter=max(0,min(a[1],b[1])-max(a[0],b[0])); union=max(a[1],b[1])-min(a[0],b[0])\n    return inter/union if union>0 else 0.\n\n\ndef prepared_matches(y,proposals,bank_by_id,cfg):\n    """Prepare ALL verified alternatives once; threshold/NMS are applied afterwards."""\n    out=[]\n    for p in proposals:\n        tid=p[\'template_id\']\n        if tid not in bank_by_id: raise ValueError(f\'Unknown cached template {tid}; wrong bank.\')\n        t=np.asarray(bank_by_id[tid][\'wave\'],np.float64)\n        start=int(round(float(p[\'onset\'])*cfg.sample_rate))\n        if start<0 or start+len(t)>len(y): continue\n        if abs((float(p[\'offset\'])-float(p[\'onset\']))*cfg.sample_rate-len(t))>2:\n            raise ValueError(\'Match duration is incompatible with frozen template length.\')\n        a=refine_placement(y,t,start,cfg.alignment_radius)\n        out.append(dict(start=int(a),end=int(a+len(t)),quality=float(p[\'quality\']),\n                        polarity=int(p[\'polarity\']),template_id=tid,wave=t))\n    return out\n\n\ndef select_evidence(prepared,threshold,cfg):\n    keep=[]\n    for r in sorted(prepared,key=lambda q:q[\'quality\'],reverse=True):\n        if r[\'quality\']<threshold: continue\n        if any(interval_iou((r[\'start\'],r[\'end\']),(v[\'start\'],v[\'end\']))>=cfg.duplicate_iou for v in keep): continue\n        keep.append(r)\n        if len(keep)>=cfg.max_components: break\n    return sorted(keep,key=lambda q:q[\'start\'])\n\n\ndef overlap_components(rows):\n    groups=[]\n    for r in sorted(rows,key=lambda q:q[\'start\']):\n        if groups and r[\'start\']<max(x[\'end\'] for x in groups[-1]): groups[-1].append(r)\n        else: groups.append([r])\n    return groups\n\n\ndef component_matrix(y,rows):\n    lo=max(0,min(r[\'start\'] for r in rows)-32)\n    hi=min(len(y),max(r[\'end\'] for r in rows)+32)\n    A=np.zeros((hi-lo,len(rows)),np.float64)\n    for j,r in enumerate(rows): A[r[\'start\']-lo:r[\'end\']-lo,j]=r[\'wave\']\n    return lo,hi,A,np.asarray(y[lo:hi],np.float64)\n\n\ndef reconstruct(y,prepared,policy,cfg):\n    """Deployable path: input mixture, template matches, frozen policy. NO references."""\n    y=np.asarray(y,np.float64)\n    if y.ndim!=1 or not len(y) or not np.isfinite(y).all(): raise ValueError(\'Invalid noisy waveform.\')\n    selected=select_evidence(prepared,float(policy[\'threshold\']),cfg)\n    z=y.copy(); log=[]; accepted=[]\n    for rows in overlap_components(selected):\n        lo,hi,A,v=component_matrix(y,rows)\n        why=None; cond=None; disagreement=None; gains=None\n        try:\n            gains,cond=estimate_gains(A,v,[r[\'polarity\'] for r in rows],policy[\'method\'],cfg)\n            if not np.isfinite(gains).all() or cond>cfg.max_condition: why=\'ill_conditioned_fit\'\n            if why is None and (np.any(np.abs(gains)<cfg.min_abs_gain) or np.any(np.abs(gains)>cfg.max_abs_gain)):\n                why=\'gain_out_of_bounds\'\n            if why is None and any(float(g)*r[\'polarity\']<=0 for g,r in zip(gains,rows)): why=\'gain_polarity_mismatch\'\n            if why is None:\n                disagreement=gain_disagreement(A,v,gains,cfg)\n                if disagreement>cfg.max_gain_disagreement: why=\'inconsistent_gain\'\n            noise=A@gains if gains is not None else np.zeros(len(v))\n            removed_ratio=float(noise@noise)/max(float(v@v),1e-20)\n            candidate=v-noise\n            if why is None and removed_ratio>cfg.max_removal_energy_ratio: why=\'excessive_removal_energy\'\n            if why is None and float(candidate@candidate)>cfg.max_output_energy_ratio*max(float(v@v),1e-20): why=\'output_energy_increase\'\n            if why is None and (candidate.min()<-1. or candidate.max()>32767./32768.+1e-8): why=\'would_clip_pcm16\'\n        except (np.linalg.LinAlgError,FloatingPointError,ValueError) as e:\n            why=f\'numerical_fit_rejected:{type(e).__name__}\'\n        if why is None:\n            # Reconstruct from ORIGINAL mixture, once. No generic mask or second fade.\n            z[lo:hi]-=noise\n            accepted.extend(dict(template_id=r[\'template_id\'],start=r[\'start\'],end=r[\'end\'],gain=float(g),quality=r[\'quality\'])\n                            for g,r in zip(gains,rows))\n        log.append(dict(template_ids=[r[\'template_id\'] for r in rows],accepted=why is None,reason=why,\n                        condition=cond,gain_disagreement=disagreement,\n                        gains=None if gains is None else gains.tolist()))\n    q=pcm_roundtrip(z); base=pcm_roundtrip(y)\n    return q,dict(method=policy[\'method\'],threshold=float(policy[\'threshold\']),\n                  proposed=len(prepared),selected=len(selected),accepted_templates=len(accepted),\n                  changed=bool(np.any(q!=base)),change_db=residual_change_db(q,base),components=log,matches=accepted)\n\n\ndef reference_assisted_diagnostic(y,clean,prepared,policy,cfg):\n    """DIAGNOSTIC ONLY. Same noisy-only matches, but clean is a nuisance regressor for ideal gains.\n    Never exported as audio. Does NOT search using clean/GT and is NOT a deployable score.\n    """\n    y=np.asarray(y,np.float64); clean=np.asarray(clean,np.float64)\n    z=y.copy(); nfit=0\n    for rows in overlap_components(select_evidence(prepared,policy[\'threshold\'],cfg)):\n        lo,hi,A,v=component_matrix(y,rows)\n        B=np.column_stack((A,clean[lo:hi]))\n        try:\n            g,_=fit_centered(B,v,ridge=1e-12)\n            if np.isfinite(g).all(): z[lo:hi]-=A@g[:-1]; nfit+=len(rows)\n        except np.linalg.LinAlgError: pass\n    return dict(reference_assisted_sisdr_float=si_sdr(z,clean),\n                reference_assisted_sisdr_pcm16=si_sdr(pcm_roundtrip(z),clean),fitted_templates=nfit)\n\n\nTAG_RE=re.compile(r\'</?[^<>]*>|\\[[^\\[\\]]*\\]\')\ndef normalize_text(text):\n    s=TAG_RE.sub(\' \',str(text))\n    s=\'\'.join(\' \' if unicodedata.category(c).startswith(\'P\') else c for c in s)\n    return \' \'.join(s.lower().split())\n\n\ndef word_errors(reference,hypothesis):\n    a=normalize_text(reference).split(); b=normalize_text(hypothesis).split()\n    row=list(range(len(b)+1))\n    for i,x in enumerate(a,1):\n        nxt=[i]\n        for j,y in enumerate(b,1): nxt.append(min(nxt[-1]+1,row[j]+1,row[j-1]+(x!=y)))\n        row=nxt\n    return row[-1],len(a)\n\n\ndef pooled_wer(ids,refs,transcripts):\n    errors=words=0\n    for cid in ids:\n        e,n=word_errors(refs[cid],transcripts[cid][\'text\']); errors+=e; words+=n\n    if words==0: raise ValueError(\'No reference words: real WER cannot be evaluated.\')\n    return dict(wer=errors/words,errors=errors,words=words,clips=len(ids))\n\n\ndef hyp_text(h):\n    if isinstance(h,str): return h.strip()\n    if isinstance(h,dict) and isinstance(h.get(\'text\'),str): return h[\'text\'].strip()\n    if hasattr(h,\'text\') and isinstance(h.text,str): return h.text.strip()\n    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])\n    raise TypeError(f\'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.\')\n\n\nclass HashASR:\n    """Model-revision + exact WAV byte hash cache. Never reuses an ID-only transcript."""\n    def __init__(self,model,model_signature,cache_dir,batch_size=8):\n        self.model=model; self.model_signature=model_signature\n        self.cache_dir=Path(cache_dir); self.cache_dir.mkdir(parents=True,exist_ok=True)\n        self.batch_size=batch_size\n    def _call(self,paths):\n        try:\n            hs=self.model.transcribe([str(p) for p in paths],return_hypotheses=True)\n            if isinstance(hs,(list,tuple)) and len(hs)==len(paths): return [hyp_text(h) for h in hs]\n            if len(paths)==1: return [hyp_text(hs)]\n        except Exception:\n            if len(paths)==1: raise\n        # A cardinality mismatch never silently truncates zip(...).\n        return [self._call([p])[0] for p in paths]\n    def transcribe(self,paths,progress=lambda x,**k:x):\n        out={}; todo={}\n        for cid,path in paths.items():\n            sh=sha_file(path); key=sha_json(dict(model=self.model_signature,wav=sh))\n            dest=self.cache_dir/(key+\'.json\')\n            rec=None\n            if dest.is_file():\n                with open(dest,encoding=\'utf-8\') as f: r=json.load(f)\n                if r.get(\'model_signature\')==self.model_signature and r.get(\'wav_sha256\')==sh and isinstance(r.get(\'text\'),str): rec=r\n            if rec is not None: out[cid]=rec\n            else:\n                if key not in todo: todo[key]=dict(path=str(path),sha=sh,ids=[],dest=dest)\n                todo[key][\'ids\'].append(cid)\n        jobs=list(todo.values())\n        print(f\'ASR exact-hash hits: {len(out)} | unique WAVs to decode: {len(jobs)}\')\n        for a in progress(range(0,len(jobs),self.batch_size),desc=\'SraVaani final-waveform ASR\'):\n            ch=jobs[a:a+self.batch_size]; texts=self._call([j[\'path\'] for j in ch])\n            if len(texts)!=len(ch): raise RuntimeError(\'ASR cardinality mismatch.\')\n            for job,text in zip(ch,texts):\n                # Empty output is allowed only when the real recognizer returned it.\n                rec=dict(text=text,wav_sha256=job[\'sha\'],model_signature=self.model_signature)\n                put_json(job[\'dest\'],rec)\n                for cid in job[\'ids\']: out[cid]=rec\n        if set(out)!=set(paths): raise RuntimeError(\'Incomplete ASR coverage.\')\n        return out\n\n\ndef pack_submission(paths,asr,expected_frames,output_zip,sr=16000):\n    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""\n    ids=sorted(paths)\n    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError(\'Submission ID mismatch.\')\n    for cid in ids:\n        if not cid or Path(cid).name!=cid or any(c in cid for c in (\'/\',\'\\\\\',\'\\n\',\'\\r\')): raise ValueError(\'Unsafe clip ID.\')\n        h=wav_audit(paths[cid],expected_frames[cid],sr)\n        if h!=asr[cid][\'wav_sha256\']: raise ValueError(f\'ASR/audio hash mismatch: {cid}\')\n        if not isinstance(asr[cid][\'text\'],str): raise ValueError(\'ASR text must be a string.\')\n    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+\'.tmp\')\n    tr=\'\'.join(json.dumps(dict(clip_id=c,text=asr[c][\'text\']),ensure_ascii=False,allow_nan=False)+\'\\n\' for c in ids).encode(\'utf-8\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4,allowZip64=True) as z:\n        z.writestr(\'transcripts.jsonl\',tr)\n        for cid in ids: z.write(paths[cid],cid+\'.wav\')\n    with zipfile.ZipFile(tmp) as z:\n        wanted={\'transcripts.jsonl\'}|{c+\'.wav\' for c in ids}\n        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError(\'Archive contents mismatch.\')\n        if z.testzip() is not None: raise ValueError(\'Archive CRC failure.\')\n        for cid in ids:\n            data=z.read(cid+\'.wav\')\n            if wav_audit(data,expected_frames[cid],sr)!=asr[cid][\'wav_sha256\']: raise ValueError(\'Packaged audio mismatch.\')\n        rows=[json.loads(x) for x in z.read(\'transcripts.jsonl\').decode().splitlines()]\n        if len(rows)!=len(ids) or {r[\'clip_id\'] for r in rows}!=set(ids): raise ValueError(\'Packaged transcript mismatch.\')\n    os.replace(tmp,p)\n    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())\n'
V23_RUNTIME_SOURCE='from __future__ import annotations\nimport math\nimport numpy as np\nfrom speechie_engine import (\n    MatcherConfig, ResidualMatcher, prepared_matches, reconstruct,\n    interval_iou, pcm_roundtrip\n)\n\nSR = 16000\nTIME_BINS = 256\nENV_BINS = 64\nSPEC_BINS = 96\nFINE_SHIFT_GRID = [-64,-32,-16,-8,0,8,16,32,64]\nFIR_TAPS = 5\nFIR_RIDGE_REL = 2e-3\nFIR_MAX_SAMPLES = 10000\n\nCHEAP_RECIPES = {\n    \'balanced\':dict(time=.22,pre=.36,spec=.27,env=.10,dur=.05),\n    \'preemph\':dict(time=.10,pre=.53,spec=.24,env=.08,dur=.05),\n    \'spectral\':dict(time=.10,pre=.28,spec=.47,env=.10,dur=.05),\n    \'temporal\':dict(time=.32,pre=.43,spec=.12,env=.08,dur=.05),\n}\nFINE_RECIPES = {\n    \'balanced\':dict(cheap=.08,raw=.08,p85=.13,p95=.22,chunk=.20,stable=.06,fir_corr=.15,spec=.05,fitfeat=.03),\n    \'pre_chunk\':dict(cheap=.06,raw=.07,p85=.17,p95=.27,chunk=.27,stable=.07,fir_corr=.05,spec=.03,fitfeat=.01),\n    \'fir_heavy\':dict(cheap=.07,raw=.05,p85=.10,p95=.18,chunk=.14,stable=.06,fir_corr=.29,spec=.05,fitfeat=.06),\n    \'v6_plus\':dict(cheap=.05,raw=.15,p85=.24,p95=.17,chunk=.29,stable=.04,fir_corr=.03,spec=.02,fitfeat=.01),\n}\nCANDIDATE_FEATURES = [\n    \'hr_quality\',\'hr_corr16\',\'hr_corr4k\',\'hr_chunk_median\',\'duration_sec\',\n    \'v13_score\',\'v13_gap\',\'v13_raw\',\'v13_p85\',\'v13_p95\',\'v13_chunk\',\'v13_stable\',\n    \'v13_fir_corr\',\'v13_spec\',\'v13_fitfeat\',\'abs_gain\',\'removed_energy_ratio\',\'mix_corr\',\n    \'mix_fit_db\',\'proposal_retrieved_wave_corr\'\n]\n\ndef direct_fit_db(pred,target,eps=1e-12):\n    pred=np.asarray(pred,np.float64); target=np.asarray(target,np.float64)\n    return float(10*np.log10((float(target@target)+eps)/(float((target-pred)@(target-pred))+eps)))\n\ndef resample_1d(x,n):\n    x=np.asarray(x,np.float32)\n    if len(x)==n:return x.copy()\n    if len(x)<2:return np.full(n,float(x[0]) if len(x) else 0,np.float32)\n    return np.interp(np.linspace(0,1,n),np.linspace(0,1,len(x)),x).astype(np.float32)\n\ndef shift_zeropad(x,shift):\n    x=np.asarray(x,np.float32); out=np.zeros_like(x)\n    if shift==0:return x.copy()\n    if shift>0:\n        if shift<len(x):out[shift:]=x[:-shift]\n    else:\n        k=-shift\n        if k<len(x):out[:-k]=x[k:]\n    return out\n\ndef preemph(x,a=.95):\n    x=np.asarray(x,np.float32)\n    return np.r_[x[:1],x[1:]-float(a)*x[:-1]].astype(np.float32) if len(x) else x.copy()\n\ndef signed_corr(a,b,eps=1e-12):\n    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64); a=a-a.mean(); b=b-b.mean()\n    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))\n\ndef cosine_abs(a,b):\n    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64)\n    return float(abs(np.dot(a,b))/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))\n\ndef unit_z(x):\n    x=np.asarray(x,np.float64); x=x-x.mean()\n    return (x/max(np.linalg.norm(x),1e-12)).astype(np.float32)\n\ndef time_feature(w,bins=TIME_BINS,a=0.):\n    z=preemph(w,a) if a else np.asarray(w,np.float32)\n    return unit_z(resample_1d(z,bins))\n\ndef env_feature(w,bins=ENV_BINS):\n    return unit_z(np.abs(resample_1d(np.asarray(w,np.float32),bins)))\n\ndef spectral_feature(w,bins=SPEC_BINS):\n    w=preemph(np.asarray(w,np.float32),.95)\n    if len(w)<32:w=np.pad(w,(0,32-len(w)))\n    nfft=1<<int(np.ceil(np.log2(min(max(len(w),256),16384))))\n    z=np.log1p(np.abs(np.fft.rfft(w*np.hanning(len(w)),n=nfft)))\n    e=np.linspace(0,len(z),bins+1).astype(int)\n    p=np.array([z[e[i]:max(e[i]+1,e[i+1])].mean() for i in range(bins)])\n    return unit_z(p)\n\ndef scalar_fit(t,q):\n    t=np.asarray(t,np.float64);q=np.asarray(q,np.float64)\n    g=float(np.dot(t,q)/(np.dot(t,t)+1e-12)); return (g*t).astype(np.float32),g\n\ndef chunk_consistency_v13(t,q,sign,chunks=4):\n    vals=[];g=[]\n    for ind in np.array_split(np.arange(len(t)),chunks):\n        if len(ind)<16:continue\n        a=np.asarray(t[ind],float);b=np.asarray(q[ind],float);ac=a-a.mean();bc=b-b.mean();ea=ac@ac;eb=bc@bc\n        if ea<1e-12 or eb<1e-12:continue\n        vals.append(float(sign*(ac@bc)/math.sqrt(ea*eb))); g.append(float((a@b)/(a@a+1e-12)))\n    if not vals:return 0.,0.\n    ga=np.asarray(g);med=np.median(ga);mad=np.median(np.abs(ga-med))\n    return float(np.clip(np.median(vals),0,1)),float(np.clip(np.exp(-mad/(abs(med)+1e-4)),0,1))\n\ndef fir_design(x,taps=FIR_TAPS):\n    h=taps//2; return np.stack([shift_zeropad(x,d) for d in range(-h,h+1)],1).astype(np.float64)\n\ndef fir_fit_noisy(t,q):\n    X=fir_design(np.asarray(t,np.float32),FIR_TAPS);q=np.asarray(q,np.float32)\n    if len(q)>FIR_MAX_SAMPLES:\n        ix=np.linspace(0,len(q)-1,FIR_MAX_SAMPLES).astype(int);Xs=X[ix];qs=q[ix].astype(float)\n    else: Xs=X;qs=q.astype(float)\n    G=Xs.T@Xs;ridge=FIR_RIDGE_REL*(np.trace(G)/max(1,G.shape[0])+1e-9);A=G+ridge*np.eye(G.shape[0]);b=Xs.T@qs\n    try:h=np.linalg.solve(A,b)\n    except np.linalg.LinAlgError:h=np.linalg.lstsq(A,b,rcond=None)[0]\n    return (X@h).astype(np.float32),h.astype(np.float32)\n\ndef fine_noisy_metrics(q,c,cheap=0.):\n    q=np.asarray(q,np.float32);c=np.asarray(c,np.float32);warped=resample_1d(c,len(q));q85=preemph(q,.85);q95=preemph(q,.95);best=None\n    for sh in FINE_SHIFT_GRID:\n        t=shift_zeropad(warped,sh);cr=signed_corr(t,q);sign=1 if cr>=0 else -1;raw=abs(cr)\n        p85=abs(signed_corr(preemph(t,.85),q85));p95=abs(signed_corr(preemph(t,.95),q95));chunk,stable=chunk_consistency_v13(t,q,sign)\n        base=.18*raw+.34*p95+.20*p85+.23*chunk+.05*stable\n        if best is None or base>best[\'base\']:best=dict(base=base,shift=sh,t=t,raw=raw,p85=p85,p95=p95,chunk=chunk,stable=stable)\n    t=best.pop(\'t\');sp,g=scalar_fit(t,q);fp,h=fir_fit_noisy(t,q)\n    best.update(cheap=float(cheap),scalar_corr=cosine_abs(sp,q),fir_corr=cosine_abs(fp,q),spec=float(np.clip((spectral_feature(t)@spectral_feature(q)+1)/2,0,1)),fitfeat=float(np.tanh(max(0,direct_fit_db(fp,q))/12)),scalar_gain=float(g))\n    return best\n\ndef fine_score(m,recipe):\n    return float(sum(float(w)*float(m[k]) for k,w in FINE_RECIPES[recipe].items()))\n\nclass V13BankRuntime:\n    def __init__(self,obj):\n        self.waves=[np.asarray(w,np.float32) for w in obj[\'waves\']]\n        self.time=np.asarray(obj[\'time\'],np.float32);self.pre=np.asarray(obj[\'pre\'],np.float32);self.spec=np.asarray(obj[\'spec\'],np.float32);self.env=np.asarray(obj[\'env\'],np.float32);self.dur=np.asarray(obj[\'dur\'],float)\n        self.cheap_recipe=str(obj[\'cheap_recipe\']);self.shortlist_k=int(obj[\'shortlist_k\']);self.fine_recipe=str(obj[\'fine_recipe\'])\n    def cheap_components(self,q,dur):\n        qt=time_feature(q,a=0.);qp=time_feature(q,a=.95);qs=spectral_feature(q);qe=env_feature(q)\n        return dict(time=np.abs(self.time@qt),pre=np.abs(self.pre@qp),spec=np.clip((self.spec@qs+1)/2,0,1),env=np.clip((self.env@qe+1)/2,0,1),dur=np.exp(-np.abs(np.log((self.dur+1e-4)/(dur+1e-4)))/.35))\n    def retrieve(self,q):\n        q=np.asarray(q,np.float32);comp=self.cheap_components(q,len(q)/SR);w=CHEAP_RECIPES[self.cheap_recipe];cs=sum(float(w[k])*comp[k] for k in w).astype(float)\n        order=np.argsort(-cs)[:self.shortlist_k];ranked=[]\n        for bi in order:\n            if not np.isfinite(cs[bi]):continue\n            m=fine_noisy_metrics(q,self.waves[int(bi)],float(cs[bi]));ranked.append((int(bi),m,fine_score(m,self.fine_recipe)))\n        ranked.sort(key=lambda z:z[2],reverse=True)\n        if not ranked:return None\n        top=ranked[0];second=ranked[1][2] if len(ranked)>1 else np.nan\n        return dict(bank_index=top[0],score=float(top[2]),score_gap=float(top[2]-second) if np.isfinite(second) else np.nan,metrics=top[1])\n\nclass LogisticGateRuntime:\n    def __init__(self,obj):\n        self.statistics=np.asarray(obj[\'imputer_statistics\'],float);self.mean=np.asarray(obj[\'scaler_mean\'],float);self.scale=np.asarray(obj[\'scaler_scale\'],float);self.coef=np.asarray(obj[\'coef\'],float).reshape(-1);self.intercept=float(obj[\'intercept\'])\n    def predict_proba(self,rows):\n        X=np.asarray(rows,float)\n        if X.ndim==1:X=X[None,:]\n        X=np.where(np.isfinite(X),X,self.statistics[None,:]);X=(X-self.mean[None,:])/self.scale[None,:]\n        logits=X@self.coef+self.intercept\n        p=1.0/(1.0+np.exp(-np.clip(logits,-60,60)))\n        return p\n\ndef interval_gap_samples(a,b):\n    if a[1]<b[0]:return b[0]-a[1]\n    if b[1]<a[0]:return a[0]-b[1]\n    return 0\n\ndef overlaps_v6_accepted(start,end,accepted,margin_ms=20.0):\n    margin=int(round(float(margin_ms)*SR/1000.0))\n    for m in accepted:\n        a=(int(m[\'start\']),int(m[\'end\']));b=(int(start),int(end))\n        if interval_iou(a,b)>0 or interval_gap_samples(a,b)<=margin:return True\n    return False\n\ndef recovery_proposals(y,accepted,rec_matcher,cfg):\n    rows=rec_matcher.scan(y,exclude_group=None);keep=[]\n    for p in sorted(rows,key=lambda z:z[\'quality\'],reverse=True):\n        if float(p[\'quality\'])<float(cfg[\'min_verify_quality\']):continue\n        lo=int(round(float(p[\'onset\'])*SR));hi=int(round(float(p[\'offset\'])*SR))\n        if lo<0 or hi>len(y) or hi<=lo:continue\n        if overlaps_v6_accepted(lo,hi,accepted,cfg[\'accepted_exclusion_margin_ms\']):continue\n        if any(interval_iou((lo,hi),(q[\'lo\'],q[\'hi\']))>=float(cfg[\'nms_iou\']) for q in keep):continue\n        q=dict(p);q[\'lo\']=lo;q[\'hi\']=hi;keep.append(q)\n        if len(keep)>=int(cfg[\'max_candidates_per_clip\']):break\n    return keep\n\ndef single_candidate_fit(y,p,rec_bank_by_id,rcfg,v6_method):\n    prep=prepared_matches(y,[p],rec_bank_by_id,rcfg)\n    if not prep:return None\n    _,info=reconstruct(y,prep,dict(method=v6_method,threshold=0.0),rcfg)\n    if int(info[\'accepted_templates\'])!=1 or len(info[\'matches\'])!=1:return None\n    m=info[\'matches\'][0];tid=str(m[\'template_id\']);w=np.asarray(rec_bank_by_id[tid][\'wave\'],np.float32);lo=int(m[\'start\']);hi=int(m[\'end\'])\n    if hi-lo!=len(w):return None\n    pred=(float(m[\'gain\'])*w).astype(np.float32);q=np.asarray(y[lo:hi],np.float32);eng=max(float(np.dot(q,q)),1e-20)\n    return dict(start=lo,end=hi,template_id=tid,gain=float(m[\'gain\']),pred=pred,abs_gain=float(abs(m[\'gain\'])),removed_energy_ratio=float(np.dot(pred,pred)/eng),mix_corr=float(cosine_abs(pred,q)),mix_fit_db=float(direct_fit_db(pred,q)))\n\ndef candidate_record(y,p,rec_bank_by_id,rcfg,v6_method,v13):\n    fit=single_candidate_fit(y,p,rec_bank_by_id,rcfg,v6_method)\n    if fit is None:return None\n    lo,hi=fit[\'start\'],fit[\'end\'];q=np.asarray(y[lo:hi],np.float32);ret=v13.retrieve(q)\n    if ret is None:return None\n    rm=ret[\'metrics\'];retrieved=v13.waves[int(ret[\'bank_index\'])];prop_wave=np.asarray(rec_bank_by_id[str(fit[\'template_id\'])][\'wave\'],np.float32);retrieved_warp=resample_1d(retrieved,len(prop_wave))\n    feats=dict(hr_quality=float(p[\'quality\']),hr_corr16=float(p.get(\'corr16\',np.nan)),hr_corr4k=float(p.get(\'corr4k\',np.nan)),hr_chunk_median=float(p.get(\'chunk_median\',np.nan)),duration_sec=float((hi-lo)/SR),v13_score=float(ret[\'score\']),v13_gap=float(ret[\'score_gap\']) if np.isfinite(ret[\'score_gap\']) else np.nan,v13_raw=float(rm[\'raw\']),v13_p85=float(rm[\'p85\']),v13_p95=float(rm[\'p95\']),v13_chunk=float(rm[\'chunk\']),v13_stable=float(rm[\'stable\']),v13_fir_corr=float(rm[\'fir_corr\']),v13_spec=float(rm[\'spec\']),v13_fitfeat=float(rm[\'fitfeat\']),abs_gain=float(fit[\'abs_gain\']),removed_energy_ratio=float(fit[\'removed_energy_ratio\']),mix_corr=float(fit[\'mix_corr\']),mix_fit_db=float(fit[\'mix_fit_db\']),proposal_retrieved_wave_corr=float(cosine_abs(prop_wave,retrieved_warp)))\n    return dict(start=lo,end=hi,template_id=str(fit[\'template_id\']),pred=fit[\'pred\'],features=feats)\n\ndef build_recovery_candidates(cid,y,accepted,rec_matcher,rec_bank_by_id,rcfg,v6_method,v13,cfg):\n    props=recovery_proposals(y,accepted,rec_matcher,cfg);out=[]\n    for k,p in enumerate(props):\n        c=candidate_record(y,p,rec_bank_by_id,rcfg,v6_method,v13)\n        if c is None:continue\n        c[\'candidate_key\']=f\'{cid}__rec{k}_{c["start"]}_{c["end"]}\';out.append(c)\n    return out\n\ndef apply_gate(v6_pcm,candidates,gate,threshold):\n    if not candidates:return np.asarray(v6_pcm,np.float32),[]\n    X=[[c[\'features\'][k] for k in CANDIDATE_FEATURES] for c in candidates];probs=gate.predict_proba(X)\n    ranked=sorted([(float(p),c) for p,c in zip(probs,candidates) if float(p)>=float(threshold)],key=lambda z:z[0],reverse=True)\n    chosen=[]\n    for p,c in ranked:\n        iv=(c[\'start\'],c[\'end\'])\n        if any(interval_iou(iv,(q[\'start\'],q[\'end\']))>0.05 for q in chosen):continue\n        chosen.append(c)\n    z=np.asarray(v6_pcm,np.float64).copy()\n    for c in chosen:z[c[\'start\']:c[\'end\']]-=np.asarray(c[\'pred\'],np.float64)\n    return pcm_roundtrip(z).astype(np.float32),chosen\n'
MAIN_SOURCE='#!/usr/bin/env python3\nfrom __future__ import annotations\nimport argparse, json, os, pickle\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom speechie_engine import (\n    VERSION, MatcherConfig, ResidualMatcher, ReconConfig,\n    read_wave_strict, write_pcm, wav_audit, wave_sha,\n    prepared_matches, reconstruct, hyp_text,\n)\nfrom v23_runtime import V13BankRuntime, LogisticGateRuntime, build_recovery_candidates, apply_gate\n\nASR_REPO = \'ARTPARK-IISc/SraVaani-1.0\'\nASR_BATCH_SIZE = 8\n\n\ndef load_assets(root: Path):\n    a=root/\'assets\'\n    needed=[\'frozen_bank.pkl\',\'frozen_policy.json\',\'recovery_bank.pkl\',\'v13_bank.pkl\',\'gate_params.json\',\'v23_config.json\',\'sravaani_revision_pin.json\']\n    for name in needed:\n        p=a/name\n        if not p.is_file():raise FileNotFoundError(f\'Required packaged asset missing: {p}\')\n    with open(a/\'frozen_bank.pkl\',\'rb\') as f:v5=pickle.load(f)\n    with open(a/\'frozen_policy.json\',encoding=\'utf-8\') as f:frozen=json.load(f)\n    with open(a/\'recovery_bank.pkl\',\'rb\') as f:rec_bank=pickle.load(f)\n    with open(a/\'v13_bank.pkl\',\'rb\') as f:v13_obj=pickle.load(f)\n    with open(a/\'gate_params.json\',encoding=\'utf-8\') as f:gate_obj=json.load(f)\n    with open(a/\'v23_config.json\',encoding=\'utf-8\') as f:cfg=json.load(f)\n    with open(a/\'sravaani_revision_pin.json\',encoding=\'utf-8\') as f:pin=json.load(f)\n    if v5.get(\'version\')!=VERSION:raise ValueError(f\'Unexpected V5 bank version: {v5.get("version")}\')\n    mcfg=MatcherConfig(**v5[\'cfg\']);v6_matcher=ResidualMatcher(v5,mcfg)\n    rcfg=ReconConfig(**frozen[\'recon_config\']);policy={\'method\':str(frozen[\'method\']),\'threshold\':float(frozen[\'threshold\'])}\n    rec_cfg=MatcherConfig(**rec_bank[\'cfg\']);rec_matcher=ResidualMatcher(rec_bank,rec_cfg)\n    v13=V13BankRuntime(v13_obj);gate=LogisticGateRuntime(gate_obj)\n    rev=pin.get(\'revision\')\n    if pin.get(\'repo\') not in (None,ASR_REPO):raise ValueError(\'SraVaani repo pin mismatch\')\n    if not isinstance(rev,str) or len(rev)!=40:raise ValueError(\'Expected immutable 40-character SraVaani revision\')\n    return v6_matcher,v5,rcfg,policy,rec_matcher,rec_bank,v13,gate,cfg,rev\n\n\ndef enumerate_wavs(root:Path):\n    wavs=sorted(p for p in root.rglob(\'*\') if p.is_file() and p.suffix.lower()==\'.wav\')\n    if not wavs:raise FileNotFoundError(f\'No WAV files found under {root}\')\n    seen={}\n    for p in wavs:\n        if p.name in seen:raise ValueError(f\'Duplicate WAV basename {p.name!r}; organizer requires same output filenames\')\n        seen[p.name]=p\n    return wavs\n\n\ndef enhance_all(input_dir,output_dir,v6_matcher,v5,rcfg,policy,rec_matcher,rec_bank,v13,gate,cfg):\n    v5_by_id={str(r[\'tid\']):r for r in v5[\'records\']};rec_by_id={str(r[\'tid\']):r for r in rec_bank[\'records\']}\n    output_dir.mkdir(parents=True,exist_ok=True);out={};wavs=enumerate_wavs(input_dir)\n    for i,src in enumerate(wavs,1):\n        cid=src.stem;y=read_wave_strict(src,sr=rcfg.sample_rate)\n        if wave_sha(y) in v6_matcher.source_hashes:\n            z=y.copy();chosen=[]\n            print(f\'WARNING: {src.name} matches a frozen V5 BANK source; identity pass-through used to avoid self-retrieval.\',flush=True)\n        else:\n            props=v6_matcher.scan(y,exclude_group=None)\n            prep=prepared_matches(y,props,v5_by_id,rcfg)\n            z_v6,info=reconstruct(y,prep,policy,rcfg)\n            accepted=info[\'matches\']\n            cands=build_recovery_candidates(cid,y,accepted,rec_matcher,rec_by_id,rcfg,policy[\'method\'],v13,cfg)\n            z,chosen=apply_gate(z_v6,cands,gate,float(cfg[\'threshold\']))\n        dst=output_dir/src.name;write_pcm(dst,z,sr=rcfg.sample_rate);wav_audit(dst,len(y),sr=rcfg.sample_rate);out[cid]=dst\n        if i==1 or i%100==0 or i==len(wavs):print(f\'Enhanced {i}/{len(wavs)} | V23 recovered events this clip: {len(chosen)}\',flush=True)\n    return out\n\n\ndef load_sravaani(revision,device):\n    if int(np.__version__.split(\'.\')[0])>=2:raise RuntimeError(\'Expected NumPy 1.26.x; install requirements.txt in a fresh Python 3.11/3.12 environment.\')\n    try:from transformers import AutoModel\n    except Exception as e:raise RuntimeError(\'SraVaani dependency import failed; install the pinned requirements.txt in a fresh environment.\') from e\n    token=os.environ.get(\'HF_TOKEN\')\n    return AutoModel.from_pretrained(ASR_REPO,revision=revision,code_revision=revision,trust_remote_code=True,token=token).to(device).eval()\n\n\ndef transcribe_all(model,paths,batch_size=ASR_BATCH_SIZE):\n    ids=sorted(paths);out={}\n    for a in range(0,len(ids),batch_size):\n        ch=ids[a:a+batch_size];ps=[str(paths[c]) for c in ch]\n        try:\n            hs=model.transcribe(ps,return_hypotheses=True)\n            if not (isinstance(hs,(list,tuple)) and len(hs)==len(ps)):\n                if len(ps)==1:hs=[hs]\n                else:raise RuntimeError(\'ASR batch cardinality mismatch\')\n        except Exception:\n            hs=[model.transcribe([p],return_hypotheses=True) for p in ps]\n        for cid,h in zip(ch,hs):out[cid]=hyp_text(h)\n        print(f\'Transcribed {min(a+batch_size,len(ids))}/{len(ids)}\',flush=True)\n    return out\n\n\ndef write_transcripts(output_dir,texts):\n    p=output_dir/\'transcripts.jsonl\'\n    with open(p,\'w\',encoding=\'utf-8\') as f:\n        for cid in sorted(texts):f.write(json.dumps({\'clip_id\':cid,\'text\':texts[cid]},ensure_ascii=False,allow_nan=False)+\'\\n\')\n    return p\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'speechie Track-2 V23 recovery inference + SraVaani ASR\')\n    ap.add_argument(\'input_folder\',type=Path);ap.add_argument(\'output_folder\',type=Path);args=ap.parse_args()\n    if not args.input_folder.is_dir():raise NotADirectoryError(args.input_folder)\n    args.output_folder.mkdir(parents=True,exist_ok=True)\n    root=Path(__file__).resolve().parent\n    assets=load_assets(root)\n    v6_matcher,v5,rcfg,policy,rec_matcher,rec_bank,v13,gate,cfg,revision=assets\n    print(\'V23 run key:\',cfg.get(\'run_key\'));print(\'Variant:\',cfg.get(\'variant\'),\'threshold=\',cfg.get(\'threshold\'));print(\'Device:\',v6_matcher.device)\n    enhanced=enhance_all(args.input_folder,args.output_folder,v6_matcher,v5,rcfg,policy,rec_matcher,rec_bank,v13,gate,cfg)\n    v6_matcher.cache.clear();rec_matcher.cache.clear()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    model=load_sravaani(revision,v6_matcher.device);texts=transcribe_all(model,enhanced);p=write_transcripts(args.output_folder,texts)\n    print(f\'Done. Enhanced WAVs: {len(enhanced)}\');print(f\'Transcript: {p}\')\n\nif __name__==\'__main__\':main()\n'
REQUIREMENTS='numpy==1.26.4\nscipy==1.11.4\nscikit-learn==1.4.2\nsoundfile==0.12.1\ntorch>=2.6\ntransformers>=4.48,<5\nhuggingface_hub>=0.28,<2\naccelerate>=1.0,<2\n'

(TEAM_DIR/'speechie_engine.py').write_text(ENGINE_SOURCE,encoding='utf-8')
(TEAM_DIR/'v23_runtime.py').write_text(V23_RUNTIME_SOURCE,encoding='utf-8')
(TEAM_DIR/'main.py').write_text(MAIN_SOURCE,encoding='utf-8')
(TEAM_DIR/'requirements.txt').write_text(REQUIREMENTS,encoding='utf-8')

# Copy exact V6 frozen assets and ASR revision pin.
shutil.copy2(V5_DIR/'frozen_bank.pkl',ASSET_DIR/'frozen_bank.pkl')
shutil.copy2(V6_RUN/'frozen_before_hold.json',ASSET_DIR/'frozen_policy.json')
shutil.copy2(V6_ROOT/'sravaani_revision_pin.json',ASSET_DIR/'sravaani_revision_pin.json')

# Freeze V23 targeted recovery bank exactly as used by the scored run.
with open(ASSET_DIR/'recovery_bank.pkl','wb') as f:
    pickle.dump(REC_BANK,f,protocol=pickle.HIGHEST_PROTOCOL)

# Freeze the V13 confirmation bank without public validation files: only residual event waves + precomputed features.
print('Serializing V13 residual-event bank ...')
v13_waves=[np.asarray(bank_event_wave(i),np.float32) for i in range(len(BANK_EVENTS))]
v13_obj=dict(
    waves=v13_waves,
    time=np.asarray(BANK_TIME,np.float32),
    pre=np.asarray(BANK_PRE,np.float32),
    spec=np.asarray(BANK_SPEC,np.float32),
    env=np.asarray(BANK_ENV,np.float32),
    dur=np.asarray(BANK_DUR,np.float64),
    cheap_recipe=str(CHEAP_RECIPE),
    shortlist_k=int(SHORTLIST_K),
    fine_recipe=str(FINE_RECIPE),
)
with open(ASSET_DIR/'v13_bank.pkl','wb') as f:
    pickle.dump(v13_obj,f,protocol=pickle.HIGHEST_PROTOCOL)

# Export the fitted LogisticRegression pipeline numerically, avoiding sklearn-pickle version fragility.
steps=list(FINAL_GATE.named_steps.values())
imp=next((x for x in steps if hasattr(x,'statistics_')),None)
scaler=next((x for x in steps if hasattr(x,'mean_') and hasattr(x,'scale_')),None)
lr=next((x for x in steps if hasattr(x,'coef_') and hasattr(x,'intercept_')),None)
if imp is None or scaler is None or lr is None:
    raise RuntimeError('Could not extract fitted logistic gate pipeline.')
classes=np.asarray(lr.classes_)
if len(classes)!=2 or not bool(classes[-1]):
    raise RuntimeError(f'Unexpected logistic classes: {classes}')
gate_obj=dict(
    imputer_statistics=np.asarray(imp.statistics_,float).tolist(),
    scaler_mean=np.asarray(scaler.mean_,float).tolist(),
    scaler_scale=np.asarray(scaler.scale_,float).tolist(),
    coef=np.asarray(lr.coef_,float).reshape(-1).tolist(),
    intercept=float(np.asarray(lr.intercept_,float).reshape(-1)[0]),
    classes=[str(x) for x in classes.tolist()],
)
with open(ASSET_DIR/'gate_params.json','w',encoding='utf-8') as f:
    json.dump(gate_obj,f,indent=2,allow_nan=False)

# Verify exported gate numerically against sklearn on the SELECT candidates.
if 'SELECT_CAND' in globals() and len(SELECT_CAND):
    X=SELECT_CAND[CANDIDATE_FEATURES].to_numpy(float)
    X2=np.where(np.isfinite(X),X,np.asarray(gate_obj['imputer_statistics'])[None,:])
    X2=(X2-np.asarray(gate_obj['scaler_mean'])[None,:])/np.asarray(gate_obj['scaler_scale'])[None,:]
    p2=1/(1+np.exp(-np.clip(X2@np.asarray(gate_obj['coef'])+gate_obj['intercept'],-60,60)))
    p1=FINAL_GATE.predict_proba(SELECT_CAND[CANDIDATE_FEATURES])[:,1]
    md=float(np.max(np.abs(p1-p2)))
    print('Gate export max |delta p|:',md)
    if md>1e-10: raise RuntimeError('Exported logistic gate does not reproduce sklearn probabilities.')

variant='BALANCED'
threshold=float(BALANCED_THRESHOLD)
if not np.isfinite(threshold): raise RuntimeError('BALANCED threshold is not finite.')
config=dict(
    version='v23_high_recall_missed_event_20260920',
    run_key=str(RUN_KEY),variant=variant,threshold=threshold,parent_v6_run='23523d3d7e5ed3ea',
    min_verify_quality=float(RECOVERY_MIN_VERIFY_QUALITY),
    max_candidates_per_clip=int(MAX_RECOVERY_CANDIDATES_PER_CLIP),
    nms_iou=float(RECOVERY_NMS_IOU),
    accepted_exclusion_margin_ms=float(ACCEPTED_EXCLUSION_MARGIN_MS),
    candidate_features=list(CANDIDATE_FEATURES),
    best_gate_model=str(BEST_GATE),
    no_hidden_clean_gt_or_domain_labels=True,
)
with open(ASSET_DIR/'v23_config.json','w',encoding='utf-8') as f:
    json.dump(config,f,indent=2,allow_nan=False)

# Include the scored run's decision report for audit transparency.
decision_path=Path(OUT_ROOT)/('run_'+str(RUN_KEY))/'decision_report.json'
if decision_path.is_file(): shutil.copy2(decision_path,ASSET_DIR/'decision_report.json')

README=f


Primary contact mobile (include country code): 9051045490
Primary contact email: raianand.1991@gmail.com
Serializing V13 residual-event bank ...
Gate export max |delta p|: 0.0


In [26]:
import shutil
from pathlib import Path

# Define source and destination paths
source_folder = Path('/content/speechie_v23_code_submission_build/speechie')
drive_destination_folder = Path('/content/drive/MyDrive/Datathon')

# Define the base name for the archive (the name of the zip file without extension)
archive_name = 'speechie'

# Create the zip archive in a temporary location (e.g., /content)
output_archive_path = shutil.make_archive(
    str(Path('/content') / archive_name),
    'zip',
    root_dir=source_folder.parent,
    base_dir=source_folder.name
)

# Move the created zip file to Google Drive
shutil.move(output_archive_path, drive_destination_folder / f'{archive_name}.zip')

print(f'Successfully created and uploaded {archive_name}.zip to {drive_destination_folder}')

Successfully created and uploaded speechie.zip to /content/drive/MyDrive/Datathon


In [27]:
# === Smoke-test speechie V23 code submission from current runtime ===

from pathlib import Path
import shutil, subprocess, sys, os, json, zipfile

# 1) EDIT THESE TWO PATHS ONLY
CODE_DIR = Path("/content/speechie_v23_code_submission_build/speechie")
INPUT_DIR = Path("/content/drive/MyDrive/PHD/Datathon")   # folder containing original input WAVs

# Temporary test folders
SMOKE_INPUT = Path("/content/speechie_smoke_input")
SMOKE_OUTPUT = Path("/content/speechie_smoke_output")

# Clean previous run
shutil.rmtree(SMOKE_INPUT, ignore_errors=True)
shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)
SMOKE_INPUT.mkdir(parents=True)
SMOKE_OUTPUT.mkdir(parents=True)

# Basic package checks
required = [
    CODE_DIR / "main.py",
    CODE_DIR / "requirements.txt",
    CODE_DIR / "speechie_engine.py",
    CODE_DIR / "v23_runtime.py",
    CODE_DIR / "assets",
]
for p in required:
    assert p.exists(), f"Missing: {p}"

print("Code package:", CODE_DIR)

# Pick first 5 WAVs
wavs = sorted(INPUT_DIR.rglob("*.wav"))
assert wavs, f"No WAV files found in {INPUT_DIR}"

N = min(5, len(wavs))
for p in wavs[:N]:
    shutil.copy2(p, SMOKE_INPUT / p.name)

print(f"Testing on {N} WAVs:")
for p in sorted(SMOKE_INPUT.glob("*.wav")):
    print(" ", p.name)

# Run exactly the way the organizers will
cmd = [
    sys.executable,
    str(CODE_DIR / "main.py"),
    str(SMOKE_INPUT),
    str(SMOKE_OUTPUT),
]

print("\nRUNNING:")
print(" ".join(cmd))
print("=" * 80)

result = subprocess.run(
    cmd,
    cwd=str(CODE_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(result.stdout)

print("=" * 80)
print("Return code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Inference FAILED. See log above.")

# Output checks
out_wavs = sorted(SMOKE_OUTPUT.glob("*.wav"))
print("\nOutput WAVs:", len(out_wavs))

expected_names = {p.name for p in SMOKE_INPUT.glob("*.wav")}
actual_names = {p.name for p in out_wavs}

assert expected_names == actual_names, (
    f"WAV filename mismatch.\n"
    f"Missing: {expected_names-actual_names}\n"
    f"Extra: {actual_names-expected_names}"
)

transcript_file = SMOKE_OUTPUT / "transcripts.jsonl"
assert transcript_file.is_file(), "transcripts.jsonl missing"

rows = [
    json.loads(x)
    for x in transcript_file.read_text(encoding="utf-8").splitlines()
    if x.strip()
]

assert len(rows) == N, f"Expected {N} transcripts, found {len(rows)}"

print("\nTranscripts:")
for r in rows:
    print(r)

print("\n" + "=" * 80)
print("SMOKE TEST PASSED")
print("main.py executed successfully with exactly two positional arguments.")
print(f"Generated {len(out_wavs)}/{N} WAVs + transcripts.jsonl")
print("=" * 80)

Code package: /content/speechie_v23_code_submission_build/speechie
Testing on 5 WAVs:
  IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213769_001_GENERIC_0098_17486_21246.wav
  IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213849_018_GENERIC_0687_5816_11600.wav
  IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213859_058_LowerDibangvalley-SPECIFIC_01676_290_4896.wav
  IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030214258_039_LowerDibangvalley-SPECIFIC_00163_5058_7966.wav
  IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030214258_042_LowerDibangvalley-SPECIFIC_00049_15330_18270.wav

RUNNING:
/usr/bin/python3 /content/speechie_v23_code_submission_build/speechie/main.py /content/speechie_smoke_input /content/speechie_smoke_output
V23 run key: a8dfcafd81bd5717
Variant: BALANCED threshold= 0.11977777777777779
Device: cpu
Enhan

RuntimeError: Inference FAILED. See log above.

In [29]:
# ============================================================
# PATCH NumPy guard + verify SraVaani environment
# ============================================================

from pathlib import Path
import shutil
import numpy as np

# Auto-find the current submission folder
# matches = list(Path("/content").rglob("speechie/main.py"))

# assert matches, "Could not find speechie/main.py under /content"

# for i, p in enumerate(matches):
#     print(i, p)

# Prefer the current build path seen in your traceback
preferred = Path(
    "/content/speechie_v23_code_submission_build/speechie/main.py"
)

MAIN = preferred if preferred.exists() else matches[0]
CODE_DIR = MAIN.parent

print("\nUsing:", CODE_DIR)
print("Current NumPy:", np.__version__)

# Backup
backup = MAIN.with_suffix(".py.before_numpy_patch")
if not backup.exists():
    shutil.copy2(MAIN, backup)
    print("Backup:", backup)

src = MAIN.read_text(encoding="utf-8")

old = (
    "if int(np.__version__.split('.')[0])>=2:"
    "raise RuntimeError('Expected NumPy 1.26.x; install requirements.txt "
    "in a fresh Python 3.11/3.12 environment.')"
)

old_spaced = (
    "if int(np.__version__.split('.')[0]) >= 2:"
    " raise RuntimeError('Expected NumPy 1.26.x; install requirements.txt "
    "in a fresh Python 3.11/3.12 environment.')"
)

if old in src:
    src = src.replace(
        old,
        "# NumPy compatibility is validated by importing the actual ASR stack."
    )
elif old_spaced in src:
    src = src.replace(
        old_spaced,
        "# NumPy compatibility is validated by importing the actual ASR stack."
    )
else:
    # More robust line-based fallback
    lines = src.splitlines()
    fixed = []
    removed = False

    for line in lines:
        if (
            "np.__version__.split" in line
            and "Expected NumPy 1.26" in line
        ):
            indent = line[:len(line)-len(line.lstrip())]
            fixed.append(
                indent +
                "# NumPy compatibility validated by actual dependency imports."
            )
            removed = True
        else:
            fixed.append(line)

    src = "\n".join(fixed) + "\n"

    if not removed:
        print("WARNING: explicit NumPy guard was not found.")

MAIN.write_text(src, encoding="utf-8")

print("\nPatched:", MAIN)

# ------------------------------------------------------------
# Test the actual dependency stack
# ------------------------------------------------------------
print("\n=== ENVIRONMENT CHECK ===")

import sys
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)

try:
    import scipy
    print("SciPy       :", scipy.__version__)
except Exception as e:
    print("SciPy import FAILED:", repr(e))

try:
    import sklearn
    print("scikit-learn:", sklearn.__version__)
except Exception as e:
    print("sklearn import FAILED:", repr(e))

try:
    import torch
    print("Torch       :", torch.__version__)
except Exception as e:
    print("Torch import FAILED:", repr(e))

try:
    import transformers
    print("Transformers:", transformers.__version__)

    from transformers import AutoModel
    print("AutoModel import: PASS")

except Exception as e:
    print("\nAutoModel import FAILED")
    raise

print("\n" + "="*70)
print("PATCH + ASR IMPORT CHECK PASSED")
print("="*70)


Using: /content/speechie_v23_code_submission_build/speechie
Current NumPy: 2.1.3
Backup: /content/speechie_v23_code_submission_build/speechie/main.py.before_numpy_patch

Patched: /content/speechie_v23_code_submission_build/speechie/main.py

=== ENVIRONMENT CHECK ===
Python      : 3.13.15
NumPy       : 2.1.3
SciPy       : 1.16.3
scikit-learn: 1.6.1
Torch       : 2.11.0+cpu
Transformers: 5.16.1
AutoModel import: PASS

PATCH + ASR IMPORT CHECK PASSED


In [30]:
# ============================================================
# V23 COMPLETE 5-FILE END-TO-END SMOKE TEST
# ============================================================

from pathlib import Path
import subprocess, sys, shutil, os, json, time

CODE_DIR = Path(
    "/content/speechie_v23_code_submission_build/speechie"
)

SMOKE_INPUT = Path("/content/speechie_smoke_input")
SMOKE_OUTPUT = Path("/content/speechie_smoke_output")

assert (CODE_DIR / "main.py").is_file()
assert SMOKE_INPUT.is_dir()

inputs = sorted(SMOKE_INPUT.glob("*.wav"))
assert inputs, "No smoke-test WAVs found."

print("Input files:", len(inputs))
print("Code:", CODE_DIR / "main.py")
print("Python:", sys.executable)

shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)
SMOKE_OUTPUT.mkdir(parents=True)

cmd = [
    sys.executable,
    str(CODE_DIR / "main.py"),
    str(SMOKE_INPUT),
    str(SMOKE_OUTPUT),
]

print("\nCOMMAND")
print(" ".join(map(str, cmd)))
print("=" * 90)

t0 = time.time()

r = subprocess.run(
    cmd,
    cwd=str(CODE_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

elapsed = time.time() - t0

print(r.stdout)
print("=" * 90)
print("Return code:", r.returncode)
print(f"Elapsed: {elapsed:.1f} sec")

if r.returncode != 0:
    raise RuntimeError("END-TO-END SMOKE TEST FAILED")

# ------------------------------------------------------------
# Verify generated WAVs
# ------------------------------------------------------------
expected = {p.name for p in inputs}
generated = {p.name for p in SMOKE_OUTPUT.glob("*.wav")}

assert generated == expected, (
    f"WAV mismatch\n"
    f"Missing: {sorted(expected-generated)}\n"
    f"Extra: {sorted(generated-expected)}"
)

# ------------------------------------------------------------
# Verify transcripts
# ------------------------------------------------------------
tp = SMOKE_OUTPUT / "transcripts.jsonl"
assert tp.is_file(), "transcripts.jsonl missing"

rows = [
    json.loads(line)
    for line in tp.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(rows) == len(inputs), (
    f"Expected {len(inputs)} transcript rows, got {len(rows)}"
)

ids = {r["clip_id"] for r in rows}

assert ids == {p.stem for p in inputs}, (
    "Transcript clip IDs do not match input filenames."
)

assert all(isinstance(r.get("text"), str) for r in rows)

print("\nTRANSCRIPTS")
print("-" * 90)

for r0 in rows:
    print(r0["clip_id"], "=>", r0["text"][:200])

print("\n" + "=" * 90)
print("END-TO-END SMOKE TEST PASSED")
print(f"WAVs       : {len(generated)}/{len(inputs)}")
print(f"Transcripts: {len(rows)}/{len(inputs)}")
print(f"Runtime    : {elapsed:.1f} sec")
print("=" * 90)

Input files: 5
Code: /content/speechie_v23_code_submission_build/speechie/main.py
Python: /usr/bin/python3

COMMAND
/usr/bin/python3 /content/speechie_v23_code_submission_build/speechie/main.py /content/speechie_smoke_input /content/speechie_smoke_output
V23 run key: a8dfcafd81bd5717
Variant: BALANCED threshold= 0.11977777777777779
Device: cpu
Enhanced 1/5 | V23 recovered events this clip: 0
Enhanced 5/5 | V23 recovered events this clip: 0

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1008.62it/s]
Transcribed 5/5
Done. Enhanced WAVs: 5
Transcript: /content/speechie_smoke_output/transcripts.jsonl

Return code: 0
Elapsed: 98.1 sec

TRANSCRIPTS
------------------------------------------------------------------------------------------
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213769_001_GENERIC_0098_17486_21246 => ফটো খনত এজন সৰু দুজন সৰু লৰা ছোৱালী আ লৰা আছে
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley8444250302138

In [31]:
# ============================================================
# Compare 5 smoke-test outputs against scored V23 leaderboard ZIP
# ============================================================

from pathlib import Path
import zipfile, hashlib, io
import soundfile as sf
import numpy as np

SMOKE_OUTPUT = Path("/content/speechie_smoke_output")

# EDIT if needed
SCORED_ZIP = Path(
    "/content/drive/MyDrive/Datathon/"
    "track2_v23_high_recall_missed_event_recovery/"
    "run_a8dfcafd81bd5717/"
    "submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip"
)

assert SCORED_ZIP.is_file(), f"Scored ZIP not found: {SCORED_ZIP}"

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def pcm_sha_from_bytes(b):
    x, sr = sf.read(io.BytesIO(b), dtype="int16")
    return sr, x.shape, hashlib.sha256(np.asarray(x).tobytes()).hexdigest()

rows = []

with zipfile.ZipFile(SCORED_ZIP) as z:
    names = set(z.namelist())

    for p in sorted(SMOKE_OUTPUT.glob("*.wav")):
        zip_name = p.name

        assert zip_name in names, f"{zip_name} missing from scored ZIP"

        local_bytes = p.read_bytes()
        scored_bytes = z.read(zip_name)

        byte_exact = sha256_bytes(local_bytes) == sha256_bytes(scored_bytes)

        sr1, sh1, pcm1 = pcm_sha_from_bytes(local_bytes)
        sr2, sh2, pcm2 = pcm_sha_from_bytes(scored_bytes)

        pcm_exact = (sr1 == sr2 and sh1 == sh2 and pcm1 == pcm2)

        rows.append((p.name, byte_exact, pcm_exact))

print("=" * 90)
print("V23 SCORED OUTPUT REPRODUCIBILITY CHECK")
print("=" * 90)

for name, byte_exact, pcm_exact in rows:
    print(
        f"{name}\n"
        f"  file bytes exact : {byte_exact}\n"
        f"  PCM audio exact  : {pcm_exact}"
    )

n = len(rows)
byte_n = sum(int(r[1]) for r in rows)
pcm_n = sum(int(r[2]) for r in rows)

print("\n" + "=" * 90)
print(f"Byte-exact files : {byte_n}/{n}")
print(f"PCM-exact audio  : {pcm_n}/{n}")

if pcm_n == n:
    print("\nREPRODUCIBILITY CHECK: PASS")
else:
    print("\nREPRODUCIBILITY CHECK: FAIL — investigate differences")

print("=" * 90)

V23 SCORED OUTPUT REPRODUCIBILITY CHECK
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213769_001_GENERIC_0098_17486_21246.wav
  file bytes exact : True
  PCM audio exact  : True
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213849_018_GENERIC_0687_5816_11600.wav
  file bytes exact : True
  PCM audio exact  : True
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213859_058_LowerDibangvalley-SPECIFIC_01676_290_4896.wav
  file bytes exact : True
  PCM audio exact  : True
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030214258_039_LowerDibangvalley-SPECIFIC_00163_5058_7966.wav
  file bytes exact : True
  PCM audio exact  : True
IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030214258_042_LowerDibangvalley-SPECIFIC_00049_15330_18270.wav
  file bytes exact : True
  PCM audio exact  : True

Byte-exact files : 5/5
PCM-exact audio  

In [33]:
# ============================================================
# FINALIZE speechie Track-2 audit/reproducibility ZIP
# Uses the EXACT environment that just reproduced scored V23 WAVs
# ============================================================

from pathlib import Path
import importlib.metadata as md
import shutil
import zipfile
import hashlib
import os
import py_compile

TEAM_NAME = "speechie"

CODE_DIR = Path(
    "/content/speechie_v23_code_submission_build/speechie"
)

FINAL_ZIP = Path(
    "/content/drive/MyDrive/Datathon/speechie.zip"
)

assert CODE_DIR.is_dir(), f"Missing package folder: {CODE_DIR}"
assert (CODE_DIR / "main.py").is_file()

# ------------------------------------------------------------
# 1. Capture versions from the environment that actually passed
# ------------------------------------------------------------

packages = [
    "numpy",
    "scipy",
    "scikit-learn",
    "soundfile",
    "torch",
    "transformers",
    "huggingface-hub",
    "accelerate",
]

versions = {}

for pkg in packages:
    try:
        versions[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        print(f"WARNING: {pkg} not installed")

print("WORKING ENVIRONMENT")
print("=" * 70)
for k, v in versions.items():
    print(f"{k:18s} {v}")

required = {
    "numpy",
    "scipy",
    "scikit-learn",
    "soundfile",
    "torch",
    "transformers",
    "huggingface-hub",
    "accelerate",
}

missing = required - set(versions)
assert not missing, f"Required packages missing: {sorted(missing)}"

# ------------------------------------------------------------
# 2. Write exact tested requirements
# ------------------------------------------------------------

requirements = "\n".join([
    f"numpy=={versions['numpy']}",
    f"scipy=={versions['scipy']}",
    f"scikit-learn=={versions['scikit-learn']}",
    f"soundfile=={versions['soundfile']}",
    f"torch=={versions['torch']}",
    f"transformers=={versions['transformers']}",
    f"huggingface_hub=={versions['huggingface-hub']}",
    f"accelerate=={versions['accelerate']}",
]) + "\n"

REQ = CODE_DIR / "requirements.txt"
REQ.write_text(requirements, encoding="utf-8")

print("\nFINAL requirements.txt")
print("=" * 70)
print(requirements)

# ------------------------------------------------------------
# 3. Remove obsolete NumPy-1.x runtime rejection if still present
# ------------------------------------------------------------

MAIN = CODE_DIR / "main.py"
src = MAIN.read_text(encoding="utf-8")

bad_lines = [
    line for line in src.splitlines()
    if "Expected NumPy 1.26" in line
    or (
        "np.__version__.split" in line
        and "RuntimeError" in line
    )
]

if bad_lines:
    print("Removing obsolete NumPy guard:")
    for x in bad_lines:
        print(" ", x)

    src = "\n".join(
        line for line in src.splitlines()
        if line not in bad_lines
    ) + "\n"

    MAIN.write_text(src, encoding="utf-8")

assert "Expected NumPy 1.26" not in MAIN.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 4. Syntax validation
# ------------------------------------------------------------

for p in CODE_DIR.rglob("*.py"):
    py_compile.compile(str(p), doraise=True)

print("Python syntax check: PASS")

# ------------------------------------------------------------
# 5. Required organizer files
# ------------------------------------------------------------

required_paths = [
    CODE_DIR / "main.py",
    CODE_DIR / "requirements.txt",
    CODE_DIR / "README.md",
    CODE_DIR / "speechie_engine.py",
    CODE_DIR / "v23_runtime.py",
    CODE_DIR / "assets",
]

for p in required_paths:
    assert p.exists(), f"Missing required package item: {p}"

# README must contain contact details
readme = (CODE_DIR / "README.md").read_text(
    encoding="utf-8",
    errors="ignore"
)

assert "__PRIMARY_MOBILE__" not in readme, \
    "README still has placeholder mobile number"

assert "__PRIMARY_EMAIL__" not in readme, \
    "README still has placeholder email"

print("Required-file check: PASS")

# ------------------------------------------------------------
# 6. Remove junk/cache files
# ------------------------------------------------------------

for p in list(CODE_DIR.rglob("__pycache__")):
    shutil.rmtree(p, ignore_errors=True)

for p in CODE_DIR.rglob("*.pyc"):
    p.unlink(missing_ok=True)

# Do NOT package our temporary backup
for p in CODE_DIR.glob("*.before_numpy_patch"):
    p.unlink(missing_ok=True)

# ------------------------------------------------------------
# 7. Rebuild SHA256SUMS.txt
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

SUMFILE = CODE_DIR / "SHA256SUMS.txt"

files_for_hash = sorted(
    p for p in CODE_DIR.rglob("*")
    if p.is_file()
    and p != SUMFILE
)

with open(SUMFILE, "w", encoding="utf-8") as f:
    for p in files_for_hash:
        rel = p.relative_to(CODE_DIR)
        f.write(f"{sha256_file(p)}  {rel.as_posix()}\n")

print("SHA256 manifest: rebuilt")

# ------------------------------------------------------------
# 8. Build exact organizer structure:
#
# speechie.zip
#   speechie/
#       main.py
#       ...
# ------------------------------------------------------------

FINAL_ZIP.parent.mkdir(parents=True, exist_ok=True)

tmp_zip = FINAL_ZIP.with_suffix(".zip.tmp")

if tmp_zip.exists():
    tmp_zip.unlink()

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    tmp_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as z:
    for p in sorted(CODE_DIR.rglob("*")):
        if p.is_file():
            archive_name = (
                Path(TEAM_NAME) /
                p.relative_to(CODE_DIR)
            )
            z.write(p, archive_name.as_posix())

# ------------------------------------------------------------
# 9. Audit ZIP structure
# ------------------------------------------------------------

with zipfile.ZipFile(tmp_zip) as z:
    assert z.testzip() is None, "ZIP CRC failure"

    names = z.namelist()

    assert names, "ZIP is empty"
    assert all(
        n.startswith(f"{TEAM_NAME}/")
        for n in names
    ), "ZIP contains files outside top-level speechie/"

    must_exist = {
        f"{TEAM_NAME}/main.py",
        f"{TEAM_NAME}/requirements.txt",
        f"{TEAM_NAME}/README.md",
        f"{TEAM_NAME}/speechie_engine.py",
        f"{TEAM_NAME}/v23_runtime.py",
    }

    missing = must_exist - set(names)
    assert not missing, f"ZIP missing: {missing}"

os.replace(tmp_zip, FINAL_ZIP)

# ------------------------------------------------------------
# 10. Final checksum
# ------------------------------------------------------------

zip_sha = sha256_file(FINAL_ZIP)
size_mb = FINAL_ZIP.stat().st_size / 1024**2

print("\n" + "=" * 80)
print("FINAL TRACK-2 CODE SUBMISSION READY")
print("=" * 80)
print("ZIP     :", FINAL_ZIP)
print(f"Size    : {size_mb:.2f} MB")
print("SHA256  :", zip_sha)
print()
print("Top folder : speechie/")
print("main.py    : present")
print("requirements.txt : updated to TESTED working environment")
print("ZIP CRC    : PASS")
print("Structure  : PASS")
print("=" * 80)

WORKING ENVIRONMENT
numpy              2.1.3
scipy              1.16.3
scikit-learn       1.6.1
soundfile          0.14.0
torch              2.11.0+cpu
transformers       5.16.1
huggingface-hub    1.29.0
accelerate         1.14.0

FINAL requirements.txt
numpy==2.1.3
scipy==1.16.3
scikit-learn==1.6.1
soundfile==0.14.0
torch==2.11.0+cpu
transformers==5.16.1
huggingface_hub==1.29.0
accelerate==1.14.0

Python syntax check: PASS


AssertionError: Missing required package item: /content/speechie_v23_code_submission_build/speechie/README.md

In [34]:
from pathlib import Path

CODE_DIR = Path("/content/speechie_v23_code_submission_build/speechie")

PRIMARY_MOBILE = "+91-9051045490"   # <-- replace
PRIMARY_EMAIL = "raianand.1991@gmail.com"   # <-- replace

README = CODE_DIR / "README.md"

text = f"""# speechie — IndoML Track 2 Final Inference Submission

## Primary contact

Mobile: {PRIMARY_MOBILE}
Email: {PRIMARY_EMAIL}

## Team

Team name: speechie

## Method

This package contains the executable inference implementation used for the
Track 2 Phase-1 submission.

The system applies:

1. Frozen V6 residual-template reconstruction
2. V23 high-recall missed-event recovery
3. SraVaani ASR transcription using:
   ARTPARK-IISc/SraVaani-1.0

The SraVaani weights are not included in this ZIP. They are downloaded from
Hugging Face at runtime, as permitted by the competition instructions.

## Package contents

- main.py
- speechie_engine.py
- v23_runtime.py
- assets/
- requirements.txt
- README.md
- SHA256SUMS.txt

## Installation

Create a clean Python environment and run:

    python -m pip install -r requirements.txt

If Hugging Face authentication is required, set:

    export HF_TOKEN=<your_token>

Do not place authentication tokens inside this package.

## Running inference

The inference script takes exactly two positional arguments:

    python main.py <input_folder> <output_folder>

Example:

    python main.py /path/to/input_wavs /path/to/output_folder

## Input

The input folder must contain the Track 2 WAV files.

The system expects the competition audio format:
16 kHz mono WAV.

## Output

For every input WAV, the output folder contains an enhanced WAV with the same
filename.

The output folder also contains:

    transcripts.jsonl

Each line follows the Codabench-style format:

    {{"clip_id": "example_clip", "text": "recognized transcript"}}

## Reproducibility

Inference uses only:

- the supplied input WAV
- packaged frozen reconstruction/retrieval assets
- the packaged V23 candidate gate/configuration
- SraVaani ASR loaded at runtime

No hidden-test clean reference, ground-truth event window, hidden-test label,
reference transcript, or evaluation metric is used during hidden-test inference.

The packaged inference implementation was smoke-tested end-to-end against the
scored V23 submission. Corresponding regenerated WAV files were verified to be
byte-for-byte identical and PCM-sample identical for the tested audit sample.

## Entry point

Required command:

    python main.py input_folder output_folder
"""

README.write_text(text, encoding="utf-8")

assert README.exists()

print("README CREATED:")
print(README)
print("\n" + "="*70)
print(README.read_text(encoding="utf-8"))

README CREATED:
/content/speechie_v23_code_submission_build/speechie/README.md

# speechie — IndoML Track 2 Final Inference Submission

## Primary contact

Mobile: +91-9051045490
Email: raianand.1991@gmail.com

## Team

Team name: speechie

## Method

This package contains the executable inference implementation used for the
Track 2 Phase-1 submission.

The system applies:

1. Frozen V6 residual-template reconstruction
2. V23 high-recall missed-event recovery
3. SraVaani ASR transcription using:
   ARTPARK-IISc/SraVaani-1.0

The SraVaani weights are not included in this ZIP. They are downloaded from
Hugging Face at runtime, as permitted by the competition instructions.

## Package contents

- main.py
- speechie_engine.py
- v23_runtime.py
- assets/
- requirements.txt
- README.md
- SHA256SUMS.txt

## Installation

Create a clean Python environment and run:

    python -m pip install -r requirements.txt

If Hugging Face authentication is required, set:

    export HF_TOKEN=<your_token>

D

In [36]:
# ============================================================
# FINAL ORGANIZER ZIP CREATION
# Creates:
#   /content/drive/MyDrive/Datathon/speechie.zip
#
# ZIP structure:
#   speechie/
#       README.md
#       main.py
#       requirements.txt
#       speechie_engine.py
#       v23_runtime.py
#       assets/
#       SHA256SUMS.txt
# ============================================================

from pathlib import Path
import shutil
import zipfile
import hashlib
import os

TEAM_NAME = "speechie"

CODE_DIR = Path(
    "/content/speechie_v23_code_submission_build/speechie"
)

FINAL_ZIP = Path(
    "/content/drive/MyDrive/speechie.zip"
)

assert CODE_DIR.is_dir(), f"Missing folder: {CODE_DIR}"

# ------------------------------------------------------------
# 1. Remove files that must NOT go to organizers
# ------------------------------------------------------------

junk_files = [
    CODE_DIR / "main.py.before_numpy_patch",
]

for p in junk_files:
    if p.exists():
        p.unlink()
        print("Removed:", p)

for p in CODE_DIR.rglob("__pycache__"):
    shutil.rmtree(p, ignore_errors=True)

for p in CODE_DIR.rglob("*.pyc"):
    p.unlink(missing_ok=True)

# ------------------------------------------------------------
# 2. Required-file verification
# ------------------------------------------------------------

required = [
    "main.py",
    "README.md",
    "requirements.txt",
    "speechie_engine.py",
    "v23_runtime.py",
]

for name in required:
    p = CODE_DIR / name
    assert p.is_file(), f"Missing required file: {p}"

assert (CODE_DIR / "assets").is_dir(), "assets/ missing"

print("\nRequired files: PASS")

# ------------------------------------------------------------
# 3. Check README contact placeholders
# ------------------------------------------------------------

readme = (CODE_DIR / "README.md").read_text(
    encoding="utf-8",
    errors="ignore"
)

bad_placeholders = [
    "XXXXXXXXXX",
    "your-email@example.com",
    "__PRIMARY_MOBILE__",
    "__PRIMARY_EMAIL__",
]

found = [x for x in bad_placeholders if x in readme]

assert not found, (
    "README still contains placeholder contact details: "
    + ", ".join(found)
)

print("README contact check: PASS")

# ------------------------------------------------------------
# 4. Check obsolete NumPy guard is gone
# ------------------------------------------------------------

main_text = (CODE_DIR / "main.py").read_text(
    encoding="utf-8",
    errors="ignore"
)

assert "Expected NumPy 1.26" not in main_text, (
    "Old NumPy guard still exists in main.py"
)

print("NumPy guard check: PASS")

# ------------------------------------------------------------
# 5. Build SHA256 manifest
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

SUMFILE = CODE_DIR / "SHA256SUMS.txt"

if SUMFILE.exists():
    SUMFILE.unlink()

files_to_hash = sorted(
    p for p in CODE_DIR.rglob("*")
    if p.is_file()
)

with open(SUMFILE, "w", encoding="utf-8") as f:
    for p in files_to_hash:
        rel = p.relative_to(CODE_DIR).as_posix()
        f.write(f"{sha256_file(p)}  {rel}\n")

print("SHA256SUMS.txt created")

# ------------------------------------------------------------
# 6. Create final ZIP
# ------------------------------------------------------------

FINAL_ZIP.parent.mkdir(parents=True, exist_ok=True)

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

tmp_zip = FINAL_ZIP.with_suffix(".zip.tmp")

if tmp_zip.exists():
    tmp_zip.unlink()

with zipfile.ZipFile(
    tmp_zip,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as z:

    for p in sorted(CODE_DIR.rglob("*")):

        if not p.is_file():
            continue

        rel = p.relative_to(CODE_DIR)

        # Required organizer top-level folder
        arcname = Path(TEAM_NAME) / rel

        z.write(
            p,
            arcname.as_posix()
        )

# ------------------------------------------------------------
# 7. Validate ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(tmp_zip, "r") as z:

    names = z.namelist()

    assert z.testzip() is None, "ZIP CRC error"

    assert names, "ZIP is empty"

    # Everything must be under speechie/
    assert all(
        n.startswith("speechie/")
        for n in names
    ), "Something exists outside speechie/"

    required_zip_entries = {
        "speechie/main.py",
        "speechie/README.md",
        "speechie/requirements.txt",
        "speechie/speechie_engine.py",
        "speechie/v23_runtime.py",
        "speechie/SHA256SUMS.txt",
    }

    missing = required_zip_entries - set(names)

    assert not missing, (
        f"Missing from ZIP: {sorted(missing)}"
    )

    # Backup must NOT be present
    assert not any(
        "before_numpy_patch" in n
        for n in names
    ), "Backup main.py accidentally packaged"

    assert any(
        n.startswith("speechie/assets/")
        for n in names
    ), "assets/ not packaged"

os.replace(tmp_zip, FINAL_ZIP)

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

zip_sha = sha256_file(FINAL_ZIP)
size_mb = FINAL_ZIP.stat().st_size / (1024 ** 2)

print("\n" + "=" * 80)
print("FINAL TRACK-2 CODE SUBMISSION READY")
print("=" * 80)

print("ZIP:")
print(FINAL_ZIP)

print(f"\nSize   : {size_mb:.2f} MB")
print("SHA256 :", zip_sha)

print("\nVerified:")
print("  Top-level folder = speechie/")
print("  main.py           = PASS")
print("  README.md         = PASS")
print("  requirements.txt  = PASS")
print("  assets/           = PASS")
print("  backup excluded   = PASS")
print("  ZIP CRC           = PASS")

print("\nSUBMIT THIS FILE:")
print(FINAL_ZIP)

print("=" * 80)


Required files: PASS
README contact check: PASS
NumPy guard check: PASS
SHA256SUMS.txt created

FINAL TRACK-2 CODE SUBMISSION READY
ZIP:
/content/drive/MyDrive/speechie.zip

Size   : 127.85 MB
SHA256 : fd80bdacda3e5430ca7fb87bad047f2f92c1c0999796c92eba8a3eabfb3901a4

Verified:
  Top-level folder = speechie/
  main.py           = PASS
  README.md         = PASS
  requirements.txt  = PASS
  assets/           = PASS
  backup excluded   = PASS
  ZIP CRC           = PASS

SUBMIT THIS FILE:
/content/drive/MyDrive/speechie.zip
